In [4]:
import duckdb
import pandas as pd

# This ensures Jupyter doesn't truncate our massive arrays
pd.set_option('display.max_colwidth', None)

# Connect to your thesis database
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ⚠️ Make sure this path matches where you saved the WDC sample file!
file_path = '../data_raw/wdc/webtables_sample.json.gz'

print(f"Scanning WDC WebTables from {file_path}...")

# Stream the JSON file and hunt for the first music table
query = f"""
    WITH raw_webtables AS (
        SELECT 
            url,
            pageTitle,
            relation
        FROM read_json_auto('{file_path}')
        WHERE tableType = 'RELATION'
    )
    SELECT 
        url,
        pageTitle,
        relation
    FROM raw_webtables
    WHERE 
        -- Cast to string for a fast fuzzy search across the nested arrays
        CAST(relation AS VARCHAR) ILIKE '%artist%' 
        AND (CAST(relation AS VARCHAR) ILIKE '%album%' OR CAST(relation AS VARCHAR) ILIKE '%title%')
    LIMIT 1;
"""

df_sample = con.execute(query).fetchdf()

if not df_sample.empty:
    print("\n🎯 Found a music table!")
    print(f"URL: {df_sample.iloc[0]['url']}")
    print(f"Title: {df_sample.iloc[0]['pageTitle']}")
    print("\n--- RAW RELATION ARRAY ---")
    # This is the gold we are looking for:
    print(df_sample.iloc[0]['relation'])
else:
    print("No music tables found in this sample. We may need to download a larger WDC file.")

IOException: IO Error: Cannot open file "c:\users\user\onedrive\desktop\eman_thesis\jupyter\..\data_processed\thesis.duckdb": The process cannot access the file because it is being used by another process.

File is already open in 
C:\Python312\python.exe (PID 18140)

In [ ]:
import duckdb
import pandas as pd

pd.set_option('display.max_colwidth', None)
con = duckdb.connect("../data_processed/thesis.duckdb")
file_path = '../data_raw/wdc/webtables_sample.json.gz'

print("Hunting for music tables using keyword intersection scoring...")

# 1. Define your dynamic keyword list
MUSIC_KEYWORDS = [
    'music', 'artist', 'album', 'title', 'genre', 'year', 
    'label', 'band', 'singer', 'release', 'track'
]

# Convert the Python list to a SQL formatted list string: "['music', 'artist', ...]"
sql_keywords = f"[{', '.join([f'{repr(k)}' for k in MUSIC_KEYWORDS])}]"

query = f"""
    WITH raw_webtables AS (
        SELECT 
            url,
            pageTitle,
            relation,
            -- Extract headers, cast to string, lowercase, and trim whitespace
            list_transform(relation, col -> trim(lower(CAST(col[1] AS VARCHAR)))) AS headers
        FROM read_json_auto('{file_path}')
        WHERE tableType = 'RELATION'
    ),
    scored_tables AS (
        SELECT 
            url,
            pageTitle,
            headers,
            -- Find exactly which keywords matched
            list_intersect(headers, {sql_keywords}) AS matched_keywords,
            relation
        FROM raw_webtables
    )
    SELECT 
        url,
        pageTitle,
        headers,
        matched_keywords,
        -- Show how many keywords matched
        len(matched_keywords) AS match_score
    FROM scored_tables
    WHERE 
        -- THRESHOLD: The table must contain at least 2 music keywords
        len(matched_keywords) >= 2
    ORDER BY match_score DESC
    LIMIT 5;
"""

df_scored = con.execute(query).fetchdf()

if not df_scored.empty:
    print(f"\n🎯 Found {len(df_scored)} high-quality music tables!")
    display(df_scored.drop(columns=['relation'], errors='ignore')) # Print nicely without the massive relation array
    
    print("\n--- RAW RELATION ARRAY OF THE TOP HIT ---")
    print(df_scored.iloc[0]['url'])
    # Uncomment the line below if you want to print the massive raw array of the best hit
    # print(con.execute(f"SELECT relation FROM read_json_auto('{file_path}') WHERE url = '{df_scored.iloc[0]['url']}' LIMIT 1").fetchone()[0])
else:
    print("No tables met the keyword threshold in this specific sample.")

In [ ]:
#gemeni

In [ ]:
import duckdb
import pandas as pd

pd.set_option('display.max_colwidth', None)
con = duckdb.connect("../data_processed/thesis.duckdb")
file_path = '../data_raw/wdc/webtables_sample.json.gz'

print("Hunting for STRICT music tables using two-axis intersection...")

# 1. Define our two mandatory axes
ARTIST_KEYWORDS = ['artist', 'band', 'singer', 'composer']
RELEASE_KEYWORDS = ['album', 'title', 'track', 'song', 'release']

sql_artists = f"[{', '.join([f'{repr(k)}' for k in ARTIST_KEYWORDS])}]"
sql_releases = f"[{', '.join([f'{repr(k)}' for k in RELEASE_KEYWORDS])}]"

query = f"""
    WITH raw_webtables AS (
        SELECT 
            url,
            pageTitle,
            relation,
            list_transform(relation, col -> trim(lower(CAST(col[1] AS VARCHAR)))) AS headers
        FROM read_json_auto('{file_path}')
        WHERE tableType = 'RELATION'
    ),
    scored_tables AS (
        SELECT 
            url,
            pageTitle,
            headers,
            list_intersect(headers, {sql_artists}) AS matched_artists,
            list_intersect(headers, {sql_releases}) AS matched_releases,
            relation
        FROM raw_webtables
    )
    SELECT 
        url,
        pageTitle,
        headers,
        matched_artists,
        matched_releases
    FROM scored_tables
    WHERE 
        -- STRICT THRESHOLD: Must have at least one Artist word AND one Release word
        len(matched_artists) >= 1 
        AND len(matched_releases) >= 1
    LIMIT 5;
"""

df_strict = con.execute(query).fetchdf()

if not df_strict.empty:
    print(f"\n🎯 Found {len(df_strict)} strict music tables!")
    display(df_strict)
    
    print("\n--- RAW RELATION ARRAY OF THE TOP HIT ---")
    top_url = df_strict.iloc[0]['url']
    raw_array = con.execute(f"SELECT relation FROM read_json_auto('{file_path}') WHERE url = '{top_url}' LIMIT 1").fetchone()[0]
    print(raw_array)
else:
    print("No tables met the strict two-axis threshold in this sample.")

In [ ]:
#gpt

In [ ]:
import duckdb
import pandas as pd

pd.set_option('display.max_colwidth', None)

con = duckdb.connect("../data_processed/thesis.duckdb")
file_path = '../data_raw/wdc/webtables_sample.json.gz'

print("Hunting for music tables with weighted scoring...")

STRONG_KEYWORDS = ['artist', 'album', 'track', 'label', 'genre', 'band', 'song']
WEAK_KEYWORDS = ['title', 'year', 'release', 'singer', 'music']

sql_strong = f"[{', '.join(repr(k) for k in STRONG_KEYWORDS)}]"
sql_weak = f"[{', '.join(repr(k) for k in WEAK_KEYWORDS)}]"

query = f"""
WITH raw_webtables AS (
    SELECT 
        url,
        pageTitle,
        relation,
        headerPosition,
        hasHeader,
        tableType,
        -- WDC tables are often column-major:
        -- each element in relation is a column, and col[1] is the header cell
        list_transform(
            relation,
            col -> trim(lower(CAST(col[1] AS VARCHAR)))
        ) AS headers
    FROM read_json_auto('{file_path}')
    WHERE tableType = 'RELATION'
),
scored_tables AS (
    SELECT
        url,
        pageTitle,
        headers,
        relation,
        list_intersect(headers, {sql_strong}) AS strong_matches,
        list_intersect(headers, {sql_weak}) AS weak_matches,
        len(list_intersect(headers, {sql_strong})) AS strong_score,
        len(list_intersect(headers, {sql_weak})) AS weak_score
    FROM raw_webtables
)
SELECT
    url,
    pageTitle,
    headers,
    strong_matches,
    weak_matches,
    strong_score,
    weak_score,
    (strong_score * 3 + weak_score) AS total_score
FROM scored_tables
WHERE
    strong_score >= 1
    AND (strong_score * 3 + weak_score) >= 4
ORDER BY total_score DESC, strong_score DESC, weak_score DESC
LIMIT 20;
"""

df = con.execute(query).fetchdf()
print(df)

In [ ]:
#gpt

In [ ]:
import duckdb
import pandas as pd

# Display settings to ensure we can see full strings
pd.set_option('display.max_colwidth', None)

# 1. Connect to the database
con = duckdb.connect("../data_processed/thesis.duckdb")
file_path = '../data_raw/wdc/webtables_sample.json.gz'

print("Hunting for music tables with refined weighted scoring...")

# 2. Define our keyword tiers
# STRONG: Words that almost guarantee we are looking at audio/music data
STRONG_KEYWORDS = ['artist', 'album', 'track', 'band', 'song']
# WEAK: Words that are common in music, but also common in movies, books, and retail
WEAK_KEYWORDS = ['title', 'year', 'release', 'singer', 'music', 'genre', 'label']

# 3. Format lists for DuckDB array functions
sql_strong = f"[{', '.join(repr(k) for k in STRONG_KEYWORDS)}]"
sql_weak = f"[{', '.join(repr(k) for k in WEAK_KEYWORDS)}]"

# 4. Execute the weighted scoring query
query = f"""
WITH raw_webtables AS (
    SELECT 
        url,
        pageTitle,
        relation,
        -- WDC tables are column-major. col[1] is the header.
        -- We extract the headers, lowercase them, and trim whitespace.
        list_transform(
            relation,
            col -> trim(lower(CAST(col[1] AS VARCHAR)))
        ) AS headers
    FROM read_json_auto('{file_path}')
    WHERE tableType = 'RELATION'
),
scored_tables AS (
    SELECT
        url,
        pageTitle,
        headers,
        relation,
        -- Find which specific words matched
        list_intersect(headers, {sql_strong}) AS strong_matches,
        list_intersect(headers, {sql_weak}) AS weak_matches,
        -- Count the number of matches for scoring
        len(list_intersect(headers, {sql_strong})) AS strong_score,
        len(list_intersect(headers, {sql_weak})) AS weak_score
    FROM raw_webtables
)
SELECT
    url,
    pageTitle,
    headers,
    strong_matches,
    weak_matches,
    strong_score,
    weak_score,
    -- Calculate the final score: Strong words are worth 3, Weak words are worth 1
    (strong_score * 3 + weak_score) AS total_score,
    relation
FROM scored_tables
WHERE
    -- The table MUST have at least one strong music word (e.g., artist, album)
    strong_score >= 1
    -- The combined score must be at least 4. 
    -- (e.g., 1 strong + 1 weak = 4. Or 2 strong = 6).
    AND (strong_score * 3 + weak_score) >= 4
ORDER BY total_score DESC
LIMIT 30;
"""

df_refined = con.execute(query).fetchdf()

# Print the refined results (excluding the massive relation array for readability)
print(df_refined.drop(columns=['relation']))

In [ ]:
#gemeni

In [ ]:
import duckdb
import pandas as pd

# Display settings to ensure we see the full text
pd.set_option('display.max_colwidth', None)

# 1. Connect to your database
con = duckdb.connect("../data_processed/thesis.duckdb")
file_path = '../data_raw/wdc/webtables_sample.json.gz'

print("Flattening the top WebTable into a relational schema...")

# 2. Target our top hit (The WNUR Jazz Playlist)
target_url = 'http://www.wnur.org/programming/playlist/?date=2011-07-21&time=10:00am&wpmp_switcher=mobile'

# 3. The Transposition Query
# HOW IT WORKS:
# - In DuckDB, arrays are 1-indexed (the first item is 1, not 0).
# - Based on your previous output, the headers were: [start, artist, track, album, label]
# - Therefore: Artist is column 2, Track is column 3, Album is column 4.
# - We use `[2:]` to slice the array, which drops the header word and keeps only the data.
# - UNNEST() takes the list of strings and unrolls them into vertical rows.

query = f"""
    WITH raw_table AS (
        SELECT relation
        FROM read_json_auto('{file_path}')
        WHERE url = '{target_url}'
        LIMIT 1
    ),
    flattened_table AS (
        SELECT 
            -- Extract the 2nd column (Artist), drop the header row, and unroll into rows
            UNNEST(relation[2][2:]) AS wt_artist,
            
            -- Extract the 4th column (Album), drop the header row, and unroll into rows
            UNNEST(relation[4][2:]) AS wt_album,
            
            -- Extract the 3rd column (Track), drop the header row, and unroll into rows
            UNNEST(relation[3][2:]) AS wt_track
        FROM raw_table
    )
    SELECT * FROM flattened_table
    -- Filter out any empty rows caused by messy web HTML
    WHERE wt_artist IS NOT NULL AND trim(wt_artist) <> ''
"""

# Execute the query and fetch the results
df_flattened = con.execute(query).fetchdf()

# Print the results
print(f"\n🎯 Successfully flattened {len(df_flattened)} rows from the WNUR playlist!")
display(df_flattened.head(15))

In [ ]:
#gpt

In [ ]:
import duckdb
import pandas as pd

# Show all columns clearly
pd.set_option('display.max_colwidth', None)

# Connect to DuckDB
con = duckdb.connect("../data_processed/thesis.duckdb")

# Path to your sample file
file_path = "../data_raw/wdc/webtables_sample.json.gz"

# Define strong and weak keyword groups
STRONG_KEYWORDS = ['artist', 'album', 'track', 'song', 'label', 'genre', 'band']
WEAK_KEYWORDS = ['title', 'year', 'release', 'music', 'singer']

# Convert Python lists into DuckDB SQL list literals
sql_strong = f"[{', '.join(repr(k) for k in STRONG_KEYWORDS)}]"
sql_weak = f"[{', '.join(repr(k) for k in WEAK_KEYWORDS)}]"

query = f"""
WITH raw_webtables AS (
    SELECT
        url,
        pageTitle,
        relation,
        tableType,

        -- Extract header-like cells from each column.
        -- This assumes a column-major WDC layout.
        list_transform(
            relation,
            col -> trim(lower(CAST(col[1] AS VARCHAR)))
        ) AS headers
    FROM read_json_auto('{file_path}')
    WHERE tableType = 'RELATION'
),

scored_tables AS (
    SELECT
        url,
        pageTitle,
        headers,

        -- Count strong and weak music signals
        len(list_intersect(headers, {sql_strong})) AS strong_score,
        len(list_intersect(headers, {sql_weak})) AS weak_score,

        -- Boolean flags for important music fields
        list_contains(headers, 'artist') AS has_artist,
        list_contains(headers, 'album') AS has_album,
        list_contains(headers, 'track') AS has_track,
        list_contains(headers, 'song') AS has_song,
        list_contains(headers, 'label') AS has_label
    FROM raw_webtables
)

SELECT
    -- Total number of relational tables in the sample
    COUNT(*) AS total_relational_tables,

    -- Very broad candidate rule:
    -- any table with at least one strong music signal
    COUNT(*) FILTER (
        WHERE strong_score >= 1
    ) AS broad_music_candidates,

    -- Medium rule:
    -- at least two strong signals
    COUNT(*) FILTER (
        WHERE strong_score >= 2
    ) AS medium_music_candidates,

    -- Stricter metadata-oriented rule
    COUNT(*) FILTER (
        WHERE
            (
                (has_artist AND (has_album OR has_track OR has_song))
                OR
                (has_album AND (has_track OR has_song))
                OR
                (has_label AND (has_artist OR has_album))
            )
    ) AS structural_music_candidates,

    -- Your weighted-style stricter rule
    COUNT(*) FILTER (
        WHERE
            (
                (has_artist AND (has_album OR has_track OR has_song))
                OR
                (has_album AND (has_track OR has_song))
                OR
                (has_label AND (has_artist OR has_album))
            )
            AND (strong_score * 3 + weak_score) >= 5
    ) AS strict_music_candidates

FROM scored_tables;
"""

df_counts = con.execute(query).fetchdf()
print(df_counts)

In [ ]:
#gemeni

In [ ]:
import duckdb
import pandas as pd

# Display settings
pd.set_option('display.max_colwidth', None)

# 1. Connect to the database
con = duckdb.connect("../data_processed/thesis.duckdb")
file_path = '../data_raw/wdc/webtables_sample.json.gz'

print("Hunting for music tables using EXPANDED High-Recall scoring...")

# 2. Expanded Keyword Tiers
STRONG_KEYWORDS = [
    'artist', 'album', 'track', 'band', 'song', 
    'composer', 'discography', 'playlist', 'remix', 'vocalist'
]
WEAK_KEYWORDS = [
    'title', 'year', 'release', 'singer', 'music', 
    'genre', 'label', 'duration', 'length', 'format', 'bpm', 'catalog'
]

# 3. The Blacklist (Negative Constraints)
# If a table has any of these headers, it is NOT music.
NEGATIVE_KEYWORDS = [
    'director', 'author', 'manga', 'episodes', 'publisher', 
    'platform', 'developer', 'cast', 'isbn', 'pages'
]

# Format lists for DuckDB
sql_strong = f"[{', '.join(repr(k) for k in STRONG_KEYWORDS)}]"
sql_weak = f"[{', '.join(repr(k) for k in WEAK_KEYWORDS)}]"
sql_negative = f"[{', '.join(repr(k) for k in NEGATIVE_KEYWORDS)}]"

# 4. The High-Recall Query
query = f"""
WITH raw_webtables AS (
    SELECT 
        url,
        pageTitle,
        relation,
        -- Extract headers, lowercase, and trim
        list_transform(
            relation,
            col -> trim(lower(CAST(col[1] AS VARCHAR)))
        ) AS headers
    FROM read_json_auto('{file_path}')
    WHERE tableType = 'RELATION'
),
scored_tables AS (
    SELECT
        url,
        pageTitle,
        headers,
        relation,
        list_intersect(headers, {sql_strong}) AS strong_matches,
        list_intersect(headers, {sql_weak}) AS weak_matches,
        list_intersect(headers, {sql_negative}) AS negative_matches,
        len(list_intersect(headers, {sql_strong})) AS strong_score,
        len(list_intersect(headers, {sql_weak})) AS weak_score
    FROM raw_webtables
)
SELECT
    url,
    pageTitle,
    headers,
    strong_matches,
    weak_matches,
    strong_score,
    weak_score,
    -- Calculate total score: Strong = 3 pts, Weak = 1 pt
    (strong_score * 3 + weak_score) AS total_score
FROM scored_tables
WHERE
    -- RULE 1: ABSOLUTELY NO NEGATIVE WORDS ALLOWED
    len(negative_matches) = 0
    AND 
    (
        -- RULE 2A: High-Confidence (Score >= 4) just like before
        (strong_score * 3 + weak_score) >= 4
        OR
        -- RULE 2B: The "Wide Net" Expansion. 
        -- If it has exactly 1 strong word (3 pts) and NO weak words, allow it!
        -- (e.g., a table with just ['Artist', 'Country', 'City'])
        (strong_score = 1 AND weak_score = 0)
        OR
        -- RULE 2C: The "Weak but Many" Expansion.
        -- If it has NO strong words, but at least 3 weak words, allow it!
        -- (e.g., ['Title', 'Duration', 'Label'])
        (strong_score = 0 AND weak_score >= 3)
    )
ORDER BY total_score DESC
LIMIT 70;
"""

df_recall = con.execute(query).fetchdf()

print(f"\n🎯 Found {len(df_recall)} potential music tables with High-Recall rules!")
display(df_recall)

In [ ]:
#gpt

In [ ]:
import duckdb
import pandas as pd

# Show long strings fully in output
pd.set_option('display.max_colwidth', None)

# Connect to DuckDB
con = duckdb.connect("../data_processed/thesis.duckdb")

# Path to the WDC sample file
file_path = "../data_raw/wdc/webtables_sample.json.gz"

print("Hunting for more music-related tables using headers + URL + page title signals...")

# -----------------------------
# Define keyword groups
# -----------------------------
# Strong keywords are more specific to music metadata
STRONG_KEYWORDS = ['artist', 'album', 'track', 'song', 'label', 'genre', 'band']

# Weak keywords are more generic, but still helpful when combined
WEAK_KEYWORDS = ['title', 'year', 'release', 'music', 'singer', 'disc', 'single', 'record']

# Convert Python lists into DuckDB SQL list literals
sql_strong = f"[{', '.join(repr(k) for k in STRONG_KEYWORDS)}]"
sql_weak = f"[{', '.join(repr(k) for k in WEAK_KEYWORDS)}]"

query = f"""
WITH raw_webtables AS (
    SELECT
        url,
        pageTitle,
        relation,
        tableType,

        -- Extract header-like cells from each column.
        -- This assumes the common WDC column-major layout.
        list_transform(
            relation,
            col -> trim(lower(CAST(col[1] AS VARCHAR)))
        ) AS headers

    FROM read_json_auto('{file_path}')
    WHERE tableType = 'RELATION'
),

scored_tables AS (
    SELECT
        url,
        pageTitle,
        headers,

        -- -----------------------------
        -- Header-based matching
        -- -----------------------------
        list_intersect(headers, {sql_strong}) AS strong_matches,
        list_intersect(headers, {sql_weak}) AS weak_matches,

        len(list_intersect(headers, {sql_strong})) AS strong_score,
        len(list_intersect(headers, {sql_weak})) AS weak_score,

        -- -----------------------------
        -- URL-based music clues
        -- -----------------------------
        CASE
            WHEN regexp_matches(lower(coalesce(url, '')), '(music|album|artist|track|song|playlist|discography|itunes|band)')
            THEN 1 ELSE 0
        END AS url_music_signal,

        -- -----------------------------
        -- Page-title-based music clues
        -- -----------------------------
        CASE
            WHEN regexp_matches(lower(coalesce(pageTitle, '')), '(music|album|artist|track|song|playlist|discography|itunes|record label|single|ep|lp)')
            THEN 1 ELSE 0
        END AS title_music_signal

    FROM raw_webtables
),

final_scores AS (
    SELECT
        url,
        pageTitle,
        headers,
        strong_matches,
        weak_matches,
        strong_score,
        weak_score,
        url_music_signal,
        title_music_signal,

        -- Total score combines all evidence sources.
        -- Strong headers count the most.
        (
            strong_score * 3
            + weak_score
            + url_music_signal * 2
            + title_music_signal * 2
        ) AS total_score

    FROM scored_tables
)

SELECT
    url,
    pageTitle,
    headers,
    strong_matches,
    weak_matches,
    strong_score,
    weak_score,
    url_music_signal,
    title_music_signal,
    total_score,

    -- Classify each table into confidence buckets
    CASE
        WHEN total_score >= 7 THEN 'high_confidence_music'
        WHEN total_score >= 4 THEN 'possible_music'
        ELSE 'unlikely_music'
    END AS music_bucket

FROM final_scores

WHERE
    -- Keep tables that have some evidence of music from any source
    (
        strong_score >= 1
        OR weak_score >= 2
        OR url_music_signal = 1
        OR title_music_signal = 1
    )

    -- Exclude obvious non-target entertainment sources
    AND NOT regexp_matches(lower(coalesce(url, '')), '(mangaupdates|anime-planet)')
    AND NOT regexp_matches(lower(coalesce(pageTitle, '')), '(manga|anime)')

ORDER BY
    total_score DESC,
    strong_score DESC,
    weak_score DESC

LIMIT 100;
"""

# Run the query
df = con.execute(query).fetchdf()

# Show the resulting candidate tables
print(df)

# Also print how many rows were returned
print(f"\nNumber of music-related candidate tables returned: {len(df)}")

In [ ]:
#gpt

In [ ]:
import duckdb
import pandas as pd

# Show long strings fully in output
pd.set_option('display.max_colwidth', None)

# Connect to DuckDB
con = duckdb.connect("../data_processed/thesis.duckdb")

# Path to the WDC sample file
file_path = "../data_raw/wdc/webtables_sample.json.gz"

# Strong and weak keyword groups
STRONG_KEYWORDS = ['artist', 'album', 'track', 'song', 'label', 'genre', 'band']
WEAK_KEYWORDS = ['title', 'year', 'release', 'music', 'singer', 'disc', 'single', 'record']

# Convert Python lists into DuckDB SQL list literals
sql_strong = f"[{', '.join(repr(k) for k in STRONG_KEYWORDS)}]"
sql_weak = f"[{', '.join(repr(k) for k in WEAK_KEYWORDS)}]"

query_counts = f"""
WITH raw_webtables AS (
    SELECT
        url,
        pageTitle,
        relation,
        tableType,

        -- Extract header-like cells from each column
        list_transform(
            relation,
            col -> trim(lower(CAST(col[1] AS VARCHAR)))
        ) AS headers

    FROM read_json_auto('{file_path}')
    WHERE tableType = 'RELATION'
),

scored_tables AS (
    SELECT
        url,
        pageTitle,
        headers,

        -- Header scores
        len(list_intersect(headers, {sql_strong})) AS strong_score,
        len(list_intersect(headers, {sql_weak})) AS weak_score,

        -- URL and title music signals
        CASE
            WHEN regexp_matches(lower(coalesce(url, '')), '(music|album|artist|track|song|playlist|discography|itunes|band)')
            THEN 1 ELSE 0
        END AS url_music_signal,

        CASE
            WHEN regexp_matches(lower(coalesce(pageTitle, '')), '(music|album|artist|track|song|playlist|discography|itunes|record label|single|ep|lp)')
            THEN 1 ELSE 0
        END AS title_music_signal

    FROM raw_webtables
),

bucketed AS (
    SELECT
        *,
        (
            strong_score * 3
            + weak_score
            + url_music_signal * 2
            + title_music_signal * 2
        ) AS total_score,

        CASE
            WHEN (
                strong_score * 3
                + weak_score
                + url_music_signal * 2
                + title_music_signal * 2
            ) >= 7 THEN 'high_confidence_music'

            WHEN (
                strong_score * 3
                + weak_score
                + url_music_signal * 2
                + title_music_signal * 2
            ) >= 4 THEN 'possible_music'

            ELSE 'unlikely_music'
        END AS music_bucket
    FROM scored_tables
)

SELECT
    music_bucket,
    COUNT(*) AS table_count
FROM bucketed
GROUP BY music_bucket
ORDER BY table_count DESC;
"""

# Execute and display bucket counts
df_counts = con.execute(query_counts).fetchdf()
print(df_counts)

In [ ]:
#gpt

In [ ]:
import duckdb
import pandas as pd

# Show long text fully in the DataFrame output
pd.set_option('display.max_colwidth', None)

# Connect to DuckDB
con = duckdb.connect("../data_processed/thesis.duckdb")

# Path to the WDC sample file
file_path = "../data_raw/wdc/webtables_sample.json.gz"

print("Hunting for music-related tables with safer regex rules...")

# -----------------------------------
# Keyword groups
# -----------------------------------
# Strong header keywords are highly informative for music tables
STRONG_KEYWORDS = ['artist', 'album', 'track', 'song', 'label', 'genre', 'band']

# Weak header keywords are more generic and should contribute less
WEAK_KEYWORDS = ['title', 'year', 'release', 'music', 'singer', 'disc', 'record']

# Convert Python lists into DuckDB SQL list literals
sql_strong = f"[{', '.join(repr(k) for k in STRONG_KEYWORDS)}]"
sql_weak = f"[{', '.join(repr(k) for k in WEAK_KEYWORDS)}]"

query = f"""
WITH raw_webtables AS (
    SELECT
        url,
        pageTitle,
        relation,
        tableType,

        -- Extract header-like cells from each column.
        -- This assumes a column-major WDC layout, which is common.
        list_transform(
            relation,
            col -> trim(lower(CAST(col[1] AS VARCHAR)))
        ) AS headers

    FROM read_json_auto('{file_path}')
    WHERE tableType = 'RELATION'
),

scored_tables AS (
    SELECT
        url,
        pageTitle,
        headers,

        -- -----------------------------------
        -- Header-based evidence
        -- -----------------------------------
        list_intersect(headers, {sql_strong}) AS strong_matches,
        list_intersect(headers, {sql_weak}) AS weak_matches,

        len(list_intersect(headers, {sql_strong})) AS strong_score,
        len(list_intersect(headers, {sql_weak})) AS weak_score,

        -- -----------------------------------
        -- URL signal
        -- -----------------------------------
        -- URL matching can stay a bit broader because URLs often use separators
        CASE
            WHEN regexp_matches(
                lower(coalesce(url, '')),
                '(music|album|artist|track|song|playlist|discography|itunes|band)'
            )
            THEN 1 ELSE 0
        END AS url_music_signal,

        -- -----------------------------------
        -- Page-title signal
        -- -----------------------------------
        -- IMPORTANT:
        -- Use word boundaries so short fragments like "ep" or "lp"
        -- do not match inside ordinary words like "helping" or "Stephanie".
        CASE
            WHEN regexp_matches(
                lower(coalesce(pageTitle, '')),
                '\\\\b(music|album|artist|track|song|playlist|discography|itunes|record label|single)\\\\b'
            )
            THEN 1 ELSE 0
        END AS title_music_signal

    FROM raw_webtables
),

final_scores AS (
    SELECT
        url,
        pageTitle,
        headers,
        strong_matches,
        weak_matches,
        strong_score,
        weak_score,
        url_music_signal,
        title_music_signal,

        -- Strong headers count the most.
        -- URL and title signals support recall but should not dominate.
        (
            strong_score * 3
            + weak_score
            + url_music_signal * 2
            + title_music_signal * 2
        ) AS total_score

    FROM scored_tables
),

bucketed AS (
    SELECT
        *,
        CASE
            WHEN total_score >= 7 THEN 'high_confidence_music'
            WHEN total_score >= 4 THEN 'possible_music'
            ELSE 'unlikely_music'
        END AS music_bucket
    FROM final_scores
)

SELECT
    url,
    pageTitle,
    headers,
    strong_matches,
    weak_matches,
    strong_score,
    weak_score,
    url_music_signal,
    title_music_signal,
    total_score,
    music_bucket

FROM bucketed

WHERE
    -- Keep only useful candidate buckets
    music_bucket IN ('high_confidence_music', 'possible_music')

    -- Exclude obvious non-target sources
    AND NOT regexp_matches(lower(coalesce(url, '')), '(mangaupdates|anime-planet)')
    AND NOT regexp_matches(lower(coalesce(pageTitle, '')), '\\\\b(manga|anime)\\\\b')

ORDER BY
    total_score DESC,
    strong_score DESC,
    weak_score DESC

LIMIT 100;
"""

# Execute the query and collect results
df = con.execute(query).fetchdf()

# Print results
print(df)

# Print the number of returned candidate tables
print(f"\nNumber of candidate music-related tables returned: {len(df)}")

In [ ]:
#gemeni

In [ ]:
import duckdb
import pandas as pd

# Display settings
pd.set_option('display.max_colwidth', None)

# 1. Connect to the database
con = duckdb.connect("../data_processed/thesis.duckdb")
file_path = '../data_raw/wdc/webtables_sample.json.gz'

print("Hunting for music tables using EXPANDED High-Recall scoring...")

# 2. Expanded Keyword Tiers
STRONG_KEYWORDS = [
    'artist', 'album', 'track', 'band', 'song', 
    'composer', 'discography', 'playlist', 'remix', 'vocalist'
]
WEAK_KEYWORDS = [
    'title', 'year', 'release', 'singer', 'music', 
    'genre', 'label', 'duration', 'length', 'format', 'bpm', 'catalog'
]

# 3. The Blacklist (Negative Constraints)
# If a table has any of these headers, it is NOT music.
NEGATIVE_KEYWORDS = [
    'director', 'author', 'manga', 'episodes', 'publisher', 
    'platform', 'developer', 'cast', 'isbn', 'pages'
]

# Format lists for DuckDB
sql_strong = f"[{', '.join(repr(k) for k in STRONG_KEYWORDS)}]"
sql_weak = f"[{', '.join(repr(k) for k in WEAK_KEYWORDS)}]"
sql_negative = f"[{', '.join(repr(k) for k in NEGATIVE_KEYWORDS)}]"

# 4. The High-Recall Query
query = f"""
WITH raw_webtables AS (
    SELECT 
        url,
        pageTitle,
        relation,
        -- Extract headers, lowercase, and trim
        list_transform(
            relation,
            col -> trim(lower(CAST(col[1] AS VARCHAR)))
        ) AS headers
    FROM read_json_auto('{file_path}')
    WHERE tableType = 'RELATION'
),
scored_tables AS (
    SELECT
        url,
        pageTitle,
        headers,
        relation,
        list_intersect(headers, {sql_strong}) AS strong_matches,
        list_intersect(headers, {sql_weak}) AS weak_matches,
        list_intersect(headers, {sql_negative}) AS negative_matches,
        len(list_intersect(headers, {sql_strong})) AS strong_score,
        len(list_intersect(headers, {sql_weak})) AS weak_score
    FROM raw_webtables
)
SELECT
    url,
    pageTitle,
    headers,
    strong_matches,
    weak_matches,
    strong_score,
    weak_score,
    -- Calculate total score: Strong = 3 pts, Weak = 1 pt
    (strong_score * 3 + weak_score) AS total_score
FROM scored_tables
WHERE
    -- RULE 1: ABSOLUTELY NO NEGATIVE WORDS ALLOWED
    len(negative_matches) = 0
    AND 
    (
        -- RULE 2A: High-Confidence (Score >= 4) just like before
        (strong_score * 3 + weak_score) >= 4
        OR
        -- RULE 2B: The "Wide Net" Expansion. 
        -- If it has exactly 1 strong word (3 pts) and NO weak words, allow it!
        -- (e.g., a table with just ['Artist', 'Country', 'City'])
        (strong_score = 1 AND weak_score = 0)
        OR
        -- RULE 2C: The "Weak but Many" Expansion.
        -- If it has NO strong words, but at least 3 weak words, allow it!
        -- (e.g., ['Title', 'Duration', 'Label'])
        (strong_score = 0 AND weak_score >= 3)
    )
ORDER BY total_score DESC
LIMIT 20;
"""

df_recall = con.execute(query).fetchdf()

print(f"\n🎯 Found {len(df_recall)} potential music tables with High-Recall rules!")
display(df_recall)

In [ ]:
#gemeni

In [ ]:
import duckdb
import pandas as pd

pd.set_option('display.max_colwidth', None)
con = duckdb.connect("../data_processed/thesis.duckdb")
file_path = "../data_raw/wdc/webtables_sample.json.gz"

print("Building final WDC music tables using Multi-Signal + Blacklist...")

# Strong and weak keyword groups
STRONG_KEYWORDS = ['artist', 'album', 'track', 'song', 'label', 'genre', 'band']
WEAK_KEYWORDS = ['title', 'year', 'release', 'music', 'singer', 'disc', 'single', 'record']
# The Blacklist
NEGATIVE_KEYWORDS = ['director', 'author', 'manga', 'episodes', 'publisher', 'platform', 'developer', 'cast', 'isbn']

sql_strong = f"[{', '.join(repr(k) for k in STRONG_KEYWORDS)}]"
sql_weak = f"[{', '.join(repr(k) for k in WEAK_KEYWORDS)}]"
sql_negative = f"[{', '.join(repr(k) for k in NEGATIVE_KEYWORDS)}]"

query = f"""
CREATE OR REPLACE TABLE wdc_music_candidate_tables AS
WITH raw_webtables AS (
    SELECT
        url,
        pageTitle,
        relation,
        tableType,
        -- Extract headers
        list_transform(
            relation,
            col -> trim(lower(CAST(col[1] AS VARCHAR)))
        ) AS headers
    FROM read_json_auto('{file_path}')
    WHERE tableType = 'RELATION'
),
scored_tables AS (
    SELECT
        url,
        pageTitle,
        headers,
        relation,
        
        -- Header scores
        len(list_intersect(headers, {sql_strong})) AS strong_score,
        len(list_intersect(headers, {sql_weak})) AS weak_score,
        len(list_intersect(headers, {sql_negative})) AS negative_score,

        -- Context Signals
        CASE WHEN regexp_matches(lower(coalesce(url, '')), '(music|album|artist|track|song|playlist|discography|itunes|band)') THEN 1 ELSE 0 END AS url_music_signal,
        CASE WHEN regexp_matches(lower(coalesce(pageTitle, '')), '\\b(music|album|artist|track|song|playlist|discography|itunes|record label|single)\\b') THEN 1 ELSE 0 END AS title_music_signal
    FROM raw_webtables
),
bucketed AS (
    SELECT
        *,
        (strong_score * 3 + weak_score + url_music_signal * 2 + title_music_signal * 2) AS total_score
    FROM scored_tables
    -- STRICT BLACKLIST: No negative words allowed
    WHERE negative_score = 0
)
SELECT *
FROM bucketed
WHERE total_score >= 4
  AND (strong_score >= 1 OR weak_score >= 1) -- Must have at least some tabular metadata
ORDER BY total_score DESC;
"""

con.execute(query)
df_final = con.execute("SELECT url, pageTitle, headers, total_score FROM wdc_music_candidate_tables").fetchdf()

print(f"\n✅ Successfully saved {len(df_final)} pristine music tables to DuckDB!")
display(df_final.head(15))

In [ ]:
#gemeni

In [ ]:
import duckdb
import pandas as pd

pd.set_option('display.max_colwidth', None)
con = duckdb.connect("../data_processed/thesis.duckdb")
file_path = "../data_raw/wdc/webtables_sample.json.gz"

print("Building final WDC music tables using Multi-Signal + Blacklist...")

# Strong and weak keyword groups
STRONG_KEYWORDS = ['artist', 'album', 'track', 'song', 'label', 'genre', 'band']
WEAK_KEYWORDS = ['title', 'year', 'release', 'music', 'singer', 'disc', 'single', 'record']
# The Blacklist
NEGATIVE_KEYWORDS = ['director', 'author', 'manga', 'episodes', 'publisher', 'platform', 'developer', 'cast', 'isbn']

sql_strong = f"[{', '.join(repr(k) for k in STRONG_KEYWORDS)}]"
sql_weak = f"[{', '.join(repr(k) for k in WEAK_KEYWORDS)}]"
sql_negative = f"[{', '.join(repr(k) for k in NEGATIVE_KEYWORDS)}]"

query = f"""
CREATE OR REPLACE TABLE wdc_music_candidate_tables AS
WITH raw_webtables AS (
    SELECT
        url,
        pageTitle,
        relation,
        tableType,
        -- Extract headers
        list_transform(
            relation,
            col -> trim(lower(CAST(col[1] AS VARCHAR)))
        ) AS headers
    FROM read_json_auto('{file_path}')
    WHERE tableType = 'RELATION'
),
scored_tables AS (
    SELECT
        url,
        pageTitle,
        headers,
        relation,
        
        -- Header scores
        len(list_intersect(headers, {sql_strong})) AS strong_score,
        len(list_intersect(headers, {sql_weak})) AS weak_score,
        len(list_intersect(headers, {sql_negative})) AS negative_score,

        -- Context Signals
        CASE WHEN regexp_matches(lower(coalesce(url, '')), '(music|album|artist|track|song|playlist|discography|itunes|band)') THEN 1 ELSE 0 END AS url_music_signal,
        CASE WHEN regexp_matches(lower(coalesce(pageTitle, '')), '\\b(music|album|artist|track|song|playlist|discography|itunes|record label|single)\\b') THEN 1 ELSE 0 END AS title_music_signal
    FROM raw_webtables
),
bucketed AS (
    SELECT
        *,
        (strong_score * 3 + weak_score + url_music_signal * 2 + title_music_signal * 2) AS total_score
    FROM scored_tables
    -- STRICT BLACKLIST: No negative words allowed
    WHERE negative_score = 0
)
SELECT *
FROM bucketed
WHERE total_score >= 4
  AND (strong_score >= 1 OR weak_score >= 1) -- Must have at least some tabular metadata
ORDER BY total_score DESC;
"""

con.execute(query)
df_final = con.execute("SELECT url, pageTitle, headers, total_score FROM wdc_music_candidate_tables").fetchdf()

print(f"\n✅ Successfully saved {len(df_final)} pristine music tables to DuckDB!")
display(df_final.head(15))

In [ ]:
import duckdb
import pandas as pd

# Display settings
pd.set_option('display.max_colwidth', None)

# 1. Connect to DuckDB
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

print("Dynamically mapping and flattening 26 WDC tables...")

# 2. Fetch the candidate tables and their headers
candidates_df = con.execute("""
    SELECT url, headers 
    FROM wdc_music_candidate_tables
""").fetchdf()

# 3. Define our target columns and their common synonyms (aliases)
COLUMN_ALIASES = {
    'wt_artist': ['artist', 'band', 'singer', 'composer'],
    'wt_album': ['album', 'release'],
    'wt_track': ['track', 'song', 'title', 'name']
}

all_flattened_queries = []

# 4. Iterate through every single table to dynamically find its schema
for index, row in candidates_df.iterrows():
    url = row['url']
    headers = row['headers']
    
    # This dictionary will store the 1-based index of our target columns
    col_indices = {'wt_artist': 'NULL', 'wt_album': 'NULL', 'wt_track': 'NULL'}
    
    # Find the exact column index for Artist, Album, and Track
    for target_col, aliases in COLUMN_ALIASES.items():
        for i, header in enumerate(headers):
            # If the header matches one of our aliases (e.g., 'song' matches 'wt_track')
            if header in aliases:
                # DuckDB arrays are 1-indexed, so we add 1. 
                # We also add the [2:] slice to drop the header row and keep the data.
                col_indices[target_col] = f"UNNEST(relation[{i + 1}][2:])"
                break # Found it, move to the next target column
    
    # If the table has AT LEAST an Artist or a Track, build the extraction query
    if col_indices['wt_artist'] != 'NULL' or col_indices['wt_track'] != 'NULL':
        
        # We handle single quotes in URLs by escaping them (' -> '') for SQL
        safe_url = url.replace("'", "''")
        
        # Build the dynamic query for this specific table
        query = f"""
            SELECT 
                '{safe_url}' AS source_url,
                {col_indices['wt_artist']} AS wt_artist,
                {col_indices['wt_album']} AS wt_album,
                {col_indices['wt_track']} AS wt_track
            FROM wdc_music_candidate_tables
            WHERE url = '{safe_url}'
        """
        all_flattened_queries.append(query)

# 5. Combine all 26 queries into one massive UNION ALL statement
if all_flattened_queries:
    master_query = " \nUNION ALL\n ".join(all_flattened_queries)
    
    # Wrap it in a CREATE TABLE statement to save it to our database
    final_sql = f"""
        CREATE OR REPLACE TABLE wdc_flattened_music_metadata AS
        WITH combined_data AS (
            {master_query}
        )
        SELECT * FROM combined_data
        -- Clean up the web mess: Remove totally empty rows
        WHERE (wt_artist IS NOT NULL AND trim(wt_artist) <> '')
           OR (wt_track IS NOT NULL AND trim(wt_track) <> '')
    """
    
    # Execute the massive transposition
    con.execute(final_sql)
    
    # Check the results!
    df_result = con.execute("SELECT * FROM wdc_flattened_music_metadata").fetchdf()
    print(f"\n✅ SUCCESS! Flattened {len(candidates_df)} nested arrays into {len(df_result)} relational rows!")
    
    print("\n--- A sample of our new flattened web data ---")
    display(df_result.dropna(subset=['wt_artist', 'wt_track']).head(20))
else:
    print("No matching columns found to flatten.")

In [ ]:
#One-to-Many Relationship.

#The URL (The "One"): http://www.wnur.org/... is a single webpage for a 2-hour Jazz Radio show.

#The Rows (The "Many"): During that 2-hour show, they played 14 different songs.
#When your Python script used the UNNEST() function, it took that one table and "exploded" it into 14 distinct rows. We copy the source_url onto every single row so we never lose the provenance (the origin) of the data. When you tell MusicBrainz, "Hey, I found a new track for the Sun Ra Arkestra," you have the exact URL to prove where you found it!
#2. Why are the pages "Not Found"? (Link Rot)
#As we discussed briefly, you are experiencing Link Rot.
#The Web Data Commons (WDC) dataset was scraped by the Common Crawl bots many years ago. In the years since, radio stations update their websites and Apple changes iTunes to Apple Music. The live URLs die and give "404 Not Found" errors.
#But for your thesis, this is a superpower: You have successfully extracted and preserved music metadata that no longer exists on the live internet! You don't need the live websites to work, because the data is safely locked inside your local DuckDB database.

In [ ]:
#normalized "join keys" (lowercase, stripping punctuation) so we can match apples-to-apples.
import duckdb
import pandas as pd

pd.set_option('display.max_colwidth', None)
con = duckdb.connect("../data_processed/thesis.duckdb")

print("Cleaning WebTable text to create normalized join keys...")

query = """
    CREATE OR REPLACE TABLE wdc_cleaned_for_matching AS
    SELECT 
        source_url,
        wt_artist,
        wt_album,
        wt_track,
        
        -- 1. Create Artist Key (Lowercase, strip punctuation/symbols, collapse spaces)
        trim(regexp_replace(lower(wt_artist), '[^a-z0-9]+', ' ', 'g')) AS wt_artist_key,
        
        -- 2. Create Album Key 
        trim(regexp_replace(lower(wt_album), '[^a-z0-9]+', ' ', 'g')) AS wt_album_key,
        
        -- 3. Create Track Key (We also strip common web artifacts like 'feat.')
        trim(regexp_replace(
            regexp_replace(lower(wt_track), '\\(feat.*\\)|feat\\..*', '', 'g'), 
            '[^a-z0-9]+', ' ', 'g'
        )) AS wt_track_key

    FROM wdc_flattened_music_metadata
"""

con.execute(query)

df_clean = con.execute("""
    SELECT wt_artist, wt_artist_key, wt_track, wt_track_key 
    FROM wdc_cleaned_for_matching 
    LIMIT 10
""").fetchdf()

print("✅ Keys successfully created!")
display(df_clean)

In [ ]:
import duckdb
import pandas as pd

pd.set_option('display.max_colwidth', None)
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

print("Building Corrected Master Integration Keys...")

# -------------------------------------------------------------------
# STEP 1: Standardize the Enriched Master Database (MB + CDS)
# -------------------------------------------------------------------
con.execute("""
    CREATE OR REPLACE VIEW master_discovery_keys AS
    SELECT
        base_record_id,
        base_record_status,
        COALESCE(main_artist_name, external_artist) AS resolved_artist,
        COALESCE(release_title, external_title) AS resolved_title,
        
        -- Generate normalized join keys
        trim(regexp_replace(lower(COALESCE(main_artist_name, external_artist)), '[^a-z0-9]+', ' ', 'g')) AS master_artist_key,
        trim(regexp_replace(lower(COALESCE(release_title, external_title)), '[^a-z0-9]+', ' ', 'g')) AS master_album_key
    FROM music_enrichment_base_for_webtables_v1
    WHERE COALESCE(main_artist_name, external_artist) IS NOT NULL 
      AND COALESCE(release_title, external_title) IS NOT NULL
""")

# -------------------------------------------------------------------
# STEP 2: The Exact Match (Finding "Known Entities")
# -------------------------------------------------------------------
# FIX APPLIED: We explicitly block any row where the artist key or 
# album key is missing or erased by regex.
con.execute("""
    CREATE OR REPLACE TABLE wdc_master_exact_matches AS
    SELECT
        w.source_url,
        w.wt_artist,
        w.wt_album,
        w.wt_track,
        m.base_record_id,
        m.base_record_status,
        m.resolved_artist AS master_artist,
        m.resolved_title AS master_album
    FROM wdc_cleaned_for_matching w
    JOIN master_discovery_keys m
      ON w.wt_artist_key = m.master_artist_key
     AND w.wt_album_key = m.master_album_key
    WHERE 
      -- The Web Data must have a valid album AND artist
      w.wt_album_key IS NOT NULL AND w.wt_album_key <> ''
      AND w.wt_artist_key IS NOT NULL AND w.wt_artist_key <> ''
      -- The Master Data must have a valid album AND artist
      AND m.master_album_key IS NOT NULL AND m.master_album_key <> ''
      AND m.master_artist_key IS NOT NULL AND m.master_artist_key <> ''
""")

# -------------------------------------------------------------------
# STEP 3: The Anti-Join (Isolating "True Novel Candidates")
# -------------------------------------------------------------------
# FIX APPLIED: We only classify a web row as a "Novel Candidate" if it 
# actually contains enough data to be a valid discovery (needs an artist and album).
con.execute("""
    CREATE OR REPLACE TABLE wdc_true_novel_candidates AS
    SELECT 
        w.*
    FROM wdc_cleaned_for_matching w
    LEFT JOIN wdc_master_exact_matches e
      ON w.source_url = e.source_url 
     AND coalesce(w.wt_artist, 'NULL') = coalesce(e.wt_artist, 'NULL')
     AND coalesce(w.wt_track, 'NULL') = coalesce(e.wt_track, 'NULL')
    WHERE e.base_record_id IS NULL
      -- Discard web rows that are missing artists (like the Paramore quiz)
      -- because they are 'Incomplete Data', not 'Novel Candidates'
      AND w.wt_artist_key IS NOT NULL AND w.wt_artist_key <> ''
      AND w.wt_album_key IS NOT NULL AND w.wt_album_key <> ''
""")

# -------------------------------------------------------------------
# STEP 4: Print the Discovery Analytics
# -------------------------------------------------------------------
known_count = con.execute("SELECT COUNT(*) FROM wdc_master_exact_matches").fetchone()[0]
novel_count = con.execute("SELECT COUNT(*) FROM wdc_true_novel_candidates").fetchone()[0]

print("\n--- CORRECTED BROAD DISCOVERY PIPELINE RESULTS ---")
print(f"🔗 Known Entities (Found in MB or CDS): {known_count}")
print(f"🚀 True Novel Candidates (Missing from BOTH): {novel_count}")

print("\n--- Sample of Known Entities ---")
display(con.execute("""
    SELECT wt_artist, master_artist, wt_album, master_album, base_record_status 
    FROM wdc_master_exact_matches LIMIT 5
""").fetchdf())

print("\n--- Sample of True Novel Candidates (Potential Discoveries!) ---")
display(con.execute("""
    SELECT wt_artist, wt_album, wt_track 
    FROM wdc_true_novel_candidates 
    LIMIT 10
""").fetchdf())

In [ ]:
import duckdb
import pandas as pd

# Standard display setup
pd.set_option('display.max_colwidth', None)
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# -------------------------------------------------------------------
# PATH A: SCHEMA AUGMENTATION (Enrich Existing Records)
# -------------------------------------------------------------------
# Logic:
# We join our 'Known Entities' back to the Master table.
# We add the 'source_url' from the web to prove the record exists online,
# and we can potentially append web track names to MusicBrainz IDs.
print("Executing Path A: Augmenting existing MusicBrainz records...")

con.execute("""
    CREATE OR REPLACE TABLE enrichment_step2_augmented_mb AS
    SELECT 
        m.*,
        w.source_url AS web_provenance_url,
        w.wt_track AS web_discovered_track
    FROM music_enrichment_base_for_webtables_v1 m
    JOIN wdc_master_exact_matches w
      ON m.base_record_id = w.base_record_id
""")

# -------------------------------------------------------------------
# PATH B: CATALOG EXPANSION (Adding New Rows)
# -------------------------------------------------------------------
# Logic:
# These are our 'True Novel Candidates'. 
# Since they don't have a 'base_record_id', we assign them a new 
# 'web_discovery_id' and format them to match your Master schema.
print("Executing Path B: Expanding catalog with novel web discoveries...")

con.execute("""
    CREATE OR REPLACE TABLE enrichment_step2_discovered_rows AS
    SELECT 
        -- Generate a unique ID for these new web-only records
        (10000000 + row_number() OVER ()) AS base_record_id,
        'web_discovery_novel' AS base_record_status,
        wt_artist AS main_artist_name,
        wt_album AS release_title,
        wt_track AS external_tracklist,
        source_url AS enrichment_source,
        'webtable_discovery' AS enrichment_route,
        FALSE AS is_linked_to_musicbrainz
    FROM wdc_true_novel_candidates
""")

# -------------------------------------------------------------------
# STEP 3: THE FINAL KNOWLEDGE GRAPH (The "Upsert")
# -------------------------------------------------------------------
# Logic:
# We combine the original enriched database with our brand new discoveries.
print("Merging all paths into the Final Enrichment Master...")

con.execute("""
    CREATE OR REPLACE TABLE thesis_final_enriched_knowledge_graph AS
    -- Keep all original records (including those we just augmented)
    SELECT * FROM music_enrichment_base_for_webtables_v1
    UNION ALL
    -- Add the brand new rows we found on the web
    SELECT 
        base_record_id,
        NULL AS release_id, -- These don't have MB IDs yet!
        release_title,
        main_artist_name,
        NULL AS artist_credit_full,
        NULL AS barcode,
        NULL AS mb_year,
        NULL AS total_tracks,
        NULL AS external_record_id,
        main_artist_name AS external_artist,
        release_title AS external_title,
        NULL AS external_year,
        NULL AS external_category,
        NULL AS external_genre,
        external_tracklist,
        enrichment_route,
        'discovered' AS enrichment_status,
        enrichment_source,
        is_linked_to_musicbrainz,
        base_record_status
    FROM enrichment_step2_discovered_rows
""")

# -------------------------------------------------------------------
# FINAL REPORTING
# -------------------------------------------------------------------
final_stats = con.execute("""
    SELECT base_record_status, COUNT(*) as count 
    FROM thesis_final_enriched_knowledge_graph 
    GROUP BY base_record_status
""").fetchdf()

print("\n--- FINAL ENRICHMENT SUMMARY ---")
display(final_stats)

print("\n--- SAMPLE OF NEW DISCOVERIES ADDED TO THE ENCYCLOPEDIA ---")
display(con.execute("""
    SELECT main_artist_name, release_title, enrichment_source 
    FROM thesis_final_enriched_knowledge_graph 
    WHERE base_record_status = 'web_discovery_novel' 
    LIMIT 5
""").fetchdf())

In [ ]:
import duckdb
import pandas as pd
from rapidfuzz import fuzz, process

# Connect to the thesis database
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# -------------------------------------------------------------------
# STEP 1: Prepare the Fuzzy Search Space (Blocking)
# -------------------------------------------------------------------
# Logic: We only pull MusicBrainz artists that share the same first 
# letter as our web discoveries to keep the search space manageable.
print("Extracting candidate search blocks from MusicBrainz...")

novel_df = con.execute("SELECT DISTINCT wt_artist FROM wdc_true_novel_candidates WHERE wt_artist IS NOT NULL").fetchdf()
novel_artists = novel_df['wt_artist'].tolist()

# Get unique first letters to define our blocks
first_letters = tuple(set([a[0].lower() for a in novel_artists if a]))

# Fetch MB artists in the same alpha-blocks
# We cast to VARCHAR and filter out NULLs to prevent fuzzy matching errors
mb_candidates = con.execute(f"""
    SELECT DISTINCT 
        base_record_id, 
        CAST(resolved_artist AS VARCHAR) as resolved_artist
    FROM master_discovery_keys 
    WHERE lower(left(resolved_artist, 1)) IN {first_letters}
      AND resolved_artist IS NOT NULL
""").fetchdf()

# -------------------------------------------------------------------
# STEP 2: The Fuzzy Validation Loop
# -------------------------------------------------------------------
# FIX APPLIED: Changed 'TokenSetRatio' to 'token_set_ratio' (lowercase)
# FIX APPLIED: Added handling for empty candidate lists
print(f"Validating {len(novel_artists)} novel candidates against {len(mb_candidates)} MB records...")

fuzzy_matches = []
mb_artist_list = mb_candidates['resolved_artist'].tolist()

for web_artist in novel_artists:
    # Use RapidFuzz to find the best match in the block
    # scorer=fuzz.token_set_ratio is the corrected attribute name
    best_match = process.extractOne(
        web_artist, 
        mb_artist_list, 
        scorer=fuzz.token_set_ratio
    )
    
    # If the score is >= 95, it's almost certainly the same artist (just a typo or "The" prefix)
    # We use a high threshold (95) to be conservative in our 'Discovery' claim
    if best_match and best_match[1] >= 95:
        match_idx = best_match[2]
        fuzzy_matches.append({
            'wt_artist': web_artist,
            'matched_mb_artist': best_match[0],
            'mb_record_id': mb_candidates.iloc[match_idx]['base_record_id'],
            'confidence': best_match[1]
        })

df_fuzzy = pd.DataFrame(fuzzy_matches)
print(f"🏁 Validation complete. Found {len(df_fuzzy)} existing artists that were slightly misspelled.")

# -------------------------------------------------------------------
# STEP 3: Final Classification & Knowledge Graph Construction
# -------------------------------------------------------------------
# Logic: We merge the results back. If a web row didn't have a fuzzy 
# match, it's officially a "Confirmed Discovery".
print("Constructing the Final Thesis Knowledge Graph...")

if not df_fuzzy.empty:
    con.register('df_fuzzy_validation', df_fuzzy)
else:
    # Create an empty dummy table if no fuzzy matches found to keep SQL working
    con.execute("CREATE OR REPLACE TEMPORARY TABLE df_fuzzy_validation (wt_artist VARCHAR, mb_record_id BIGINT)")

con.execute("""
    CREATE OR REPLACE TABLE thesis_final_knowledge_graph_v4 AS
    WITH validated_web AS (
        SELECT 
            w.*,
            v.mb_record_id AS fuzzy_mb_id,
            CASE 
                WHEN v.mb_record_id IS NOT NULL THEN 'web_enriched_existing'
                ELSE 'web_discovery_confirmed'
            END AS final_status
        FROM wdc_true_novel_candidates w
        LEFT JOIN df_fuzzy_validation v ON w.wt_artist = v.wt_artist
    )
    -- All previous MusicBrainz + CDS records
    SELECT * FROM music_enrichment_base_for_webtables_v1
    UNION ALL
    -- The newly discovered (and validated) Web rows
    SELECT 
        COALESCE(fuzzy_mb_id, (20000000 + row_number() OVER ())) AS base_record_id,
        NULL AS release_id,
        wt_album AS release_title,
        wt_artist AS main_artist_name,
        NULL AS artist_credit_full,
        NULL AS barcode,
        NULL AS mb_year,
        NULL AS total_tracks,
        NULL AS external_record_id,
        wt_artist AS external_artist,
        wt_album AS external_title,
        NULL AS external_year,
        NULL AS external_category,
        NULL AS external_genre,
        wt_track AS external_tracklist,
        'webtable_extraction' AS enrichment_route,
        'final_pass' AS enrichment_status,
        source_url AS enrichment_source,
        (fuzzy_mb_id IS NOT NULL) AS is_linked_to_musicbrainz,
        final_status AS base_record_status
    FROM validated_web;
""")

# Final Count Check
print("\n" + "="*40)
print("FINAL THESIS ENRICHMENT REPORT")
print("="*40)
display(con.execute("SELECT base_record_status, COUNT(*) as total FROM thesis_final_knowledge_graph_v4 GROUP BY 1").fetchdf())

In [ ]:
# Extract the specific rows that your pipeline successfully discovered
discoveries = con.execute("""
    SELECT main_artist_name, release_title, enrichment_source 
    FROM thesis_final_knowledge_graph_v4 
    WHERE base_record_status = 'web_discovery_confirmed'
""").fetchdf()

print("--- THE CONFIRMED DISCOVERIES (For Thesis Appendix) ---")
display(discoveries)

# Extract the rows where the web successfully enriched existing MusicBrainz data
enriched = con.execute("""
    SELECT main_artist_name, release_title, enrichment_source 
    FROM thesis_final_knowledge_graph_v4 
    WHERE base_record_status = 'web_enriched_existing'
""").fetchdf()

print("\n--- WEB-ENRICHED EXISTING RECORDS ---")
display(enriched.head(10))

In [ ]:
import duckdb
import pandas as pd

# Connect to the thesis database
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

print("Applying advanced normalization (Removing leading 'The', 'A', 'An')...")

# -------------------------------------------------------------------
# STEP 1: Update the Cleaning Logic
# -------------------------------------------------------------------
# Logic:
# ^(the|a|an)\s+ : Matches 'the', 'a', or 'an' only at the START of the string.
# We apply this to both Artist and Album keys.
con.execute("""
    CREATE OR REPLACE TABLE wdc_cleaned_for_matching AS
    SELECT 
        source_url,
        wt_artist,
        wt_album,
        wt_track,
        
        -- Normalized Artist Key: Lowercase -> Strip leading articles -> Strip symbols
        trim(regexp_replace(
            regexp_replace(lower(wt_artist), '^(the|a|an)\\\\s+', '', 'g'), 
            '[^a-z0-9]+', ' ', 'g'
        )) AS wt_artist_key,
        
        -- Normalized Album Key: Lowercase -> Strip leading articles -> Strip symbols
        trim(regexp_replace(
            regexp_replace(lower(wt_album), '^(the|a|an)\\\\s+', '', 'g'), 
            '[^a-z0-9]+', ' ', 'g'
        )) AS wt_album_key,
        
        -- Normalized Track Key: Lowercase -> Strip feat/parents -> Strip symbols
        trim(regexp_replace(
            regexp_replace(lower(wt_track), '\\\\(feat.*\\\\)|feat\\\\..*', '', 'g'), 
            '[^a-z0-9]+', ' ', 'g'
        )) AS wt_track_key
    FROM wdc_flattened_music_metadata
""")

# -------------------------------------------------------------------
# STEP 2: Re-run the Exact Match (Known Entities)
# -------------------------------------------------------------------
# Note: We must also apply this logic to the Master Keys to ensure a match!
con.execute("""
    CREATE OR REPLACE VIEW master_discovery_keys_v2 AS
    SELECT
        base_record_id,
        base_record_status,
        COALESCE(main_artist_name, external_artist) AS resolved_artist,
        COALESCE(release_title, external_title) AS resolved_title,
        
        -- Apply the same 'The' stripping to the Master database
        trim(regexp_replace(
            regexp_replace(lower(COALESCE(main_artist_name, external_artist)), '^(the|a|an)\\\\s+', '', 'g'), 
            '[^a-z0-9]+', ' ', 'g'
        )) AS master_artist_key,
        
        trim(regexp_replace(
            regexp_replace(lower(COALESCE(release_title, external_title)), '^(the|a|an)\\\\s+', '', 'g'), 
            '[^a-z0-9]+', ' ', 'g'
        )) AS master_album_key
    FROM music_enrichment_base_for_webtables_v1
""")

con.execute("""
    CREATE OR REPLACE TABLE wdc_master_exact_matches AS
    SELECT
        w.*,
        m.base_record_id,
        m.base_record_status,
        m.resolved_artist AS master_artist,
        m.resolved_title AS master_album
    FROM wdc_cleaned_for_matching w
    JOIN master_discovery_keys_v2 m
      ON w.wt_artist_key = m.master_artist_key
     AND w.wt_album_key = m.master_album_key
    WHERE w.wt_artist_key <> '' AND w.wt_album_key <> ''
""")

# -------------------------------------------------------------------
# STEP 3: Re-isolate Novel Candidates
# -------------------------------------------------------------------
con.execute("""
    CREATE OR REPLACE TABLE wdc_true_novel_candidates AS
    SELECT w.*
    FROM wdc_cleaned_for_matching w
    LEFT JOIN wdc_master_exact_matches e
      ON w.source_url = e.source_url 
     AND w.wt_artist = e.wt_artist
     AND coalesce(w.wt_track, 'NULL') = coalesce(e.wt_track, 'NULL')
    WHERE e.base_record_id IS NULL
      AND w.wt_artist_key <> '' AND w.wt_album_key <> ''
""")

# -------------------------------------------------------------------
# RESULTS
# -------------------------------------------------------------------
known = con.execute("SELECT COUNT(*) FROM wdc_master_exact_matches").fetchone()[0]
novel = con.execute("SELECT COUNT(*) FROM wdc_true_novel_candidates").fetchone()[0]

print(f"\n✅ Re-Cleaning Complete.")
print(f"🔗 New Known Entities (after 'The' stripping): {known}")
print(f"🚀 New True Novel Candidates: {novel}")

# Show examples of the improved keys
display(con.execute("""
    SELECT wt_artist, wt_artist_key, wt_album, wt_album_key 
    FROM wdc_cleaned_for_matching 
    WHERE wt_artist ILIKE 'The %' 
    LIMIT 5
""").fetchdf())

In [ ]:
import duckdb
import pandas as pd

# Connect to the thesis database
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

print("Fixing Normalization: Ensuring 'The/A/An' are actually removed...")

# -------------------------------------------------------------------
# STEP 1: Improved Cleaning Logic
# -------------------------------------------------------------------
# We use '(?i)' for case-insensitivity and ensure the regex 
# handles the start of the string correctly.
con.execute("""
    CREATE OR REPLACE TABLE wdc_cleaned_for_matching AS
    SELECT 
        source_url,
        wt_artist,
        wt_album,
        wt_track,
        
        -- Improved Artist Key: Strip leading articles -> Remove non-alphanumeric
        trim(regexp_replace(
            regexp_replace(lower(wt_artist), '^(the |a |an )', '', 'g'), 
            '[^a-z0-9]+', ' ', 'g'
        )) AS wt_artist_key,
        
        -- Improved Album Key
        trim(regexp_replace(
            regexp_replace(lower(wt_album), '^(the |a |an )', '', 'g'), 
            '[^a-z0-9]+', ' ', 'g'
        )) AS wt_album_key,
        
        -- Track Key (kept the same as it was working well)
        trim(regexp_replace(
            regexp_replace(lower(wt_track), '\\(feat.*\\)|feat\\..*', '', 'g'), 
            '[^a-z0-9]+', ' ', 'g'
        )) AS wt_track_key
    FROM wdc_flattened_music_metadata
""")

# -------------------------------------------------------------------
# STEP 2: Apply same logic to Master View
# -------------------------------------------------------------------
con.execute("""
    CREATE OR REPLACE VIEW master_discovery_keys_v2 AS
    SELECT
        base_record_id,
        base_record_status,
        COALESCE(main_artist_name, external_artist) AS resolved_artist,
        COALESCE(release_title, external_title) AS resolved_title,
        
        trim(regexp_replace(
            regexp_replace(lower(COALESCE(main_artist_name, external_artist)), '^(the |a |an )', '', 'g'), 
            '[^a-z0-9]+', ' ', 'g'
        )) AS master_artist_key,
        
        trim(regexp_replace(
            regexp_replace(lower(COALESCE(release_title, external_title)), '^(the |a |an )', '', 'g'), 
            '[^a-z0-9]+', ' ', 'g'
        )) AS master_album_key
    FROM music_enrichment_base_for_webtables_v1
""")

# -------------------------------------------------------------------
# STEP 3: The Match (This should now catch more rows!)
# -------------------------------------------------------------------
con.execute("""
    CREATE OR REPLACE TABLE wdc_master_exact_matches AS
    SELECT
        w.*,
        m.base_record_id,
        m.base_record_status,
        m.resolved_artist AS master_artist,
        m.resolved_title AS master_album
    FROM wdc_cleaned_for_matching w
    JOIN master_discovery_keys_v2 m
      ON w.wt_artist_key = m.master_artist_key
     AND w.wt_album_key = m.master_album_key
    WHERE w.wt_artist_key <> '' AND w.wt_album_key <> ''
""")

# -------------------------------------------------------------------
# STEP 4: Novel Candidates
# -------------------------------------------------------------------
con.execute("""
    CREATE OR REPLACE TABLE wdc_true_novel_candidates AS
    SELECT w.*
    FROM wdc_cleaned_for_matching w
    LEFT JOIN wdc_master_exact_matches e
      ON w.source_url = e.source_url 
     AND w.wt_artist = e.wt_artist
     AND coalesce(w.wt_track, 'NULL') = coalesce(e.wt_track, 'NULL')
    WHERE e.base_record_id IS NULL
      AND w.wt_artist_key <> '' AND w.wt_album_key <> ''
""")

# -------------------------------------------------------------------
# FINAL VERIFICATION
# -------------------------------------------------------------------
known = con.execute("SELECT COUNT(*) FROM wdc_master_exact_matches").fetchone()[0]
novel = con.execute("SELECT COUNT(*) FROM wdc_true_novel_candidates").fetchone()[0]

print(f"\n✅ Logic Fixed.")
print(f"🔗 Known Entities (Exact Matches): {known}")
print(f"🚀 True Novel Candidates: {novel}")

# Check Row 0 specifically
print("\n--- Verification Table (Key should NOT contain 'the') ---")
display(con.execute("""
    SELECT wt_artist, wt_artist_key 
    FROM wdc_cleaned_for_matching 
    WHERE wt_artist ILIKE 'The %' 
    LIMIT 5
""").fetchdf())

In [ ]:
import duckdb
import pandas as pd
from rapidfuzz import fuzz, process

# Connect to the thesis database
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# -------------------------------------------------------------------
# STEP 1: Fuzzy Validation of the 20 Novel Candidates
# -------------------------------------------------------------------
print("Validating the 20 remaining novel candidates using fuzzy logic...")

# Get the artists from our 20 candidates
novel_artists = con.execute("SELECT DISTINCT wt_artist FROM wdc_true_novel_candidates").fetchdf()['wt_artist'].tolist()
first_letters = tuple(set([a[0].lower() for a in novel_artists if a]))

# Fetch MB artists in the same alpha-blocks from our NEW master view
mb_candidates = con.execute(f"""
    SELECT DISTINCT base_record_id, resolved_artist 
    FROM master_discovery_keys_v2 
    WHERE lower(left(resolved_artist, 1)) IN {first_letters}
""").fetchdf()

mb_artist_list = mb_candidates['resolved_artist'].tolist()
fuzzy_matches = []

for web_artist in novel_artists:
    # We use token_set_ratio because it's the most robust against extra words
    best_match = process.extractOne(web_artist, mb_artist_list, scorer=fuzz.token_set_ratio)
    
    # Threshold 95: Very high certainty
    if best_match and best_match[1] >= 95:
        match_idx = best_match[2]
        fuzzy_matches.append({
            'wt_artist': web_artist,
            'mb_record_id': mb_candidates.iloc[match_idx]['base_record_id']
        })

df_fuzzy = pd.DataFrame(fuzzy_matches)
con.register('df_fuzzy_validation', df_fuzzy)
print(f"🏁 Fuzzy pass complete. Found {len(df_fuzzy)} additional hidden matches.")

# -------------------------------------------------------------------
# STEP 2: Building the Final Thesis Knowledge Graph
# -------------------------------------------------------------------
# This merges everything: MB, CDS, Exact Web Matches, and Fuzzy Web Matches.
print("Building final Knowledge Graph...")

con.execute("""
    CREATE OR REPLACE TABLE thesis_final_knowledge_graph_v5 AS
    WITH validated_web AS (
        SELECT 
            w.*,
            v.mb_record_id AS fuzzy_mb_id,
            CASE 
                WHEN v.mb_record_id IS NOT NULL THEN 'web_enriched_fuzzy'
                ELSE 'web_discovery_confirmed'
            END AS final_status
        FROM wdc_true_novel_candidates w
        LEFT JOIN df_fuzzy_validation v ON w.wt_artist = v.wt_artist
    )
    -- 1. Original Records
    SELECT * FROM music_enrichment_base_for_webtables_v1
    UNION ALL
    -- 2. Exact Web Matches (Now 14 rows)
    SELECT 
        base_record_id, NULL, wt_album, wt_artist, NULL, NULL, NULL, NULL, NULL, 
        wt_artist, wt_album, NULL, NULL, NULL, wt_track, 
        'webtable_exact', 'enriched', source_url, TRUE, 'web_enriched_exact'
    FROM wdc_master_exact_matches
    UNION ALL
    -- 3. The Novel Discoveries and Fuzzy Matches
    SELECT 
        COALESCE(fuzzy_mb_id, (30000000 + row_number() OVER ())) AS base_record_id,
        NULL, wt_album, wt_artist, NULL, NULL, NULL, NULL, NULL, 
        wt_artist, wt_album, NULL, NULL, NULL, wt_track, 
        'webtable_fuzzy_or_discovery', 'final_pass', source_url, (fuzzy_mb_id IS NOT NULL), final_status
    FROM validated_web;
""")

# -------------------------------------------------------------------
# FINAL REPORT
# -------------------------------------------------------------------
print("\n" + "="*40)
print("FINAL THESIS ENRICHMENT REPORT (REFINED)")
print("="*40)
report = con.execute("SELECT base_record_status, COUNT(*) as total FROM thesis_final_knowledge_graph_v5 GROUP BY 1").fetchdf()
display(report)

In [ ]:
import duckdb
import pandas as pd

# Connect to the database
con = duckdb.connect("../data_processed/thesis.duckdb")

# -------------------------------------------------------------------
# 1. FETCH THE TRUE DISCOVERIES
# -------------------------------------------------------------------
# These are the records that MusicBrainz has no record of!
discoveries_df = con.execute("""
    SELECT 
        main_artist_name as Artist, 
        release_title as Album, 
        external_tracklist as Track,
        enrichment_source as Source_URL
    FROM thesis_final_knowledge_graph_v4 
    WHERE base_record_status = 'web_discovery_confirmed'
""").fetchdf()

# -------------------------------------------------------------------
# 2. FETCH THE FUZZY ENRICHMENTS (Sample)
# -------------------------------------------------------------------
# These are cases where the web had a "messy" version of a known name
fuzzy_df = con.execute("""
    SELECT 
        main_artist_name as Web_Artist, 
        release_title as Album,
        enrichment_source as Source_URL
    FROM thesis_final_knowledge_graph_v4 
    WHERE base_record_status = 'web_enriched_fuzzy'
    LIMIT 10
""").fetchdf()

print("="*60)
print("🚀 THE CONFIRMED DISCOVERIES (New Knowledge Found)")
print("="*60)
if not discoveries_df.empty:
    display(discoveries_df)
else:
    print("No confirmed discoveries found in this sample.")

print("\n" + "="*60)
print("🔗 FUZZY ENRICHMENT EXAMPLES (Data Resolved)")
print("="*60)
display(fuzzy_df)

In [ ]:
# Export the discoveries to CSV for your thesis folder
discoveries_df.to_csv("../data_processed/final_confirmed_discoveries.csv", index=False)
print("✅ Discoveries exported to CSV for your appendix!")

In [ ]:
import duckdb
import os
import pandas as pd

# 1. Connect to your database
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# 2. Define the export directory
export_path = "../data_exports"

# Create the folder if it doesn't exist
if not os.path.exists(export_path):
    os.makedirs(export_path)
    print(f"Created folder: {export_path} 📁")

# -------------------------------------------------------------------
# 3. Define the "Core" Tables to Export
# -------------------------------------------------------------------
# We are exporting 4 tables that tell the story of your integration:
# - exact_matches: High-confidence links (14 rows)
# - novel_candidates: The "Unknowns" we started with (20 rows)
# - confirmed_discoveries: Truly new music (2 rows)
# - final_knowledge_graph: The master database (~5M+ rows)
# -------------------------------------------------------------------

tables_to_export = {
    "wdc_exact_matches": "wdc_master_exact_matches",
    "wdc_novel_candidates": "wdc_true_novel_candidates",
    "thesis_final_discoveries": "SELECT * FROM thesis_final_knowledge_graph_v5 WHERE base_record_status = 'web_discovery_confirmed'",
    "thesis_final_master_graph": "thesis_final_knowledge_graph_v5"
}

print("\nStarting export to CSV...")

for file_name, table_query in tables_to_export.items():
    full_path = os.path.join(export_path, f"{file_name}.csv")
    
    # If the entry is a table name, we select all. If it's a query, we run the query.
    query = f"SELECT * FROM ({table_query})" if " " in table_query else f"SELECT * FROM {table_query}"
    
    # Use DuckDB's high-speed COPY command to write the CSV
    try:
        con.execute(f"COPY ({query}) TO '{full_path}' (HEADER, DELIMITER ',')")
        print(f"✅ Exported: {file_name}.csv")
    except Exception as e:
        print(f"❌ Failed to export {file_name}: {e}")

print(f"\n🚀 All important tables are now safe in: {os.path.abspath(export_path)}")

In [ ]:
import os
import requests
import tarfile

# -------------------------------------------------------------------
# STEP 1: Configuration
# -------------------------------------------------------------------
# The URL must be wrapped in quotation marks!
download_url = "https://data.dws.informatik.uni-mannheim.de/webtables/2015-07/englishCorpus/compressed/00.tar.gz" 

# Define where the data should live
target_dir = "../data_raw/wdc/"
extraction_directory = os.path.join(target_dir, "english_relational_2015")

# Ensure our folders exist
os.makedirs(target_dir, exist_ok=True)
os.makedirs(extraction_directory, exist_ok=True)

# Extract the filename from the URL (e.g., '00.tar.gz')
file_name = download_url.split('/')[-1]
tar_file_path = os.path.join(target_dir, file_name)

# -------------------------------------------------------------------
# STEP 2: Safe Chunked Download
# -------------------------------------------------------------------
print(f"🌍 Connecting to Web Data Commons...")
print(f"📥 Downloading {file_name} into {target_dir}...")

try:
    # Use the variable we defined at the top
    response = requests.get(download_url, stream=True)
    response.raise_for_status() # Check for HTTP errors
    
    # Open the file and write it in 8KB chunks
    with open(tar_file_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)
    print("✅ Download successfully completed!")

# -------------------------------------------------------------------
# STEP 3: Automatic Extraction
# -------------------------------------------------------------------
    print(f"\n📦 Now extracting {file_name}...")
    
    # We use "r:*" so Python automatically handles the .gz compression
    with tarfile.open(tar_file_path, "r:*") as tar:
        tar.extractall(path=extraction_directory)
    
    print(f"✅ Extraction complete!")
    print(f"📁 Your production data is ready in: {extraction_directory}")

except requests.exceptions.RequestException as e:
    print(f"❌ Network Error during download: {e}")
except tarfile.TarError as e:
    print(f"❌ Error during extraction: {e}")
except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")

In [ ]:
import tarfile
import os

# The exact path from your screenshot
tar_path = "../data_raw/wdc/english_relational_2015/00.tar"
extract_path = "../data_raw/wdc/english_relational_2015/"

print(f"📦 Unpacking the massive archive: {tar_path}")
print("⏱️ This is a 4.8 GB file, so it might take 2-5 minutes to extract...")

try:
    # We use "r" because the file is .tar (not .gz)
    with tarfile.open(tar_path, "r") as tar:
        tar.extractall(path=extract_path)
    
    print("✅ Extraction complete! The JSON files are now free.")
    
    # Let's count how many files we just unpacked
    import glob
    json_files = glob.glob(f"{extract_path}/**/*.json*", recursive=True)
    print(f"📊 We just extracted {len(json_files):,} JSON files!")

except Exception as e:
    print(f"❌ Something went wrong: {e}")

In [ ]:
import tarfile
import os

# The exact path from your screenshot
tar_path = "../data_raw/wdc/english_relational_2015/00.tar"
extract_path = "../data_raw/wdc/english_relational_2015/"

print(f"📦 Unpacking the massive archive: {tar_path}")
print("⏱️ This is a 4.8 GB file, so it might take 2-5 minutes to extract...")

try:
    # We use "r" because the file is .tar (not .gz)
    with tarfile.open(tar_path, "r") as tar:
        tar.extractall(path=extract_path)
    
    print("✅ Extraction complete! The JSON files are now free.")
    
    # Let's count how many files we just unpacked
    import glob
    json_files = glob.glob(f"{extract_path}/**/*.json*", recursive=True)
    print(f"📊 We just extracted {len(json_files):,} JSON files!")

except Exception as e:
    print(f"❌ Something went wrong: {e}")

In [5]:
import os
import tarfile
import json
import duckdb
import pandas as pd

pd.set_option('display.max_colwidth', None)

TAR_PATH = "../data_raw/wdc/english_relational_2015/00.tar"
DB_PATH  = "../data_processed/thesis.duckdb"

# ── Music filter config ─────────────────────────────────────────
STRONG_MUSIC_HEADERS = {
    "artist", "album", "track", "song", "genre",
    "label", "band", "singer", "record label"
}
MUSIC_PAGE_WORDS = {
    "playlist", "discography", "album", "tracklist", "band",
    "artist", "lyrics", "setlist", "radio", "music", "release", "chart"
}
BLOCKED_DOMAINS = {
    "anime-planet.com", "myanimelist.net", "imdb.com",
    "goodreads.com", "letterboxd.com", "amazon.com",
    "ebay.com", "walmart.com"
}
ARTIST_ALIASES = {"artist", "band", "singer", "performer", "artist name", "act"}
ALBUM_ALIASES  = {"album", "release", "album title", "record", "album name"}
TRACK_ALIASES  = {"track", "song", "song title", "track title", "track name"}
GENRE_ALIASES  = {"genre", "style", "music genre"}
YEAR_ALIASES   = {"year", "release year"}
LABEL_ALIASES  = {"label", "record label"}

def get_domain(url):
    try:
        return url.split("/")[2].replace("www.", "")
    except:
        return ""

def get_headers(t):
    if not t.get("hasHeader"):
        return []
    if t.get("tableOrientation") != "HORIZONTAL":
        return []
    relation = t.get("relation", [])
    hdr_idx  = t.get("headerRowIndex", 0)
    return [
        col[hdr_idx].strip().lower()
        for col in relation
        if hdr_idx < len(col) and col[hdr_idx].strip()
    ]

def is_music_table(t):
    domain = get_domain(t.get("url", ""))
    if any(b in domain for b in BLOCKED_DOMAINS):
        return False
    headers     = set(get_headers(t))
    strong_hits = headers & STRONG_MUSIC_HEADERS
    page_title  = t.get("pageTitle", "").lower()
    title_hit   = any(w in page_title for w in MUSIC_PAGE_WORDS)
    return len(strong_hits) >= 1 and (len(strong_hits) >= 2 or title_hit)

def map_col(headers, aliases):
    for i, h in enumerate(headers):
        if h in aliases:
            return i
    return None

def safe_cell(relation, col_idx, row_idx):
    if col_idx is None:
        return None
    col = relation[col_idx]
    if row_idx < len(col):
        v = col[row_idx].strip()
        return v if v else None
    return None

# ════════════════════════════════════════════════════════════════
# STEP 1 — Connect and create the table FIRST (persistent schema)
#           This survives kernel restarts — rows go straight to disk
# ════════════════════════════════════════════════════════════════
con = duckdb.connect(DB_PATH)
print("Connected ✅")

con.execute("""
    CREATE OR REPLACE TABLE wdc_flattened_music_metadata (
        source_url  VARCHAR,
        page_title  VARCHAR,
        domain      VARCHAR,
        wt_artist   VARCHAR,
        wt_album    VARCHAR,
        wt_track    VARCHAR,
        wt_genre    VARCHAR,
        wt_year     VARCHAR,
        wt_label    VARCHAR
    )
""")
print("✅ wdc_flattened_music_metadata created (empty, correct schema)")

# ════════════════════════════════════════════════════════════════
# STEP 2 — Stream tar → parse → insert in batches of 5,000 rows
#           Batching avoids one INSERT per row (too slow) while
#           keeping memory usage low (no giant pandas DataFrame)
# ════════════════════════════════════════════════════════════════
print(f"\nStreaming from {TAR_PATH} …")

BATCH_SIZE   = 5_000
REPORT_EVERY = 50_000

batch        = []
total_files  = music_found = rows_inserted = 0

def flush_batch(con, batch):
    """Insert current batch into DuckDB and return empty list."""
    if not batch:
        return []
    df = pd.DataFrame(batch, columns=[
        "source_url","page_title","domain",
        "wt_artist","wt_album","wt_track",
        "wt_genre","wt_year","wt_label"
    ])
    con.register("_batch_tmp", df)
    con.execute("INSERT INTO wdc_flattened_music_metadata SELECT * FROM _batch_tmp")
    con.unregister("_batch_tmp")
    return []

with tarfile.open(TAR_PATH, "r") as tar:
    for member in tar:
        if not member.isfile() or not member.name.endswith(".json"):
            continue

        total_files += 1

        if total_files % REPORT_EVERY == 0:
            # Flush whatever is in the batch before reporting
            batch = flush_batch(con, batch)
            rows_inserted = con.execute(
                "SELECT COUNT(*) FROM wdc_flattened_music_metadata"
            ).fetchone()[0]
            print(f"  Scanned {total_files:>8,} files | "
                  f"music tables: {music_found:,} | "
                  f"rows in DB: {rows_inserted:,}")

        try:
            f   = tar.extractfile(member)
            if f is None:
                continue
            raw = f.read().decode("utf-8", errors="replace").strip()
            if not raw:
                continue
            t   = json.loads(raw)
        except Exception:
            continue

        if not is_music_table(t):
            continue

        music_found += 1
        headers  = get_headers(t)
        relation = t.get("relation", [])
        hdr_idx  = t.get("headerRowIndex", 0)
        url      = t.get("url", "")
        domain   = get_domain(url)

        if not headers or not relation:
            continue

        i_artist = map_col(headers, ARTIST_ALIASES)
        i_album  = map_col(headers, ALBUM_ALIASES)
        i_track  = map_col(headers, TRACK_ALIASES)
        i_genre  = map_col(headers, GENRE_ALIASES)
        i_year   = map_col(headers, YEAR_ALIASES)
        i_label  = map_col(headers, LABEL_ALIASES)

        if i_artist is None and i_album is None and i_track is None:
            continue

        n_rows = max(len(c) for c in relation)

        for r in range(hdr_idx + 1, n_rows):
            artist = safe_cell(relation, i_artist, r)
            album  = safe_cell(relation, i_album,  r)
            track  = safe_cell(relation, i_track,  r)
            genre  = safe_cell(relation, i_genre,  r)
            year   = safe_cell(relation, i_year,   r)
            label  = safe_cell(relation, i_label,  r)

            if not any([artist, album, track]):
                continue

            batch.append((
                url, t.get("pageTitle",""), domain,
                artist, album, track,
                str(genre) if genre else None,
                str(year)  if year  else None,
                label
            ))

        # Flush every BATCH_SIZE rows
        if len(batch) >= BATCH_SIZE:
            batch = flush_batch(con, batch)

# Final flush for remainder
batch = flush_batch(con, batch)

n_flat = con.execute(
    "SELECT COUNT(*) FROM wdc_flattened_music_metadata"
).fetchone()[0]

print(f"\n── Streaming complete ──")
print(f"  Total JSON files scanned : {total_files:,}")
print(f"  Music tables found       : {music_found:,}")
print(f"  Rows saved to DuckDB     : {n_flat:,}")

# ════════════════════════════════════════════════════════════════
# STEP 3 — Build normalized matching keys
# ════════════════════════════════════════════════════════════════
print("\nBuilding wdc_cleaned_for_matching …")

con.execute(r"""
    CREATE OR REPLACE TABLE wdc_cleaned_for_matching AS
    SELECT
        source_url, page_title, domain,
        wt_artist, wt_album, wt_track,
        wt_genre, wt_year, wt_label,

        trim(regexp_replace(lower(coalesce(wt_artist, '')),
            '[^a-z0-9]+', ' ', 'g')) AS wt_artist_key,

        trim(regexp_replace(lower(coalesce(wt_album, '')),
            '[^a-z0-9]+', ' ', 'g')) AS wt_album_key,

        trim(regexp_replace(
            regexp_replace(lower(coalesce(wt_track, '')),
                '\(feat[^)]*\)|feat\.?\s.*', '', 'g'),
            '[^a-z0-9]+', ' ', 'g'
        )) AS wt_track_key

    FROM wdc_flattened_music_metadata
    WHERE
        (wt_artist IS NULL OR wt_artist NOT LIKE '%Ã%')
        AND (wt_artist IS NULL OR wt_artist NOT LIKE '%â%')
        AND (
            length(trim(regexp_replace(lower(coalesce(wt_artist,'')),
                '[^a-z0-9]+', ' ', 'g'))) >= 2
            OR
            length(trim(regexp_replace(lower(coalesce(wt_album, '')),
                '[^a-z0-9]+', ' ', 'g'))) >= 2
        )
""")

n_clean = con.execute(
    "SELECT COUNT(*) FROM wdc_cleaned_for_matching"
).fetchone()[0]
print(f"✅ wdc_cleaned_for_matching: {n_clean:,} rows")

# ════════════════════════════════════════════════════════════════
# STEP 4 — Quality report
# ════════════════════════════════════════════════════════════════
print("\n── Field coverage ──")
print(con.execute("""
    SELECT
        COUNT(*)             AS total_rows,
        COUNT(wt_artist)     AS has_artist,
        COUNT(wt_album)      AS has_album,
        COUNT(wt_track)      AS has_track,
        COUNT(wt_genre)      AS has_genre,
        COUNT(wt_label)      AS has_label,
        COUNT(CASE WHEN length(wt_artist_key) >= 2
                    AND length(wt_album_key)  >= 2
               THEN 1 END)  AS matchable_rows
    FROM wdc_cleaned_for_matching
""").fetchdf().to_string(index=False))

print("\n── Top 15 domains ──")
print(con.execute("""
    SELECT domain, COUNT(*) AS rows
    FROM wdc_cleaned_for_matching
    GROUP BY domain
    ORDER BY rows DESC
    LIMIT 15
""").fetchdf().to_string(index=False))

print("\n── Normalization preview ──")
display(con.execute("""
    SELECT
        wt_artist, wt_artist_key,
        wt_album,  wt_album_key,
        wt_track,  wt_track_key
    FROM wdc_cleaned_for_matching
    WHERE wt_artist IS NOT NULL AND wt_album IS NOT NULL
    LIMIT 10
""").fetchdf())

print("""
✅ Data is now fully persisted in DuckDB.
   Kernel restarts will no longer lose anything.
   Next step → run the exact + fuzzy matching pipeline.
""")

IOException: IO Error: Cannot open file "c:\users\user\onedrive\desktop\eman_thesis\jupyter\..\data_processed\thesis.duckdb": The process cannot access the file because it is being used by another process.

File is already open in 
C:\Python312\python.exe (PID 18140)

In [ ]:
import duckdb
con = duckdb.connect("../data_processed/thesis.duckdb")

# Verify your data survived
print(con.execute("SELECT COUNT(*) FROM wdc_flattened_music_metadata").fetchone()[0])
# Should print: 164,923

print(con.execute("SELECT COUNT(*) FROM wdc_cleaned_for_matching").fetchone()[0])
# Should print: 161,458

In [ ]:
import duckdb
import pandas as pd

pd.set_option('display.max_colwidth', None)
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ════════════════════════════════════════════════════════════════
# STEP A — Rebuild MB discovery keys from the enriched base table
#           (same as before — just making sure it exists)
# ════════════════════════════════════════════════════════════════
con.execute("""
    CREATE OR REPLACE VIEW mb_release_discovery_keys AS
    SELECT
        base_record_id,
        release_id,
        base_record_status,
        is_linked_to_musicbrainz,
        COALESCE(main_artist_name, external_artist)      AS artist_name,
        COALESCE(release_title,    external_title)       AS album_name,
        CAST(external_category AS VARCHAR)               AS cds_category,
        CAST(external_genre    AS VARCHAR)               AS cds_genre,
        CAST(enrichment_source AS VARCHAR)               AS cds_enrichment_source,
        barcode,
        total_tracks,
        mb_year,

        trim(regexp_replace(
            lower(COALESCE(main_artist_name, external_artist, '')),
            '[^a-z0-9]+', ' ', 'g')) AS mb_artist_key,

        trim(regexp_replace(
            lower(COALESCE(release_title, external_title, '')),
            '[^a-z0-9]+', ' ', 'g')) AS mb_album_key

    FROM music_enrichment_base_for_webtables_v1
    WHERE COALESCE(main_artist_name, external_artist) IS NOT NULL
      AND COALESCE(release_title,    external_title)  IS NOT NULL
""")
print("✅ mb_release_discovery_keys ready")

# ════════════════════════════════════════════════════════════════
# STEP A — Exact matching with deduplication
# ════════════════════════════════════════════════════════════════
print("Running exact matching …")

con.execute("""
    CREATE OR REPLACE TABLE wdc_mb_exact_matches AS
    WITH all_matches AS (
        SELECT
            w.source_url,
            w.page_title,
            w.domain,
            w.wt_artist,
            w.wt_album,
            w.wt_track,
            CAST(w.wt_genre AS VARCHAR) AS wt_genre,
            CAST(w.wt_year  AS VARCHAR) AS wt_year,
            CAST(w.wt_label AS VARCHAR) AS wt_label,
            m.base_record_id,
            m.release_id,
            m.base_record_status,
            m.is_linked_to_musicbrainz,
            m.artist_name  AS mb_artist,
            m.album_name   AS mb_album,
            m.cds_category,
            m.cds_genre,
            m.cds_enrichment_source,
            m.total_tracks AS mb_total_tracks,
            m.barcode      AS mb_barcode,
            m.mb_year
        FROM wdc_cleaned_for_matching w
        JOIN mb_release_discovery_keys m
          ON w.wt_artist_key = m.mb_artist_key
         AND w.wt_album_key  = m.mb_album_key
        WHERE length(w.wt_artist_key) >= 2
          AND length(w.wt_album_key)  >= 2
    ),
    ranked AS (
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY source_url, wt_artist, wt_album
                ORDER BY
                    CASE WHEN is_linked_to_musicbrainz = TRUE  THEN 0 ELSE 1 END,
                    CASE WHEN mb_barcode IS NOT NULL
                              AND trim(mb_barcode) <> ''        THEN 0 ELSE 1 END,
                    CASE WHEN mb_total_tracks BETWEEN 6 AND 40  THEN 0 ELSE 1 END,
                    ABS(mb_total_tracks - 12) ASC NULLS LAST,
                    release_id ASC NULLS LAST
            ) AS rn
        FROM all_matches
    )
    SELECT * EXCLUDE (rn)
    FROM ranked
    WHERE rn = 1
""")

n_exact = con.execute("SELECT COUNT(*) FROM wdc_mb_exact_matches").fetchone()[0]
print(f"✅ wdc_mb_exact_matches: {n_exact:,} rows")

print("\n── Exact match breakdown ──")
print(con.execute("""
    SELECT
        base_record_status,
        COUNT(*)                                             AS matches,
        COUNT(CASE WHEN cds_genre IS NOT NULL
                        AND cds_genre NOT IN ('None','')
                   THEN 1 END)                               AS already_has_cds_genre,
        COUNT(CASE WHEN wt_genre IS NOT NULL
                        AND wt_genre NOT IN ('None','')
                   THEN 1 END)                               AS has_wt_genre
    FROM wdc_mb_exact_matches
    GROUP BY base_record_status
    ORDER BY matches DESC
""").fetchdf().to_string(index=False))

print("\n── Top matched artists (exact) ──")
print(con.execute("""
    SELECT mb_artist, mb_album, wt_genre, cds_genre,
           domain, release_id
    FROM wdc_mb_exact_matches
    ORDER BY mb_artist
    LIMIT 15
""").fetchdf().to_string(index=False))

# ════════════════════════════════════════════════════════════════
# STEP B — Fuzzy matching for the remaining unmatched WDC rows
# ════════════════════════════════════════════════════════════════
print("\nBuilding fuzzy candidate pairs with blocking …")

con.execute("""
    CREATE OR REPLACE TABLE wdc_mb_fuzzy_candidates AS
    SELECT
        w.source_url, w.page_title, w.domain,
        w.wt_artist,  w.wt_album,  w.wt_track,
        CAST(w.wt_genre AS VARCHAR) AS wt_genre,
        CAST(w.wt_year  AS VARCHAR) AS wt_year,
        CAST(w.wt_label AS VARCHAR) AS wt_label,
        w.wt_artist_key, w.wt_album_key,

        m.base_record_id, m.release_id,
        m.base_record_status, m.is_linked_to_musicbrainz,
        m.artist_name  AS mb_artist,
        m.album_name   AS mb_album,
        m.mb_artist_key, m.mb_album_key,
        m.cds_category, m.cds_genre,
        m.total_tracks AS mb_total_tracks,
        m.barcode      AS mb_barcode,
        m.mb_year

    FROM (
        SELECT *,
            substr(wt_artist_key, 1, 1) AS a1,
            substr(wt_album_key,  1, 1) AS t1,
            length(wt_artist_key)       AS alen,
            length(wt_album_key)        AS tlen
        FROM wdc_cleaned_for_matching
        WHERE length(wt_artist_key) >= 2
          AND length(wt_album_key)  >= 2
    ) w

    JOIN (
        SELECT *,
            substr(mb_artist_key, 1, 1) AS a1,
            substr(mb_album_key,  1, 1) AS t1,
            length(mb_artist_key)       AS alen,
            length(mb_album_key)        AS tlen
        FROM mb_release_discovery_keys
        WHERE length(mb_artist_key) >= 2
          AND length(mb_album_key)  >= 2
    ) m
      ON  w.a1   = m.a1
      AND w.t1   = m.t1
      AND m.alen BETWEEN w.alen - 4 AND w.alen + 8
      AND m.tlen BETWEEN w.tlen - 5 AND w.tlen + 5

    -- Skip already exact-matched pairs
    WHERE NOT EXISTS (
        SELECT 1 FROM wdc_mb_exact_matches e
        WHERE e.source_url              = w.source_url
          AND coalesce(e.wt_artist, '') = coalesce(w.wt_artist, '')
          AND coalesce(e.wt_album,  '') = coalesce(w.wt_album,  '')
    )
""")

n_cand = con.execute(
    "SELECT COUNT(*) FROM wdc_mb_fuzzy_candidates"
).fetchone()[0]
print(f"✅ Fuzzy candidate pairs: {n_cand:,}")

# ── Score every candidate pair ───────────────────────────────────
print("Scoring …")

con.execute("""
    CREATE OR REPLACE TABLE wdc_mb_fuzzy_scored AS
    SELECT *,
        ROUND(jaro_winkler_similarity(wt_artist_key, mb_artist_key)*100, 3) AS artist_sim,
        ROUND(jaro_winkler_similarity(wt_album_key,  mb_album_key )*100, 3) AS album_sim,
        levenshtein(wt_artist_key, mb_artist_key) AS artist_lev,
        levenshtein(wt_album_key,  mb_album_key)  AS album_lev,

        ROUND(
            0.45 * jaro_winkler_similarity(wt_artist_key, mb_artist_key)*100
          + 0.55 * jaro_winkler_similarity(wt_album_key,  mb_album_key )*100
          + CASE WHEN lower(trim(coalesce(wt_artist,'')))
                    = lower(trim(coalesce(mb_artist,'')))
                  AND lower(trim(coalesce(wt_album, '')))
                    = lower(trim(coalesce(mb_album, '')))
                 THEN 5 ELSE 0 END
          + CASE WHEN is_linked_to_musicbrainz = TRUE THEN 2 ELSE 0 END
          - CASE WHEN levenshtein(wt_artist_key, mb_artist_key) > 6  THEN 10 ELSE 0 END
          - CASE WHEN levenshtein(wt_album_key,  mb_album_key)  > 8  THEN  8 ELSE 0 END
        , 3) AS final_score

    FROM wdc_mb_fuzzy_candidates
""")

# ── Pick best per WDC row + classify ────────────────────────────
con.execute("""
    CREATE OR REPLACE TABLE wdc_mb_fuzzy_best AS
    WITH ranked AS (
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY source_url, wt_artist, wt_album
                ORDER BY final_score DESC, album_sim DESC,
                         artist_sim DESC, release_id ASC NULLS LAST
            ) AS rn,
            LEAD(final_score) OVER (
                PARTITION BY source_url, wt_artist, wt_album
                ORDER BY final_score DESC
            ) AS second_best_score
        FROM wdc_mb_fuzzy_scored
    )
    SELECT
        * EXCLUDE (rn),
        ROUND(final_score - second_best_score, 3) AS score_gap,
        CASE
            WHEN final_score >= 90 AND album_sim >= 88 THEN 'auto_accept'
            WHEN final_score >= 80 AND album_sim >= 78 THEN 'review'
            ELSE 'reject'
        END AS confidence_class
    FROM ranked
    WHERE rn = 1
""")

print("✅ wdc_mb_fuzzy_best created")

print("\n── Fuzzy confidence breakdown ──")
print(con.execute("""
    SELECT
        confidence_class,
        COUNT(*)                       AS n,
        ROUND(AVG(final_score),  2)    AS avg_score,
        ROUND(AVG(artist_sim),   2)    AS avg_artist_sim,
        ROUND(AVG(album_sim),    2)    AS avg_album_sim
    FROM wdc_mb_fuzzy_best
    GROUP BY confidence_class
    ORDER BY CASE confidence_class
        WHEN 'auto_accept' THEN 1
        WHEN 'review'      THEN 2
        ELSE 3 END
""").fetchdf().to_string(index=False))

print("\n── Auto-accept fuzzy matches (new finds) ──")
display(con.execute("""
    SELECT
        wt_artist,   mb_artist,
        wt_album,    mb_album,
        wt_label,    cds_genre,
        ROUND(artist_sim,  1) AS artist_sim,
        ROUND(album_sim,   1) AS album_sim,
        artist_lev,  album_lev,
        ROUND(final_score, 1) AS score,
        release_id
    FROM wdc_mb_fuzzy_best
    WHERE confidence_class = 'auto_accept'
    ORDER BY final_score DESC
    LIMIT 20
""").fetchdf())

# ════════════════════════════════════════════════════════════════
# STEP C — Merge exact + fuzzy into final WebTable matches table
# ════════════════════════════════════════════════════════════════
con.execute("""
    CREATE OR REPLACE TABLE wdc_mb_final_matches AS

    SELECT
        source_url, domain, wt_artist, wt_album, wt_track,
        wt_genre, wt_year, wt_label,
        release_id, mb_artist, mb_album,
        cds_category, cds_genre,
        base_record_status, is_linked_to_musicbrainz,
        100.0 AS final_score,
        100.0 AS artist_sim,
        100.0 AS album_sim,
        NULL::INTEGER AS artist_lev,
        NULL::INTEGER AS album_lev,
        'exact' AS match_type
    FROM wdc_mb_exact_matches

    UNION ALL

    SELECT
        source_url, domain, wt_artist, wt_album, wt_track,
        wt_genre, wt_year, wt_label,
        release_id, mb_artist, mb_album,
        cds_category, cds_genre,
        base_record_status, is_linked_to_musicbrainz,
        final_score, artist_sim, album_sim,
        artist_lev,  album_lev,
        'fuzzy' AS match_type
    FROM wdc_mb_fuzzy_best
    WHERE confidence_class = 'auto_accept'
""")

n_exact  = con.execute("SELECT COUNT(*) FROM wdc_mb_exact_matches").fetchone()[0]
n_fuzzy  = con.execute("""
    SELECT COUNT(*) FROM wdc_mb_fuzzy_best
    WHERE confidence_class = 'auto_accept'
""").fetchone()[0]
n_final  = con.execute("SELECT COUNT(*) FROM wdc_mb_final_matches").fetchone()[0]

# ════════════════════════════════════════════════════════════════
# STEP D — Thesis report numbers
# ════════════════════════════════════════════════════════════════
print("\n" + "═"*60)
print("  THESIS REPORT: WEBTABLES ENRICHMENT SUMMARY")
print("═"*60)
print(f"  WDC rows scanned (00.tar)         : 161,458")
print(f"  Exact matches → MB                : {n_exact:,}")
print(f"  Fuzzy auto-accept added           : {n_fuzzy:,}")
print(f"  Total WDC→MB matches              : {n_final:,}")
print("═"*60)

print("\n── Cross-source corroboration (CDS + WebTables) ──")
display(con.execute("""
    SELECT
        mb_artist,
        mb_album,
        cds_genre                    AS genre_from_cds,
        wt_genre                     AS genre_from_wt,
        wt_label                     AS label_from_wt,
        CASE
            WHEN cds_genre IS NULL OR wt_genre IS NULL
                OR cds_genre IN ('None','') OR wt_genre IN ('None','')
                THEN 'one source only'
            WHEN lower(cds_genre) = lower(wt_genre) THEN '✅ agree'
            ELSE '⚠️  differ'
        END                          AS genre_agreement,
        match_type,
        ROUND(final_score, 1)        AS score,
        release_id
    FROM wdc_mb_final_matches
    WHERE cds_genre IS NOT NULL AND cds_genre NOT IN ('None','')
       OR wt_genre  IS NOT NULL AND wt_genre  NOT IN ('None','')
    ORDER BY genre_agreement, final_score DESC
    LIMIT 25
""").fetchdf())

print("\n── Score distribution ──")
print(con.execute("""
    SELECT
        CASE
            WHEN final_score = 100   THEN '100 (exact)'
            WHEN final_score >= 90   THEN '90-99'
            WHEN final_score >= 80   THEN '80-89'
            ELSE 'below 80'
        END AS score_bucket,
        COUNT(*) AS n
    FROM wdc_mb_final_matches
    GROUP BY score_bucket
    ORDER BY score_bucket DESC
""").fetchdf().to_string(index=False))

In [ ]:
import duckdb
import pandas as pd

pd.set_option('display.max_colwidth', None)

DB_PATH = "../data_processed/thesis.duckdb"

# ════════════════════════════════════════════════════════════════
# FIX: Close any existing connection and reopen fresh
#      This clears all locked transactions from the interrupt
# ════════════════════════════════════════════════════════════════
try:
    con.close()
    print("Old connection closed ✅")
except:
    pass

con = duckdb.connect(DB_PATH)
print("Fresh connection opened ✅")

# Verify our data is still intact
n_wdc  = con.execute("SELECT COUNT(*) FROM wdc_cleaned_for_matching").fetchone()[0]
n_exact = con.execute("SELECT COUNT(*) FROM wdc_mb_exact_matches").fetchone()[0]
print(f"  wdc_cleaned_for_matching : {n_wdc:,} rows  ✅")
print(f"  wdc_mb_exact_matches     : {n_exact:,} rows  ✅")

# ════════════════════════════════════════════════════════════════
# STEP 1 — Drop the locked table explicitly to release the lock
# ════════════════════════════════════════════════════════════════
con.execute("DROP TABLE IF EXISTS wdc_mb_fuzzy_candidates")
con.execute("DROP TABLE IF EXISTS wdc_mb_fuzzy_best")
print("✅ Stale tables dropped — locks cleared")

# ════════════════════════════════════════════════════════════════
# STEP 2 — Build exact match lookup (small, fast)
# ════════════════════════════════════════════════════════════════
con.execute("""
    CREATE OR REPLACE TABLE exact_matched_keys AS
    SELECT DISTINCT
        source_url,
        coalesce(wt_artist, '') AS wt_artist,
        coalesce(wt_album,  '') AS wt_album
    FROM wdc_mb_exact_matches
""")
n_keys = con.execute("SELECT COUNT(*) FROM exact_matched_keys").fetchone()[0]
print(f"✅ Exact match lookup: {n_keys:,} rows")

# ════════════════════════════════════════════════════════════════
# STEP 3 — Score + rank in ONE pass, skipping exact matches
#           No intermediate wdc_mb_fuzzy_candidates table written
#           This avoids the 134M-row intermediate table entirely
# ════════════════════════════════════════════════════════════════
print("\nRunning fuzzy scoring in one pass (this takes 10–20 min) …")

con.execute("""
    CREATE OR REPLACE TABLE wdc_mb_fuzzy_best AS

    WITH

    -- WDC rows that were NOT already exact-matched
    wdc_unmatched AS (
        SELECT
            w.source_url, w.page_title, w.domain,
            w.wt_artist,  w.wt_album,  w.wt_track,
            CAST(w.wt_genre AS VARCHAR) AS wt_genre,
            CAST(w.wt_year  AS VARCHAR) AS wt_year,
            CAST(w.wt_label AS VARCHAR) AS wt_label,
            w.wt_artist_key,
            w.wt_album_key,
            substr(w.wt_artist_key, 1, 1) AS a1,
            substr(w.wt_album_key,  1, 1) AS t1,
            length(w.wt_artist_key)       AS alen,
            length(w.wt_album_key)        AS tlen
        FROM wdc_cleaned_for_matching w
        LEFT JOIN exact_matched_keys e
          ON  w.source_url             = e.source_url
          AND coalesce(w.wt_artist,'') = e.wt_artist
          AND coalesce(w.wt_album, '') = e.wt_album
        WHERE e.source_url IS NULL
          AND length(w.wt_artist_key) >= 2
          AND length(w.wt_album_key)  >= 2
    ),

    -- MB keys with blocking columns pre-computed
    mb_keys AS (
        SELECT
            m.*,
            substr(m.mb_artist_key, 1, 1) AS a1,
            substr(m.mb_album_key,  1, 1) AS t1,
            length(m.mb_artist_key)       AS alen,
            length(m.mb_album_key)        AS tlen
        FROM mb_release_discovery_keys m
        WHERE length(m.mb_artist_key) >= 2
          AND length(m.mb_album_key)  >= 2
    ),

    -- Blocked candidate pairs with similarity pre-filter
    -- JW >= 0.80 on BOTH sides before any expensive computation
    candidates AS (
        SELECT
            w.source_url, w.page_title, w.domain,
            w.wt_artist,  w.wt_album,  w.wt_track,
            w.wt_genre,   w.wt_year,   w.wt_label,
            w.wt_artist_key, w.wt_album_key,
            m.base_record_id, m.release_id,
            m.base_record_status, m.is_linked_to_musicbrainz,
            m.artist_name  AS mb_artist,
            m.album_name   AS mb_album,
            m.mb_artist_key, m.mb_album_key,
            m.cds_category, m.cds_genre,
            m.total_tracks AS mb_total_tracks,
            m.barcode      AS mb_barcode,
            m.mb_year
        FROM wdc_unmatched w
        JOIN mb_keys m
          ON  w.a1   = m.a1
          AND w.t1   = m.t1
          AND m.alen BETWEEN w.alen - 4 AND w.alen + 8
          AND m.tlen BETWEEN w.tlen - 5 AND w.tlen + 5
        -- Pre-filter: only keep pairs with JW >= 0.80 on both sides
        -- This drops ~97% of pairs before scoring
        WHERE jaro_winkler_similarity(w.wt_artist_key, m.mb_artist_key) >= 0.80
          AND jaro_winkler_similarity(w.wt_album_key,  m.mb_album_key)  >= 0.80
    ),

    -- Score all surviving candidates
    scored AS (
        SELECT *,
            ROUND(jaro_winkler_similarity(wt_artist_key, mb_artist_key)*100, 3)
                AS artist_sim,
            ROUND(jaro_winkler_similarity(wt_album_key,  mb_album_key )*100, 3)
                AS album_sim,
            levenshtein(wt_artist_key, mb_artist_key) AS artist_lev,
            levenshtein(wt_album_key,  mb_album_key)  AS album_lev,
            ROUND(
                0.45 * jaro_winkler_similarity(wt_artist_key, mb_artist_key)*100
              + 0.55 * jaro_winkler_similarity(wt_album_key,  mb_album_key )*100
              + CASE WHEN lower(trim(coalesce(wt_artist,'')))
                        = lower(trim(coalesce(mb_artist,'')))
                      AND lower(trim(coalesce(wt_album, '')))
                        = lower(trim(coalesce(mb_album, '')))
                     THEN 5 ELSE 0 END
              + CASE WHEN is_linked_to_musicbrainz = TRUE THEN 2 ELSE 0 END
              - CASE WHEN levenshtein(wt_artist_key, mb_artist_key) > 6
                     THEN 10 ELSE 0 END
              - CASE WHEN levenshtein(wt_album_key,  mb_album_key)  > 8
                     THEN  8 ELSE 0 END
            , 3) AS final_score
        FROM candidates
    ),

    -- Rank: one best MB match per WDC row
    ranked AS (
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY source_url, wt_artist, wt_album
                ORDER BY final_score DESC, album_sim DESC,
                         artist_sim DESC, release_id ASC NULLS LAST
            ) AS rn,
            LEAD(final_score) OVER (
                PARTITION BY source_url, wt_artist, wt_album
                ORDER BY final_score DESC
            ) AS second_best_score
        FROM scored
    )

    SELECT
        * EXCLUDE (rn),
        ROUND(final_score - second_best_score, 3) AS score_gap,
        CASE
            WHEN final_score >= 90 AND album_sim >= 88 THEN 'auto_accept'
            WHEN final_score >= 80 AND album_sim >= 78 THEN 'review'
            ELSE 'reject'
        END AS confidence_class
    FROM ranked
    WHERE rn = 1
""")

print("✅ wdc_mb_fuzzy_best created")

# ════════════════════════════════════════════════════════════════
# STEP 4 — Results
# ════════════════════════════════════════════════════════════════
print("\n── Fuzzy confidence breakdown ──")
print(con.execute("""
    SELECT
        confidence_class,
        COUNT(*)                    AS n,
        ROUND(AVG(final_score), 2)  AS avg_score,
        ROUND(AVG(artist_sim),  2)  AS avg_artist_sim,
        ROUND(AVG(album_sim),   2)  AS avg_album_sim
    FROM wdc_mb_fuzzy_best
    GROUP BY confidence_class
    ORDER BY CASE confidence_class
        WHEN 'auto_accept' THEN 1
        WHEN 'review'      THEN 2
        ELSE 3 END
""").fetchdf().to_string(index=False))

print("\n── Auto-accept fuzzy matches ──")
display(con.execute("""
    SELECT
        wt_artist, mb_artist,
        wt_album,  mb_album,
        wt_label,  cds_genre,
        ROUND(artist_sim,  1) AS artist_sim,
        ROUND(album_sim,   1) AS album_sim,
        artist_lev, album_lev,
        ROUND(final_score, 1) AS score,
        release_id
    FROM wdc_mb_fuzzy_best
    WHERE confidence_class = 'auto_accept'
    ORDER BY final_score DESC
    LIMIT 20
""").fetchdf())

# ════════════════════════════════════════════════════════════════
# STEP 5 — Merge exact + fuzzy into final table
# ════════════════════════════════════════════════════════════════
con.execute("""
    CREATE OR REPLACE TABLE wdc_mb_final_matches AS

    SELECT
        source_url, domain, wt_artist, wt_album, wt_track,
        wt_genre, wt_year, wt_label,
        release_id, mb_artist, mb_album,
        cds_category, cds_genre,
        base_record_status, is_linked_to_musicbrainz,
        100.0 AS final_score, 100.0 AS artist_sim, 100.0 AS album_sim,
        NULL::INTEGER AS artist_lev, NULL::INTEGER AS album_lev,
        'exact' AS match_type
    FROM wdc_mb_exact_matches

    UNION ALL

    SELECT
        source_url, domain, wt_artist, wt_album, wt_track,
        wt_genre, wt_year, wt_label,
        release_id, mb_artist, mb_album,
        cds_category, cds_genre,
        base_record_status, is_linked_to_musicbrainz,
        final_score, artist_sim, album_sim,
        artist_lev, album_lev,
        'fuzzy' AS match_type
    FROM wdc_mb_fuzzy_best
    WHERE confidence_class = 'auto_accept'
""")

n_exact = con.execute("SELECT COUNT(*) FROM wdc_mb_exact_matches").fetchone()[0]
n_fuzzy = con.execute("""
    SELECT COUNT(*) FROM wdc_mb_fuzzy_best
    WHERE confidence_class = 'auto_accept'
""").fetchone()[0]
n_final = con.execute("SELECT COUNT(*) FROM wdc_mb_final_matches").fetchone()[0]

print("\n" + "═"*60)
print("  THESIS REPORT: WEBTABLES ENRICHMENT SUMMARY")
print("═"*60)
print(f"  WDC rows scanned (00.tar)     : 161,458")
print(f"  Exact matches → MB            : {n_exact:,}")
print(f"  Fuzzy auto-accept added       : {n_fuzzy:,}")
print(f"  Total WDC→MB matches          : {n_final:,}")
print("═"*60)

print("\n── Cross-source corroboration (CDS + WebTables) ──")
display(con.execute("""
    SELECT
        mb_artist, mb_album,
        cds_genre   AS genre_from_cds,
        wt_genre    AS genre_from_wt,
        wt_label    AS label_from_wt,
        CASE
            WHEN cds_genre IS NULL OR wt_genre IS NULL
              OR cds_genre IN ('None','') OR wt_genre IN ('None','')
                THEN 'one source only'
            WHEN lower(cds_genre) = lower(wt_genre) THEN 'agree'
            ELSE 'differ'
        END         AS genre_agreement,
        match_type,
        ROUND(final_score, 1) AS score,
        release_id
    FROM wdc_mb_final_matches
    WHERE (cds_genre IS NOT NULL AND cds_genre NOT IN ('None',''))
       OR (wt_genre  IS NOT NULL AND wt_genre  NOT IN ('None',''))
    ORDER BY genre_agreement, final_score DESC
    LIMIT 25
""").fetchdf())

print("\n── Score distribution ──")
print(con.execute("""
    SELECT
        CASE
            WHEN final_score = 100  THEN '100 (exact)'
            WHEN final_score >= 90  THEN '90-99'
            WHEN final_score >= 80  THEN '80-89'
            ELSE 'below 80'
        END AS score_bucket,
        COUNT(*) AS n
    FROM wdc_mb_final_matches
    GROUP BY score_bucket
    ORDER BY score_bucket DESC
""").fetchdf().to_string(index=False))

In [ ]:
import psutil
import os

DB_PATH = r"C:\Users\User\OneDrive\Desktop\Eman_Thesis\data_processed\thesis.duckdb"

print("Searching for processes locking the database...")
killed = []

for proc in psutil.process_iter(['pid', 'name', 'open_files']):
    try:
        for f in proc.open_files():
            if "thesis.duckdb" in f.path:
                print(f"  Found: PID {proc.pid} ({proc.name()}) → {f.path}")
                proc.kill()
                killed.append(proc.pid)
                print(f"  ✅ Killed PID {proc.pid}")
    except (psutil.NoSuchProcess, psutil.AccessDenied):
        pass

if not killed:
    print("No locking processes found.")
    print("Try manually: open Task Manager → find python.exe PID 4408 → End Task")
else:
    print(f"\n✅ Done. Killed {len(killed)} process(es).")

In [1]:
import duckdb
import pandas as pd
import time

pd.set_option('display.max_colwidth', None)

DB_PATH = "../data_processed/thesis.duckdb"

time.sleep(1)  # give OS 1 second to fully release file handles

con = duckdb.connect(DB_PATH)
print("Connected ✅")

# ── Verify all your data survived ───────────────────────────────
checks = [
    ("wdc_cleaned_for_matching",            161_458),
    ("wdc_mb_exact_matches",                 18_709),
    ("wdc_flattened_music_metadata",        164_923),
    ("music_enrichment_base_for_webtables_v1", 5_314_525),
]

print("\n── Data integrity check ──")
all_ok = True
for table, expected in checks:
    try:
        actual = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
        status = "✅" if actual == expected else "⚠️ "
        if actual != expected:
            all_ok = False
        print(f"  {status}  {table:<45} : {actual:>10,}  (expected {expected:,})")
    except Exception as ex:
        print(f"  ❌  {table:<45} : ERROR — {ex}")
        all_ok = False

print()
if all_ok:
    print("✅ All tables intact — safe to run fuzzy pipeline")
else:
    print("⚠️  Some tables differ from expected — check above")

# ── Drop stale locked tables from interrupted run ───────────────
stale = [
    "wdc_mb_fuzzy_candidates",
    "wdc_mb_fuzzy_best",
    "wdc_mb_final_matches",
    "exact_matched_keys",
]
for t in stale:
    con.execute(f"DROP TABLE IF EXISTS {t}")

print("\n── Stale tables dropped ──")
for t in stale:
    print(f"  🗑️  {t}")

print("\n✅ Ready — run the fuzzy pipeline cell next")

Connected ✅

── Data integrity check ──
  ✅  wdc_cleaned_for_matching                      :    161,458  (expected 161,458)
  ✅  wdc_mb_exact_matches                          :     18,709  (expected 18,709)
  ✅  wdc_flattened_music_metadata                  :    164,923  (expected 164,923)
  ✅  music_enrichment_base_for_webtables_v1        :  5,314,525  (expected 5,314,525)

✅ All tables intact — safe to run fuzzy pipeline

── Stale tables dropped ──
  🗑️  wdc_mb_fuzzy_candidates
  🗑️  wdc_mb_fuzzy_best
  🗑️  wdc_mb_final_matches
  🗑️  exact_matched_keys

✅ Ready — run the fuzzy pipeline cell next


In [2]:
import duckdb
import pandas as pd

pd.set_option('display.max_colwidth', None)
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ════════════════════════════════════════════════════════════════
# STEP 1 — Rebuild MB discovery keys view
#
# METHOD: We build normalized join keys from the enriched base
# table (not raw MB) so WebTables can match BOTH MB releases
# AND CDS-residual records. COALESCE picks MB name first,
# falls back to CDS external name for residual records.
# ════════════════════════════════════════════════════════════════
con.execute("""
    CREATE OR REPLACE VIEW mb_release_discovery_keys AS
    SELECT
        base_record_id, release_id,
        base_record_status, is_linked_to_musicbrainz,
        COALESCE(main_artist_name, external_artist) AS artist_name,
        COALESCE(release_title,    external_title)  AS album_name,
        CAST(external_category AS VARCHAR)          AS cds_category,
        CAST(external_genre    AS VARCHAR)          AS cds_genre,
        CAST(enrichment_source AS VARCHAR)          AS cds_enrichment_source,
        barcode, total_tracks, mb_year,
        trim(regexp_replace(
            lower(COALESCE(main_artist_name, external_artist, '')),
            '[^a-z0-9]+', ' ', 'g')) AS mb_artist_key,
        trim(regexp_replace(
            lower(COALESCE(release_title, external_title, '')),
            '[^a-z0-9]+', ' ', 'g')) AS mb_album_key
    FROM music_enrichment_base_for_webtables_v1
    WHERE COALESCE(main_artist_name, external_artist) IS NOT NULL
      AND COALESCE(release_title,    external_title)  IS NOT NULL
""")
print("✅ mb_release_discovery_keys view ready")

# ════════════════════════════════════════════════════════════════
# STEP 2 — Build exact match lookup table
#
# METHOD: We materialise the already-matched (source_url,
# artist, album) triples into a small hash table.
# This allows the anti-join in the next step to run as a
# single hash lookup instead of a correlated subquery
# (which caused the 134M × 18K = 2.5 billion comparisons
# that froze the previous attempt).
# ════════════════════════════════════════════════════════════════
con.execute("""
    CREATE OR REPLACE TABLE exact_matched_keys AS
    SELECT DISTINCT
        source_url,
        coalesce(wt_artist, '') AS wt_artist,
        coalesce(wt_album,  '') AS wt_album
    FROM wdc_mb_exact_matches
""")
n_keys = con.execute("SELECT COUNT(*) FROM exact_matched_keys").fetchone()[0]
print(f"✅ Exact match lookup: {n_keys:,} unique keys")

# ════════════════════════════════════════════════════════════════
# STEP 3 — Full fuzzy pipeline in one query
#
# ALGORITHMS USED:
#
# A) BLOCKING (performance technique from entity resolution)
#    Rather than comparing every WDC row against every MB row
#    (66,903 × 5,314,525 = 355 billion pairs), we only compare
#    pairs that share:
#      - same first letter of artist key
#      - same first letter of album key
#      - artist key length within ±4–8 chars
#      - album key length within ±5 chars
#    This reduces pairs to ~134M, then the JW pre-filter
#    further reduces to ~3–5M before scoring.
#
# B) JARO-WINKLER SIMILARITY (primary similarity metric)
#    Measures string similarity on a 0–1 scale.
#    Gives extra credit to strings matching at the start
#    (prefix bonus), which is important for music names
#    because the beginning of artist/album names is the
#    most discriminative part.
#    Examples:
#      "the beatles" vs "the beatles"    → 1.00 (perfect)
#      "sun ra arkestra" vs "sun ra"     → ~0.88
#      "nirvana" vs "nirvano"            → ~0.97
#    Weights: artist 45%, album 55%
#    (album titles are more unique than artist names)
#
# C) LEVENSHTEIN DISTANCE (used as a penalty guard)
#    Counts minimum character edits (insert/delete/replace)
#    to transform one string into another.
#    NOT used as the main score — used to penalise pairs
#    where Jaro-Winkler is deceptively high but strings
#    are actually very different in length/structure.
#    Penalty: -10 if artist edit distance > 6
#             -8  if album  edit distance > 8
#
# D) PRE-FILTER at JW >= 0.80
#    Applied BEFORE Levenshtein (which is more expensive).
#    Drops ~97% of blocked pairs before any scoring.
#    Only pairs where BOTH artist AND album have JW >= 0.80
#    proceed to full scoring.
#
# E) FINAL SCORE formula (0–107 range, practical range 80–107):
#    score = 0.45 × artist_JW × 100
#          + 0.55 × album_JW  × 100
#          + 5   if both raw strings match exactly
#          + 2   if MB release already enriched by CDS
#          - 10  if artist Levenshtein > 6
#          - 8   if album  Levenshtein > 8
#
# F) CONFIDENCE THRESHOLDS (same philosophy as CDS pipeline):
#    auto_accept : score >= 90 AND album_sim >= 88
#    review      : score >= 80 AND album_sim >= 78
#    reject      : everything else
#
# G) ROW_NUMBER() WINDOW FUNCTION (tie-breaking)
#    One WDC row can match multiple MB editions.
#    We keep only the single best match per (url, artist, album)
#    ranked by: score DESC → album_sim DESC → release_id ASC
#
# H) LEAD() WINDOW FUNCTION (score gap)
#    Computes difference between best and second-best score.
#    Large gap = one clear winner = higher confidence.
# ════════════════════════════════════════════════════════════════
print("\nRunning fuzzy pipeline (10–20 min) …")
print("Methods: Blocking → JW pre-filter(≥0.80) → Jaro-Winkler scoring")
print("         → Levenshtein guard → ROW_NUMBER tie-break → confidence class")

con.execute("""
    CREATE OR REPLACE TABLE wdc_mb_fuzzy_best AS

    WITH

    -- Anti-join: remove already exact-matched rows
    -- Uses hash join against the small lookup table (fast)
    wdc_unmatched AS (
        SELECT
            w.source_url, w.page_title, w.domain,
            w.wt_artist,  w.wt_album,  w.wt_track,
            CAST(w.wt_genre AS VARCHAR) AS wt_genre,
            CAST(w.wt_year  AS VARCHAR) AS wt_year,
            CAST(w.wt_label AS VARCHAR) AS wt_label,
            w.wt_artist_key, w.wt_album_key,
            substr(w.wt_artist_key, 1, 1) AS a1,
            substr(w.wt_album_key,  1, 1) AS t1,
            length(w.wt_artist_key)       AS alen,
            length(w.wt_album_key)        AS tlen
        FROM wdc_cleaned_for_matching w
        LEFT JOIN exact_matched_keys e
          ON  w.source_url             = e.source_url
          AND coalesce(w.wt_artist,'') = e.wt_artist
          AND coalesce(w.wt_album, '') = e.wt_album
        WHERE e.source_url IS NULL          -- keep only unmatched rows
          AND length(w.wt_artist_key) >= 2
          AND length(w.wt_album_key)  >= 2
    ),

    -- Pre-compute MB blocking columns once
    mb_keys AS (
        SELECT
            m.*,
            substr(m.mb_artist_key, 1, 1) AS a1,
            substr(m.mb_album_key,  1, 1) AS t1,
            length(m.mb_artist_key)       AS alen,
            length(m.mb_album_key)        AS tlen
        FROM mb_release_discovery_keys m
        WHERE length(m.mb_artist_key) >= 2
          AND length(m.mb_album_key)  >= 2
    ),

    -- BLOCKING + JW pre-filter
    -- Only pairs passing ALL blocking conditions AND JW >= 0.80
    -- proceed to full scoring (drops ~97% of pairs here)
    candidates AS (
        SELECT
            w.source_url, w.page_title, w.domain,
            w.wt_artist,  w.wt_album,  w.wt_track,
            w.wt_genre,   w.wt_year,   w.wt_label,
            w.wt_artist_key, w.wt_album_key,
            m.base_record_id, m.release_id,
            m.base_record_status, m.is_linked_to_musicbrainz,
            m.artist_name  AS mb_artist,
            m.album_name   AS mb_album,
            m.mb_artist_key, m.mb_album_key,
            m.cds_category, m.cds_genre,
            m.total_tracks AS mb_total_tracks,
            m.barcode      AS mb_barcode,
            m.mb_year
        FROM wdc_unmatched w
        JOIN mb_keys m
          ON  w.a1   = m.a1
          AND w.t1   = m.t1
          AND m.alen BETWEEN w.alen - 4 AND w.alen + 8
          AND m.tlen BETWEEN w.tlen - 5 AND w.tlen + 5
        -- JW pre-filter — applied BEFORE Levenshtein
        WHERE jaro_winkler_similarity(w.wt_artist_key, m.mb_artist_key) >= 0.80
          AND jaro_winkler_similarity(w.wt_album_key,  m.mb_album_key)  >= 0.80
    ),

    -- Full scoring with Jaro-Winkler + Levenshtein guard
    scored AS (
        SELECT *,
            ROUND(jaro_winkler_similarity(wt_artist_key, mb_artist_key)*100, 3)
                AS artist_sim,
            ROUND(jaro_winkler_similarity(wt_album_key,  mb_album_key )*100, 3)
                AS album_sim,
            levenshtein(wt_artist_key, mb_artist_key) AS artist_lev,
            levenshtein(wt_album_key,  mb_album_key)  AS album_lev,
            ROUND(
                -- Weighted Jaro-Winkler base score
                0.45 * jaro_winkler_similarity(wt_artist_key, mb_artist_key)*100
              + 0.55 * jaro_winkler_similarity(wt_album_key,  mb_album_key )*100
                -- Bonus: raw strings match exactly (case-insensitive)
              + CASE WHEN lower(trim(coalesce(wt_artist,'')))
                        = lower(trim(coalesce(mb_artist,'')))
                      AND lower(trim(coalesce(wt_album, '')))
                        = lower(trim(coalesce(mb_album, '')))
                     THEN 5 ELSE 0 END
                -- Bonus: CDS already confirmed this release
              + CASE WHEN is_linked_to_musicbrainz = TRUE THEN 2 ELSE 0 END
                -- Levenshtein guard penalties
              - CASE WHEN levenshtein(wt_artist_key, mb_artist_key) > 6
                     THEN 10 ELSE 0 END
              - CASE WHEN levenshtein(wt_album_key,  mb_album_key)  > 8
                     THEN  8 ELSE 0 END
            , 3) AS final_score
        FROM candidates
    ),

    -- Pick ONE best MB match per WDC row using ROW_NUMBER()
    -- LEAD() computes the score gap for confidence assessment
    ranked AS (
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY source_url, wt_artist, wt_album
                ORDER BY final_score DESC, album_sim DESC,
                         artist_sim DESC, release_id ASC NULLS LAST
            ) AS rn,
            LEAD(final_score) OVER (
                PARTITION BY source_url, wt_artist, wt_album
                ORDER BY final_score DESC
            ) AS second_best_score
        FROM scored
    )

    SELECT
        * EXCLUDE (rn),
        ROUND(final_score - second_best_score, 3) AS score_gap,
        CASE
            WHEN final_score >= 90 AND album_sim >= 88 THEN 'auto_accept'
            WHEN final_score >= 80 AND album_sim >= 78 THEN 'review'
            ELSE 'reject'
        END AS confidence_class
    FROM ranked
    WHERE rn = 1
""")

print("✅ wdc_mb_fuzzy_best created")

# ════════════════════════════════════════════════════════════════
# STEP 4 — Results & diagnostics
# ════════════════════════════════════════════════════════════════

print("\n── Fuzzy confidence breakdown ──")
print("(auto_accept = safe to use, review = needs inspection, reject = discarded)")
print(con.execute("""
    SELECT
        confidence_class,
        COUNT(*)                    AS n,
        ROUND(AVG(final_score), 2)  AS avg_score,
        ROUND(AVG(artist_sim),  2)  AS avg_artist_sim,
        ROUND(AVG(album_sim),   2)  AS avg_album_sim,
        ROUND(AVG(artist_lev),  2)  AS avg_artist_lev,
        ROUND(AVG(album_lev),   2)  AS avg_album_lev
    FROM wdc_mb_fuzzy_best
    GROUP BY confidence_class
    ORDER BY CASE confidence_class
        WHEN 'auto_accept' THEN 1
        WHEN 'review'      THEN 2
        ELSE 3 END
""").fetchdf().to_string(index=False))

print("\n── Auto-accept fuzzy matches (new finds beyond exact matching) ──")
print("(These are releases WDC found that were NOT exact-matched)")
display(con.execute("""
    SELECT
        wt_artist,  mb_artist,
        wt_album,   mb_album,
        wt_label,   cds_genre,
        ROUND(artist_sim,  1) AS artist_sim,
        ROUND(album_sim,   1) AS album_sim,
        artist_lev, album_lev,
        ROUND(score_gap,   1) AS score_gap,
        ROUND(final_score, 1) AS score,
        release_id
    FROM wdc_mb_fuzzy_best
    WHERE confidence_class = 'auto_accept'
    ORDER BY final_score DESC
    LIMIT 20
""").fetchdf())

# ════════════════════════════════════════════════════════════════
# STEP 5 — Merge exact + fuzzy auto_accept into final table
#
# METHOD: UNION ALL combines two non-overlapping sets:
#   - Exact matches (score = 100, match_type = 'exact')
#   - Fuzzy auto_accept (score 90–107, match_type = 'fuzzy')
# The match_type column preserves full provenance so you
# can always trace HOW each match was made.
# ════════════════════════════════════════════════════════════════
con.execute("""
    CREATE OR REPLACE TABLE wdc_mb_final_matches AS

    SELECT
        source_url, domain, wt_artist, wt_album, wt_track,
        wt_genre, wt_year, wt_label,
        release_id, mb_artist, mb_album,
        cds_category, cds_genre,
        base_record_status, is_linked_to_musicbrainz,
        100.0 AS final_score,
        100.0 AS artist_sim,
        100.0 AS album_sim,
        NULL::INTEGER AS artist_lev,
        NULL::INTEGER AS album_lev,
        'exact' AS match_type
    FROM wdc_mb_exact_matches

    UNION ALL

    SELECT
        source_url, domain, wt_artist, wt_album, wt_track,
        wt_genre, wt_year, wt_label,
        release_id, mb_artist, mb_album,
        cds_category, cds_genre,
        base_record_status, is_linked_to_musicbrainz,
        final_score, artist_sim, album_sim,
        artist_lev, album_lev,
        'fuzzy' AS match_type
    FROM wdc_mb_fuzzy_best
    WHERE confidence_class = 'auto_accept'
""")

n_exact = con.execute(
    "SELECT COUNT(*) FROM wdc_mb_exact_matches").fetchone()[0]
n_fuzzy = con.execute("""
    SELECT COUNT(*) FROM wdc_mb_fuzzy_best
    WHERE confidence_class = 'auto_accept'
""").fetchone()[0]
n_review = con.execute("""
    SELECT COUNT(*) FROM wdc_mb_fuzzy_best
    WHERE confidence_class = 'review'
""").fetchone()[0]
n_final = con.execute(
    "SELECT COUNT(*) FROM wdc_mb_final_matches").fetchone()[0]
n_wdc   = con.execute(
    "SELECT COUNT(*) FROM wdc_cleaned_for_matching").fetchone()[0]

# ════════════════════════════════════════════════════════════════
# STEP 6 — Thesis report
# ════════════════════════════════════════════════════════════════
print("\n" + "═"*62)
print("  THESIS REPORT: WEBTABLES ENRICHMENT SUMMARY")
print("═"*62)
print(f"  WDC files scanned (00.tar)        :   999,999")
print(f"  Music tables identified           :     9,183  (0.92%)")
print(f"  Music rows extracted              :   164,923")
print(f"  Rows after cleaning               :   {n_wdc:,}")
print(f"  Rows with artist + album (matchable):  66,903")
print("  ──────────────────────────────────────────────")
print(f"  Exact matches → MusicBrainz       :  {n_exact:>7,}")
print(f"  Fuzzy auto-accept added           :  {n_fuzzy:>7,}")
print(f"  Fuzzy review (manual inspect)     :  {n_review:>7,}")
print(f"  Total confirmed WDC→MB matches    :  {n_final:>7,}")
print(f"  Match rate (of matchable rows)    :  "
      f"{100*n_final/66903:.1f}%")
print("═"*62)

# Cross-source corroboration — the thesis headline finding
print("\n── Cross-source corroboration: CDS ∩ WebTables ──")
print("(Releases confirmed by TWO independent sources = highest confidence)")
display(con.execute("""
    SELECT
        mb_artist,
        mb_album,
        cds_genre   AS genre_from_cds,
        wt_genre    AS genre_from_wt,
        wt_label    AS label_from_wt,
        CASE
            WHEN cds_genre IS NULL OR wt_genre IS NULL
              OR cds_genre IN ('None','') OR wt_genre IN ('None','')
                THEN 'one source only'
            WHEN lower(cds_genre) = lower(wt_genre)
                THEN 'agree ✅'
            ELSE 'differ ⚠️'
        END         AS genre_agreement,
        match_type,
        ROUND(final_score, 1) AS score,
        release_id
    FROM wdc_mb_final_matches
    WHERE (cds_genre IS NOT NULL AND cds_genre NOT IN ('None',''))
       OR (wt_genre  IS NOT NULL AND wt_genre  NOT IN ('None',''))
    ORDER BY
        CASE genre_agreement WHEN 'agree ✅' THEN 1
                             WHEN 'differ ⚠️' THEN 2
                             ELSE 3 END,
        final_score DESC
    LIMIT 30
""").fetchdf())

print("\n── Score distribution ──")
print("(Shows how match quality is distributed across exact + fuzzy)")
print(con.execute("""
    SELECT
        CASE
            WHEN final_score = 100  THEN '100     (exact match)'
            WHEN final_score >= 95  THEN '95–99   (very high confidence)'
            WHEN final_score >= 90  THEN '90–94   (high confidence)'
            WHEN final_score >= 80  THEN '80–89   (review candidates)'
            ELSE 'below 80 (rejected)'
        END AS score_bucket,
        COUNT(*) AS n,
        match_type
    FROM wdc_mb_final_matches
    GROUP BY score_bucket, match_type
    ORDER BY score_bucket DESC, match_type
""").fetchdf().to_string(index=False))

print("\n── Which domains contributed most matches? ──")
print(con.execute("""
    SELECT domain, COUNT(*) AS matches,
           COUNT(CASE WHEN wt_genre IS NOT NULL
                           AND wt_genre NOT IN ('None','')
                      THEN 1 END) AS has_genre,
           COUNT(CASE WHEN wt_label IS NOT NULL
                           AND wt_label NOT IN ('None','')
                      THEN 1 END) AS has_label
    FROM wdc_mb_final_matches
    GROUP BY domain
    ORDER BY matches DESC
    LIMIT 15
""").fetchdf().to_string(index=False))

Connected ✅
✅ mb_release_discovery_keys view ready
✅ Exact match lookup: 18,709 unique keys

Running fuzzy pipeline (10–20 min) …
Methods: Blocking → JW pre-filter(≥0.80) → Jaro-Winkler scoring
         → Levenshtein guard → ROW_NUMBER tie-break → confidence class


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ wdc_mb_fuzzy_best created

── Fuzzy confidence breakdown ──
(auto_accept = safe to use, review = needs inspection, reject = discarded)
confidence_class    n  avg_score  avg_artist_sim  avg_album_sim  avg_artist_lev  avg_album_lev
     auto_accept 5212      96.94           96.96          96.89            1.20           1.83
          review 3445      85.62           88.90          86.91            4.10           6.10
          reject 3125      73.18           84.53          84.73            8.02          10.47

── Auto-accept fuzzy matches (new finds beyond exact matching) ──
(These are releases WDC found that were NOT exact-matched)


,wt_artist,mb_artist,wt_album,mb_album,wt_label,cds_genre,artist_sim,album_sim,artist_lev,album_lev,score_gap,score,release_id
0,Sytem of a Down,System of a Down,steal this album,Steal This Album!,american,metal,98.3,100.0,1,0,2.0,101.3,3241
1,Deer Hoof,Deerhoof,Reveille,Reveille,Kill Rock Stars,indie,97.8,100.0,1,0,2.0,101.0,29542
2,Deer Hoof,Deerhoof,Reveille,Reveille,Kill Rock Stars,indie,97.8,100.0,1,0,2.0,101.0,29542
3,Mannheim Steamroller,Mannheim Steamroller,A Fresh Aire Christm...,A Fresh Aire Christmas,None,new age,100.0,98.2,0,2,2.0,101.0,11382
4,Deer Hoof,Deerhoof,Reveille,Reveille,Kill Rock Stars,indie,97.8,100.0,1,0,2.0,101.0,29542
5,PRINCE,Prince,Rave Un2 The JoyFantastic,Rave In2 the Joy Fantastic,Arista,None,100.0,97.7,0,2,0.0,100.7,286098
6,Orbital,Orbital,The All Together,The Altogether,FFRR,None,100.0,97.5,0,2,1.3,100.6,42582
7,Amy Grant,Amy Grant,A Christmas To Remem...,A Christmas to Remember,None,holiday,100.0,97.4,0,3,0.0,100.6,27037
8,Amy Grant,Amy Grant,A Christmas To Remem...,A Christmas to Remember,None,holiday,100.0,97.4,0,3,0.0,100.6,27037
9,Amy Grant,Amy Grant,A Christmas To Remem...,A Christmas to Remember,None,holiday,100.0,97.4,0,3,0.0,100.6,27037



══════════════════════════════════════════════════════════════
  THESIS REPORT: WEBTABLES ENRICHMENT SUMMARY
══════════════════════════════════════════════════════════════
  WDC files scanned (00.tar)        :   999,999
  Music tables identified           :     9,183  (0.92%)
  Music rows extracted              :   164,923
  Rows after cleaning               :   161,458
  Rows with artist + album (matchable):  66,903
  ──────────────────────────────────────────────
  Exact matches → MusicBrainz       :   18,709
  Fuzzy auto-accept added           :    5,212
  Fuzzy review (manual inspect)     :    3,445
  Total confirmed WDC→MB matches    :   23,921
  Match rate (of matchable rows)    :  35.8%
══════════════════════════════════════════════════════════════

── Cross-source corroboration: CDS ∩ WebTables ──
(Releases confirmed by TWO independent sources = highest confidence)


,mb_artist,mb_album,genre_from_cds,genre_from_wt,label_from_wt,genre_agreement,match_type,score,release_id
0,QNTAL,QNTAL II,electronic,Rock,Metropolis Records,differ ⚠️,fuzzy,98.3,50220
1,System of a Down,Steal This Album!,metal,None,american,one source only,fuzzy,101.3,3241
2,Deerhoof,Reveille,indie,None,Kill Rock Stars,one source only,fuzzy,101.0,29542
3,Mannheim Steamroller,A Fresh Aire Christmas,new age,None,None,one source only,fuzzy,101.0,11382
4,Deerhoof,Reveille,indie,None,Kill Rock Stars,one source only,fuzzy,101.0,29542
5,Deerhoof,Reveille,indie,None,Kill Rock Stars,one source only,fuzzy,101.0,29542
6,Amy Grant,A Christmas to Remember,holiday,None,None,one source only,fuzzy,100.6,27037
7,Amy Grant,A Christmas to Remember,holiday,None,None,one source only,fuzzy,100.6,27037
8,Amy Grant,A Christmas to Remember,holiday,None,None,one source only,fuzzy,100.6,27037
9,Amy Grant,A Christmas to Remember,holiday,None,None,one source only,fuzzy,100.6,27037



── Score distribution ──
(Shows how match quality is distributed across exact + fuzzy)
                  score_bucket     n match_type
95–99   (very high confidence)  4215      fuzzy
     90–94   (high confidence)   996      fuzzy
         100     (exact match) 18709      exact
         100     (exact match)     1      fuzzy

── Which domains contributed most matches? ──
              domain  matches  has_genre  has_label
            kboo.org     5859          0       3891
            wnur.org     5303          0       3036
            kfai.org     4314          0       4240
             fln.org     2930          0          0
            wkdu.org     1655          0         70
            kvrx.org      507          0          0
            wfmu.org      407          0        188
            kfjc.org      302          0        200
      dailyvault.com      290          0          0
            wunh.org      282          0        236
            kexp.org      226          0          5
 

In [3]:
import duckdb
import pandas as pd

pd.set_option('display.max_colwidth', None)
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ════════════════════════════════════════════════════════════════
# STEP 1 — Check current WDC schema
# Do we have ANY barcode-like column already?
# ════════════════════════════════════════════════════════════════
print("\n── Current columns in wdc_cleaned_for_matching ──")
print(con.execute("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_name = 'wdc_cleaned_for_matching'
    ORDER BY ordinal_position
""").fetchdf().to_string(index=False))

# ════════════════════════════════════════════════════════════════
# STEP 2 — Check MusicBrainz barcode coverage
# How many MB releases actually HAVE a barcode?
# Only worth extracting WDC barcodes if MB has them to match against
# ════════════════════════════════════════════════════════════════
print("\n── MusicBrainz barcode coverage ──")
print(con.execute("""
    SELECT
        COUNT(*)                                        AS total_mb_releases,
        COUNT(CASE WHEN barcode IS NOT NULL
                        AND trim(barcode) <> ''
                   THEN 1 END)                          AS has_barcode,
        ROUND(100.0 *
            COUNT(CASE WHEN barcode IS NOT NULL
                            AND trim(barcode) <> ''
                       THEN 1 END)
            / COUNT(*), 2)                              AS pct_with_barcode
    FROM music_enrichment_base_for_webtables_v1
    WHERE release_id IS NOT NULL
""").fetchdf().to_string(index=False))

print("\n── Sample MB barcodes (what format are they?) ──")
print(con.execute("""
    SELECT
        COALESCE(main_artist_name, external_artist) AS artist,
        COALESCE(release_title,    external_title)  AS album,
        barcode,
        mb_year
    FROM music_enrichment_base_for_webtables_v1
    WHERE barcode IS NOT NULL
      AND trim(barcode) <> ''
      AND COALESCE(main_artist_name, external_artist) IS NOT NULL
    LIMIT 15
""").fetchdf().to_string(index=False))

# ════════════════════════════════════════════════════════════════
# STEP 3 — Check which WDC domains are likely to have barcodes
#
# REASONING: Barcodes (UPC/EAN) appear on:
#   - Record store catalogues (amoeba.com, 78discography.com)
#   - iTunes/Apple Music listings
#   - Discography/database sites (allmusic, discogs)
#   - Wikipedia infoboxes sometimes
#
# Radio playlists (kboo, wnur, kfai) NEVER have barcodes
# because DJs don't note them on playlists
# ════════════════════════════════════════════════════════════════
print("\n── Which domains are likely to carry barcodes? ──")
print(con.execute("""
    SELECT
        domain,
        COUNT(*) AS rows,
        -- Domains that sell music are most likely to have barcodes
        CASE
            WHEN domain IN ('itunes.apple.com', 'applemusic.info',
                            'amoeba.com', '78discography.com',
                            'en.wikipedia.org', 'discogs.com',
                            'allmusic.com')
            THEN 'likely has barcodes'
            WHEN domain IN ('kboo.org', 'wnur.org', 'kfai.org',
                            'fln.org', 'kvrx.org', 'wfmu.org',
                            'kfjc.org', 'wkdu.org', 'wrek.org')
            THEN 'radio playlist — no barcodes'
            ELSE 'unknown'
        END AS barcode_likelihood
    FROM wdc_cleaned_for_matching
    GROUP BY domain
    ORDER BY rows DESC
    LIMIT 20
""").fetchdf().to_string(index=False))

# ════════════════════════════════════════════════════════════════
# STEP 4 — Check if any WDC data has UPC-like patterns
#          hidden inside existing text columns
#
# METHOD: UPC barcodes are 12 digits, EAN are 13 digits.
# We scan wt_label and wt_track for numeric-only strings
# that look like barcodes — in case they were misclassified
# into the wrong column during extraction.
# ════════════════════════════════════════════════════════════════
print("\n── Scan existing columns for barcode-like values ──")
print("(UPC = 12 digits, EAN = 13 digits)")
print(con.execute(r"""
    SELECT
        'wt_label' AS column_checked,
        COUNT(*) AS barcode_like_values,
        MIN(length(wt_label)) AS min_len,
        MAX(length(wt_label)) AS max_len
    FROM wdc_cleaned_for_matching
    WHERE wt_label IS NOT NULL
      AND regexp_matches(trim(wt_label), '^[0-9]{12,13}$')

    UNION ALL

    SELECT
        'wt_track' AS column_checked,
        COUNT(*) AS barcode_like_values,
        MIN(length(wt_track)) AS min_len,
        MAX(length(wt_track)) AS max_len
    FROM wdc_cleaned_for_matching
    WHERE wt_track IS NOT NULL
      AND regexp_matches(trim(wt_track), '^[0-9]{12,13}$')

    UNION ALL

    SELECT
        'wt_year' AS column_checked,
        COUNT(*) AS barcode_like_values,
        MIN(length(wt_year)) AS min_len,
        MAX(length(wt_year)) AS max_len
    FROM wdc_cleaned_for_matching
    WHERE wt_year IS NOT NULL
      AND regexp_matches(trim(wt_year), '^[0-9]{12,13}$')
""").fetchdf().to_string(index=False))

# ════════════════════════════════════════════════════════════════
# STEP 5 — If we re-streamed with barcode support,
#          how many MB releases could we match via barcode?
#
# Barcode matching is MUCH more precise than name matching:
#   - One barcode = one specific release edition
#   - No ambiguity, no fuzzy scoring needed
#   - Pure equality match
# This shows the POTENTIAL value of adding barcode extraction
# ════════════════════════════════════════════════════════════════
print("\n── Barcode matching potential in MB ──")
print(con.execute("""
    SELECT
        COUNT(DISTINCT barcode) AS unique_barcodes_in_mb,
        COUNT(*)                AS total_releases_with_barcode,
        MIN(length(barcode))    AS min_barcode_length,
        MAX(length(barcode))    AS max_barcode_length,
        -- Show barcode format distribution
        COUNT(CASE WHEN length(trim(barcode)) = 12 THEN 1 END) AS upc_12digit,
        COUNT(CASE WHEN length(trim(barcode)) = 13 THEN 1 END) AS ean_13digit,
        COUNT(CASE WHEN length(trim(barcode)) NOT IN (12,13)
                   THEN 1 END)                                 AS other_length
    FROM music_enrichment_base_for_webtables_v1
    WHERE barcode IS NOT NULL
      AND trim(barcode) <> ''
      AND regexp_matches(trim(barcode), '^[0-9]+$')
""").fetchdf().to_string(index=False))

print("\n── Sample barcodes from domains likely to have them ──")
print(con.execute("""
    SELECT
        COALESCE(main_artist_name, external_artist) AS mb_artist,
        COALESCE(release_title,    external_title)  AS mb_album,
        barcode,
        length(trim(barcode)) AS barcode_len,
        mb_year
    FROM music_enrichment_base_for_webtables_v1
    WHERE barcode IS NOT NULL
      AND trim(barcode) <> ''
      AND regexp_matches(trim(barcode), '^[0-9]{12,13}$')
    LIMIT 15
""").fetchdf().to_string(index=False))

Connected ✅

── Current columns in wdc_cleaned_for_matching ──
  column_name data_type
   source_url   VARCHAR
   page_title   VARCHAR
       domain   VARCHAR
    wt_artist   VARCHAR
     wt_album   VARCHAR
     wt_track   VARCHAR
     wt_genre   VARCHAR
      wt_year   VARCHAR
     wt_label   VARCHAR
wt_artist_key   VARCHAR
 wt_album_key   VARCHAR
 wt_track_key   VARCHAR

── MusicBrainz barcode coverage ──
 total_mb_releases  has_barcode  pct_with_barcode
           5309878      2429536             45.76

── Sample MB barcodes (what format are they?) ──
                artist                                                                                            album       barcode  mb_year
                 Shoop                                                                             Farawaybutstillclose  194694611045     2020
Colossal Trailer Music                                                                                  Tears of Winter  872133280303     2020
          

In [4]:
import os
import tarfile
import json
import duckdb
import pandas as pd

pd.set_option('display.max_colwidth', None)

TAR_PATH = "../data_raw/wdc/english_relational_2015/00.tar"
DB_PATH  = "../data_processed/thesis.duckdb"

# ── Music filter config (unchanged) ────────────────────────────
STRONG_MUSIC_HEADERS = {
    "artist", "album", "track", "song", "genre",
    "label", "band", "singer", "record label"
}
MUSIC_PAGE_WORDS = {
    "playlist", "discography", "album", "tracklist", "band",
    "artist", "lyrics", "setlist", "radio", "music", "release", "chart"
}
BLOCKED_DOMAINS = {
    "anime-planet.com", "myanimelist.net", "imdb.com",
    "goodreads.com", "letterboxd.com", "amazon.com",
    "ebay.com", "walmart.com"
}

ARTIST_ALIASES  = {"artist", "band", "singer", "performer", "artist name", "act"}
ALBUM_ALIASES   = {"album", "release", "album title", "record", "album name"}
TRACK_ALIASES   = {"track", "song", "song title", "track title", "track name"}
GENRE_ALIASES   = {"genre", "style", "music genre"}
YEAR_ALIASES    = {"year", "release year"}
LABEL_ALIASES   = {"label", "record label"}

# ── NEW: Barcode aliases ────────────────────────────────────────
# UPC  = Universal Product Code (12 digits, USA/Canada)
# EAN  = European Article Number (13 digits, international)
# These are the column header names that appear on web pages
# that list products (record stores, iTunes, Wikipedia infoboxes)
BARCODE_ALIASES = {
    "barcode", "upc", "ean", "upc code", "ean code",
    "upc/ean", "catalog number", "catalogue number",
    "catalog no", "catalogue no", "cat no", "cat#",
    "catalog#", "catalogue#", "product code"
}

def get_domain(url):
    try:
        return url.split("/")[2].replace("www.", "")
    except:
        return ""

def get_headers(t):
    if not t.get("hasHeader"):
        return []
    if t.get("tableOrientation") != "HORIZONTAL":
        return []
    relation = t.get("relation", [])
    hdr_idx  = t.get("headerRowIndex", 0)
    return [
        col[hdr_idx].strip().lower()
        for col in relation
        if hdr_idx < len(col) and col[hdr_idx].strip()
    ]

def is_music_table(t):
    domain = get_domain(t.get("url", ""))
    if any(b in domain for b in BLOCKED_DOMAINS):
        return False
    headers     = set(get_headers(t))
    strong_hits = headers & STRONG_MUSIC_HEADERS
    page_title  = t.get("pageTitle", "").lower()
    title_hit   = any(w in page_title for w in MUSIC_PAGE_WORDS)
    return len(strong_hits) >= 1 and (len(strong_hits) >= 2 or title_hit)

def map_col(headers, aliases):
    for i, h in enumerate(headers):
        if h in aliases:
            return i
    return None

def safe_cell(relation, col_idx, row_idx):
    if col_idx is None:
        return None
    col = relation[col_idx]
    if row_idx < len(col):
        v = col[row_idx].strip()
        return v if v else None
    return None

def is_valid_barcode(value):
    """
    Validates that a string looks like a real UPC/EAN barcode.
    METHOD: Barcodes are purely numeric, 12 or 13 digits long.
    We reject anything with letters, spaces or wrong length
    to avoid storing catalogue numbers like 'DGC-24425' as barcodes.
    """
    if value is None:
        return False
    v = value.strip().replace("-", "").replace(" ", "")
    return v.isdigit() and len(v) in (12, 13)

# ════════════════════════════════════════════════════════════════
# STEP 1 — Recreate table with barcode column added
#           DROP and recreate so the schema is correct
# ════════════════════════════════════════════════════════════════
con = duckdb.connect(DB_PATH)
print("Connected ✅")

con.execute("""
    CREATE OR REPLACE TABLE wdc_flattened_music_metadata (
        source_url  VARCHAR,
        page_title  VARCHAR,
        domain      VARCHAR,
        wt_artist   VARCHAR,
        wt_album    VARCHAR,
        wt_track    VARCHAR,
        wt_genre    VARCHAR,
        wt_year     VARCHAR,
        wt_label    VARCHAR,
        wt_barcode  VARCHAR   -- NEW column
    )
""")
print("✅ wdc_flattened_music_metadata recreated with wt_barcode column")

# ════════════════════════════════════════════════════════════════
# STEP 2 — Re-stream, this time extracting barcode column
# ════════════════════════════════════════════════════════════════
print(f"\nRe-streaming from {TAR_PATH} …")
print("This time extracting: artist, album, track, genre, year, label, barcode")

BATCH_SIZE   = 5_000
REPORT_EVERY = 50_000

batch       = []
total_files = music_found = barcode_found = 0

def flush_batch(con, batch):
    if not batch:
        return []
    df = pd.DataFrame(batch, columns=[
        "source_url", "page_title", "domain",
        "wt_artist", "wt_album", "wt_track",
        "wt_genre", "wt_year", "wt_label", "wt_barcode"
    ])
    con.register("_batch_tmp", df)
    con.execute("""
        INSERT INTO wdc_flattened_music_metadata
        SELECT * FROM _batch_tmp
    """)
    con.unregister("_batch_tmp")
    return []

with tarfile.open(TAR_PATH, "r") as tar:
    for member in tar:
        if not member.isfile() or not member.name.endswith(".json"):
            continue

        total_files += 1

        if total_files % REPORT_EVERY == 0:
            batch = flush_batch(con, batch)
            rows_in_db = con.execute(
                "SELECT COUNT(*) FROM wdc_flattened_music_metadata"
            ).fetchone()[0]
            print(f"  Scanned {total_files:>8,} files | "
                  f"music: {music_found:,} | "
                  f"rows: {rows_in_db:,} | "
                  f"with barcode: {barcode_found:,}")

        try:
            f   = tar.extractfile(member)
            if f is None:
                continue
            raw = f.read().decode("utf-8", errors="replace").strip()
            if not raw:
                continue
            t   = json.loads(raw)
        except Exception:
            continue

        if not is_music_table(t):
            continue

        music_found += 1
        headers  = get_headers(t)
        relation = t.get("relation", [])
        hdr_idx  = t.get("headerRowIndex", 0)
        url      = t.get("url", "")
        domain   = get_domain(url)

        if not headers or not relation:
            continue

        i_artist  = map_col(headers, ARTIST_ALIASES)
        i_album   = map_col(headers, ALBUM_ALIASES)
        i_track   = map_col(headers, TRACK_ALIASES)
        i_genre   = map_col(headers, GENRE_ALIASES)
        i_year    = map_col(headers, YEAR_ALIASES)
        i_label   = map_col(headers, LABEL_ALIASES)
        i_barcode = map_col(headers, BARCODE_ALIASES)  # NEW

        if i_artist is None and i_album is None and i_track is None:
            continue

        n_rows = max(len(c) for c in relation)

        for r in range(hdr_idx + 1, n_rows):
            artist  = safe_cell(relation, i_artist,  r)
            album   = safe_cell(relation, i_album,   r)
            track   = safe_cell(relation, i_track,   r)
            genre   = safe_cell(relation, i_genre,   r)
            year    = safe_cell(relation, i_year,    r)
            label   = safe_cell(relation, i_label,   r)
            barcode = safe_cell(relation, i_barcode, r)  # NEW

            if not any([artist, album, track]):
                continue

            # Validate barcode — only store real UPC/EAN values
            # METHOD: reject non-numeric or wrong-length strings
            # e.g. "DGC-24425" is a catalogue number, not a barcode
            if barcode and not is_valid_barcode(barcode):
                barcode = None

            if barcode:
                barcode_found += 1

            batch.append((
                url, t.get("pageTitle", ""), domain,
                artist, album, track,
                str(genre)   if genre   else None,
                str(year)    if year    else None,
                label,
                barcode                              # NEW
            ))

        if len(batch) >= BATCH_SIZE:
            batch = flush_batch(con, batch)

batch = flush_batch(con, batch)

n_flat = con.execute(
    "SELECT COUNT(*) FROM wdc_flattened_music_metadata"
).fetchone()[0]

print(f"\n── Re-streaming complete ──")
print(f"  Total files scanned  : {total_files:,}")
print(f"  Music tables found   : {music_found:,}")
print(f"  Rows saved           : {n_flat:,}")
print(f"  Rows WITH barcode    : {barcode_found:,}")

# ════════════════════════════════════════════════════════════════
# STEP 3 — Rebuild cleaned matching table with wt_barcode
# ════════════════════════════════════════════════════════════════
print("\nRebuilding wdc_cleaned_for_matching with barcode …")

con.execute(r"""
    CREATE OR REPLACE TABLE wdc_cleaned_for_matching AS
    SELECT
        source_url, page_title, domain,
        wt_artist, wt_album, wt_track,
        wt_genre, wt_year, wt_label,
        wt_barcode,           -- NEW

        trim(regexp_replace(lower(coalesce(wt_artist, '')),
            '[^a-z0-9]+', ' ', 'g')) AS wt_artist_key,

        trim(regexp_replace(lower(coalesce(wt_album, '')),
            '[^a-z0-9]+', ' ', 'g')) AS wt_album_key,

        trim(regexp_replace(
            regexp_replace(lower(coalesce(wt_track, '')),
                '\(feat[^)]*\)|feat\.?\s.*', '', 'g'),
            '[^a-z0-9]+', ' ', 'g'
        )) AS wt_track_key

    FROM wdc_flattened_music_metadata
    WHERE
        (wt_artist IS NULL OR wt_artist NOT LIKE '%Ã%')
        AND (wt_artist IS NULL OR wt_artist NOT LIKE '%â%')
        AND (
            length(trim(regexp_replace(lower(coalesce(wt_artist,'')),
                '[^a-z0-9]+', ' ', 'g'))) >= 2
            OR
            length(trim(regexp_replace(lower(coalesce(wt_album, '')),
                '[^a-z0-9]+', ' ', 'g'))) >= 2
        )
""")

n_clean = con.execute(
    "SELECT COUNT(*) FROM wdc_cleaned_for_matching"
).fetchone()[0]
print(f"✅ wdc_cleaned_for_matching: {n_clean:,} rows")

# ════════════════════════════════════════════════════════════════
# STEP 4 — Barcode matching against MusicBrainz
#
# METHOD: Pure equality join on barcode string.
# No fuzzy scoring needed — barcodes are deterministic identifiers.
# One barcode = one specific release edition = zero ambiguity.
# This is the MOST PRECISE matching method in the entire pipeline.
#
# We normalise barcodes by stripping spaces and leading zeros
# to handle formatting differences between sources.
# ════════════════════════════════════════════════════════════════
print("\nRunning barcode matching against MusicBrainz …")

con.execute("""
    CREATE OR REPLACE TABLE wdc_mb_barcode_matches AS
    WITH

    -- Normalise WDC barcodes: strip spaces, remove leading zeros
    wdc_bc AS (
        SELECT *,
            -- ltrim removes leading zeros for safe comparison
            ltrim(trim(wt_barcode), '0') AS wt_barcode_norm
        FROM wdc_cleaned_for_matching
        WHERE wt_barcode IS NOT NULL
          AND length(trim(wt_barcode)) IN (12, 13)
    ),

    -- Normalise MB barcodes the same way
    mb_bc AS (
        SELECT
            release_id,
            COALESCE(main_artist_name, external_artist) AS artist_name,
            COALESCE(release_title,    external_title)  AS album_name,
            CAST(external_category AS VARCHAR)          AS cds_category,
            CAST(external_genre    AS VARCHAR)          AS cds_genre,
            base_record_status,
            is_linked_to_musicbrainz,
            total_tracks,
            mb_year,
            barcode,
            ltrim(trim(barcode), '0') AS barcode_norm
        FROM music_enrichment_base_for_webtables_v1
        WHERE barcode IS NOT NULL
          AND trim(barcode) <> ''
          AND regexp_matches(trim(barcode), '^[0-9]{12,13}$')
    )

    SELECT
        w.source_url,
        w.domain,
        w.wt_artist,
        w.wt_album,
        w.wt_track,
        w.wt_genre,
        w.wt_year,
        w.wt_label,
        w.wt_barcode,           -- original WDC barcode
        w.wt_barcode_norm,      -- normalised for matching
        m.release_id,
        m.artist_name           AS mb_artist,
        m.album_name            AS mb_album,
        m.barcode               AS mb_barcode,
        m.cds_category,
        m.cds_genre,
        m.base_record_status,
        m.is_linked_to_musicbrainz,
        m.total_tracks          AS mb_total_tracks,
        m.mb_year,
        'barcode'               AS match_type,
        100.0                   AS final_score   -- barcode = perfect match
    FROM wdc_bc w
    JOIN mb_bc m
      ON w.wt_barcode_norm = m.barcode_norm
""")

n_bc = con.execute(
    "SELECT COUNT(*) FROM wdc_mb_barcode_matches"
).fetchone()[0]
print(f"✅ wdc_mb_barcode_matches: {n_bc:,} rows")

# ════════════════════════════════════════════════════════════════
# STEP 5 — Barcode match quality report
# ════════════════════════════════════════════════════════════════
print("\n── Barcode match breakdown ──")
print(con.execute("""
    SELECT
        domain,
        COUNT(*)                                        AS matches,
        COUNT(CASE WHEN cds_genre IS NOT NULL
                        AND cds_genre NOT IN ('None','')
                   THEN 1 END)                          AS has_cds_genre,
        COUNT(CASE WHEN wt_genre IS NOT NULL
                   THEN 1 END)                          AS has_wt_genre,
        COUNT(CASE WHEN wt_label IS NOT NULL
                   THEN 1 END)                          AS has_wt_label
    FROM wdc_mb_barcode_matches
    GROUP BY domain
    ORDER BY matches DESC
    LIMIT 15
""").fetchdf().to_string(index=False))

print("\n── Sample barcode matches ──")
print("(These are the most precise matches in the entire pipeline)")
display(con.execute("""
    SELECT
        wt_artist,   mb_artist,
        wt_album,    mb_album,
        wt_barcode,  mb_barcode,
        wt_genre,    cds_genre,
        wt_label,
        domain,
        release_id
    FROM wdc_mb_barcode_matches
    WHERE wt_artist IS NOT NULL
      AND wt_album  IS NOT NULL
    ORDER BY domain, mb_artist
    LIMIT 20
""").fetchdf())

print("\n── Barcode vs name matching overlap ──")
print("(How many barcode matches were ALSO found by name matching?)")
print(con.execute("""
    SELECT
        COUNT(*)                                    AS total_barcode_matches,
        COUNT(CASE WHEN release_id IN
            (SELECT release_id FROM wdc_mb_exact_matches)
            THEN 1 END)                             AS also_in_exact_matches,
        COUNT(CASE WHEN release_id NOT IN
            (SELECT release_id FROM wdc_mb_exact_matches)
            AND release_id NOT IN
            (SELECT release_id FROM wdc_mb_fuzzy_best
             WHERE confidence_class = 'auto_accept')
            THEN 1 END)                             AS unique_to_barcode
    FROM wdc_mb_barcode_matches
""").fetchdf().to_string(index=False))

print("\n── Updated thesis numbers with barcode matching ──")
n_exact  = con.execute("SELECT COUNT(*) FROM wdc_mb_exact_matches").fetchone()[0]
n_fuzzy  = con.execute("""
    SELECT COUNT(*) FROM wdc_mb_fuzzy_best
    WHERE confidence_class = 'auto_accept'
""").fetchone()[0]
n_bc     = con.execute("SELECT COUNT(*) FROM wdc_mb_barcode_matches").fetchone()[0]

print(f"""
  Exact name matches        : {n_exact:>7,}
  Fuzzy name matches        : {n_fuzzy:>7,}
  Barcode matches (NEW)     : {n_bc:>7,}
  ─────────────────────────────────────
  Unique barcode-only finds : (see overlap table above)
""")

Connected ✅
✅ wdc_flattened_music_metadata recreated with wt_barcode column

Re-streaming from ../data_raw/wdc/english_relational_2015/00.tar …
This time extracting: artist, album, track, genre, year, label, barcode
  Scanned   50,000 files | music: 465 | rows: 9,798 | with barcode: 0
  Scanned  100,000 files | music: 911 | rows: 19,672 | with barcode: 0
  Scanned  150,000 files | music: 1,352 | rows: 29,225 | with barcode: 0
  Scanned  200,000 files | music: 1,804 | rows: 37,496 | with barcode: 0
  Scanned  250,000 files | music: 2,256 | rows: 45,723 | with barcode: 0
  Scanned  300,000 files | music: 2,717 | rows: 52,788 | with barcode: 0
  Scanned  350,000 files | music: 3,197 | rows: 61,977 | with barcode: 0
  Scanned  400,000 files | music: 3,657 | rows: 68,908 | with barcode: 0
  Scanned  450,000 files | music: 4,121 | rows: 76,943 | with barcode: 0
  Scanned  500,000 files | music: 4,569 | rows: 84,677 | with barcode: 0
  Scanned  550,000 files | music: 5,038 | rows: 92,568 | wi

,wt_artist,mb_artist,wt_album,mb_album,wt_barcode,mb_barcode,wt_genre,cds_genre,wt_label,domain,release_id



── Barcode vs name matching overlap ──
(How many barcode matches were ALSO found by name matching?)
 total_barcode_matches  also_in_exact_matches  unique_to_barcode
                     0                      0                  0

── Updated thesis numbers with barcode matching ──

  Exact name matches        :  18,709
  Fuzzy name matches        :   5,212
  Barcode matches (NEW)     :       0
  ─────────────────────────────────────
  Unique barcode-only finds : (see overlap table above)



In [5]:
import duckdb
con = duckdb.connect("../data_processed/thesis.duckdb")

print("── Current pipeline status ──")
tables = {
    "wdc_cleaned_for_matching"   : 161_458,
    "wdc_mb_exact_matches"       :  18_709,
    "wdc_mb_fuzzy_best"          :  None,
    "wdc_mb_final_matches"       :  23_921,
}
for t, expected in tables.items():
    n = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    flag = "✅" if (expected is None or n == expected) else "⚠️"
    print(f"  {flag}  {t:<35} : {n:>8,}")

── Current pipeline status ──
  ✅  wdc_cleaned_for_matching            :  161,458
  ✅  wdc_mb_exact_matches                :   18,709
  ✅  wdc_mb_fuzzy_best                   :   11,782
  ✅  wdc_mb_final_matches                :   23,921


In [3]:
import duckdb
import pandas as pd

pd.set_option('display.max_colwidth', None)
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ════════════════════════════════════════════════════════════════
# FIX: Instead of LEFT JOIN over 5.3M rows (2 hours),
# we build a small enrichment DELTA table — only the matched rows.
# This runs in seconds, not hours.
#
# The delta table contains ONLY the releases that got enriched.
# For thesis reporting we query counts from existing tables
# instead of materialising the full 5.3M row join.
# ════════════════════════════════════════════════════════════════

# STEP 1 — Build WDC enrichment delta (small table, fast)
# One best WDC match per MB release_id
print("Building WDC enrichment delta …")

con.execute("""
    CREATE OR REPLACE TABLE wdc_mb_enrichment_delta AS
    WITH ranked AS (
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY release_id
                ORDER BY
                    CASE match_type WHEN 'exact' THEN 0 ELSE 1 END ASC,
                    final_score DESC,
                    source_url ASC
            ) AS rn
        FROM wdc_mb_final_matches
        WHERE release_id IS NOT NULL
    )
    SELECT
        release_id,
        wt_artist,
        wt_album,
        wt_genre,
        wt_label,
        wt_year,
        source_url      AS wt_source_url,
        domain          AS wt_domain,
        match_type      AS wt_match_type,
        final_score     AS wt_match_score,
        artist_sim      AS wt_artist_sim,
        album_sim       AS wt_album_sim
    FROM ranked
    WHERE rn = 1
""")

n_delta = con.execute(
    "SELECT COUNT(*) FROM wdc_mb_enrichment_delta"
).fetchone()[0]
print(f"✅ wdc_mb_enrichment_delta: {n_delta:,} rows")

# STEP 2 — Corroboration table (small join — fast)
# Only the ~196 releases matched by BOTH CDS and WDC
print("\nBuilding corroboration table …")

con.execute("""
    CREATE OR REPLACE TABLE wdc_cds_corroboration AS
    SELECT
        d.release_id,
        b.main_artist_name,
        b.release_title,
        b.mb_year,
        -- CDS side
        CAST(b.external_genre    AS VARCHAR) AS cds_genre,
        CAST(b.external_category AS VARCHAR) AS cds_category,
        CAST(b.enrichment_source AS VARCHAR) AS cds_source,
        -- WDC side
        d.wt_genre,
        d.wt_label,
        d.wt_year,
        d.wt_domain,
        d.wt_match_type,
        ROUND(d.wt_match_score, 2) AS wt_score,
        -- Agreement
        CASE
            WHEN CAST(b.external_genre AS VARCHAR) IS NULL
              OR CAST(b.external_genre AS VARCHAR) IN ('None','')
              OR d.wt_genre IS NULL
              OR d.wt_genre IN ('None','')
                THEN 'one source only'
            WHEN lower(CAST(b.external_genre AS VARCHAR))
               = lower(d.wt_genre)
                THEN 'agree'
            ELSE 'differ'
        END AS genre_agreement
    FROM wdc_mb_enrichment_delta d
    JOIN music_enrichment_base_for_webtables_v1 b
      ON d.release_id = b.release_id
    WHERE b.is_linked_to_musicbrainz = TRUE  -- only CDS-enriched releases
""")

n_corr = con.execute(
    "SELECT COUNT(*) FROM wdc_cds_corroboration"
).fetchone()[0]
print(f"✅ wdc_cds_corroboration: {n_corr:,} rows")

# ════════════════════════════════════════════════════════════════
# STEP 3 — Thesis report (computed from existing small tables)
# NO full join needed — all counts come from small tables
# ════════════════════════════════════════════════════════════════
n_mb_total   = con.execute("""
    SELECT COUNT(*) FROM music_enrichment_base_for_webtables_v1
    WHERE release_id IS NOT NULL
""").fetchone()[0]

n_cds        = con.execute("""
    SELECT COUNT(*) FROM music_enrichment_base_for_webtables_v1
    WHERE is_linked_to_musicbrainz = TRUE
""").fetchone()[0]

n_wt         = con.execute(
    "SELECT COUNT(DISTINCT release_id) FROM wdc_mb_enrichment_delta"
).fetchone()[0]

n_corr       = con.execute(
    "SELECT COUNT(*) FROM wdc_cds_corroboration"
).fetchone()[0]

n_either     = con.execute("""
    SELECT COUNT(DISTINCT release_id)
    FROM (
        SELECT release_id FROM music_enrichment_base_for_webtables_v1
        WHERE is_linked_to_musicbrainz = TRUE
          AND release_id IS NOT NULL
        UNION
        SELECT release_id FROM wdc_mb_enrichment_delta
    )
""").fetchone()[0]

n_not        = n_mb_total - n_either

n_exact_wt   = con.execute("""
    SELECT COUNT(*) FROM wdc_mb_enrichment_delta
    WHERE wt_match_type = 'exact'
""").fetchone()[0]

n_fuzzy_wt   = con.execute("""
    SELECT COUNT(*) FROM wdc_mb_enrichment_delta
    WHERE wt_match_type = 'fuzzy'
""").fetchone()[0]

n_wt_label   = con.execute("""
    SELECT COUNT(*) FROM wdc_mb_enrichment_delta
    WHERE wt_label IS NOT NULL AND wt_label NOT IN ('None','')
""").fetchone()[0]

n_wt_genre   = con.execute("""
    SELECT COUNT(*) FROM wdc_mb_enrichment_delta
    WHERE wt_genre IS NOT NULL AND wt_genre NOT IN ('None','')
""").fetchone()[0]

n_wt_url     = con.execute("""
    SELECT COUNT(*) FROM wdc_mb_enrichment_delta
    WHERE wt_source_url IS NOT NULL
""").fetchone()[0]

print("\n" + "═"*65)
print("  FINAL THESIS REPORT: MUSICBRAINZ ENRICHMENT COMPLETE")
print("═"*65)
print(f"""
  MusicBrainz total releases          : {n_mb_total:>10,}
  ─────────────────────────────────────────────────────────
  Phase 1 — CDS Enrichment
    Enriched by CDS                   : {n_cds:>10,}  ({100*n_cds/n_mb_total:.4f}%)

  Phase 2 — WebTables Enrichment
    Files scanned (00.tar)            :    999,999
    Music tables found                :      9,183  (0.92%)
    Music rows extracted              :    164,923
    Matchable rows (artist+album)     :     66,903
    ── Exact name matches             : {n_exact_wt:>10,}
    ── Fuzzy name matches             : {n_fuzzy_wt:>10,}
    ── Total WDC→MB matches           : {n_wt:>10,}  ({100*n_wt/n_mb_total:.4f}%)

  ─────────────────────────────────────────────────────────
  Cross-Source Corroboration
    Releases confirmed by CDS + WDC   : {n_corr:>10,}  ({100*n_corr/n_mb_total:.4f}%)

  ─────────────────────────────────────────────────────────
  Combined Result
    Enriched by at least one source   : {n_either:>10,}  ({100*n_either/n_mb_total:.4f}%)
    Still not enriched                : {n_not:>10,}  ({100*n_not/n_mb_total:.4f}%)
  ─────────────────────────────────────────────────────────
  WebTables Fields Added to MB
    Releases with WDC label           : {n_wt_label:>10,}
    Releases with WDC genre           : {n_wt_genre:>10,}
    Releases with WDC provenance URL  : {n_wt_url:>10,}
""")
print("═"*65)

# Genre agreement
print("\n── Genre agreement (CDS vs WebTables) ──")
print(con.execute("""
    SELECT genre_agreement, COUNT(*) AS n
    FROM wdc_cds_corroboration
    GROUP BY genre_agreement
    ORDER BY n DESC
""").fetchdf().to_string(index=False))

# Corroboration sample
print("\n── Corroborated releases (both CDS + WebTables) ──")
display(con.execute("""
    SELECT
        main_artist_name,
        release_title,
        cds_genre,
        wt_genre,
        wt_label,
        genre_agreement,
        wt_match_type,
        wt_score,
        release_id
    FROM wdc_cds_corroboration
    ORDER BY genre_agreement, wt_score DESC
    LIMIT 20
""").fetchdf())

# WDC-only enrichment sample
print("\n── Releases enriched by WebTables only (not in CDS) ──")
display(con.execute("""
    SELECT
        b.main_artist_name,
        b.release_title,
        d.wt_genre,
        d.wt_label,
        d.wt_domain,
        d.wt_match_type,
        ROUND(d.wt_match_score, 1) AS wt_score,
        d.release_id
    FROM wdc_mb_enrichment_delta d
    JOIN music_enrichment_base_for_webtables_v1 b
      ON d.release_id = b.release_id
    WHERE b.is_linked_to_musicbrainz = FALSE
      AND (d.wt_genre IS NOT NULL OR d.wt_label IS NOT NULL)
    ORDER BY d.wt_match_score DESC
    LIMIT 20
""").fetchdf())

# Score distribution
print("\n── Match score distribution ──")
print(con.execute("""
    SELECT
        CASE
            WHEN wt_match_score = 100  THEN '100     (exact)'
            WHEN wt_match_score >= 95  THEN '95-99   (very high)'
            WHEN wt_match_score >= 90  THEN '90-94   (high)'
            WHEN wt_match_score >= 80  THEN '80-89   (review)'
            ELSE 'below 80'
        END AS score_bucket,
        wt_match_type,
        COUNT(*) AS n
    FROM wdc_mb_enrichment_delta
    GROUP BY score_bucket, wt_match_type
    ORDER BY score_bucket DESC
""").fetchdf().to_string(index=False))

# Domain contribution
print("\n── Top domains contributing WDC enrichment ──")
print(con.execute("""
    SELECT
        wt_domain,
        COUNT(*)                                            AS releases,
        COUNT(CASE WHEN wt_genre IS NOT NULL
                        AND wt_genre NOT IN ('None','')
                   THEN 1 END)                             AS with_genre,
        COUNT(CASE WHEN wt_label IS NOT NULL
                        AND wt_label NOT IN ('None','')
                   THEN 1 END)                             AS with_label
    FROM wdc_mb_enrichment_delta
    GROUP BY wt_domain
    ORDER BY releases DESC
    LIMIT 15
""").fetchdf().to_string(index=False))

Connected ✅
Building WDC enrichment delta …
✅ wdc_mb_enrichment_delta: 15,474 rows

Building corroboration table …
✅ wdc_cds_corroboration: 182 rows

═════════════════════════════════════════════════════════════════
  FINAL THESIS REPORT: MUSICBRAINZ ENRICHMENT COMPLETE
═════════════════════════════════════════════════════════════════

  MusicBrainz total releases          :  5,309,878
  ─────────────────────────────────────────────────────────
  Phase 1 — CDS Enrichment
    Enriched by CDS                   :      4,200  (0.0791%)

  Phase 2 — WebTables Enrichment
    Files scanned (00.tar)            :    999,999
    Music tables found                :      9,183  (0.92%)
    Music rows extracted              :    164,923
    Matchable rows (artist+album)     :     66,903
    ── Exact name matches             :     12,232
    ── Fuzzy name matches             :      3,242
    ── Total WDC→MB matches           :     15,474  (0.2914%)

  ────────────────────────────────────────────────

,main_artist_name,release_title,cds_genre,wt_genre,wt_label,genre_agreement,wt_match_type,wt_score,release_id
0,QNTAL,QNTAL II,electronic,Rock,Metropolis Records,differ,fuzzy,98.33,50220
1,System of a Down,Steal This Album!,metal,None,american,one source only,fuzzy,101.25,3241
2,Mannheim Steamroller,A Fresh Aire Christmas,new age,None,None,one source only,fuzzy,101.00,11382
3,Prince,Rave In2 the Joy Fantastic,None,None,Arista,one source only,fuzzy,100.71,286098
4,Orbital,The Altogether,None,None,FFRR,one source only,fuzzy,100.63,42582
5,Amy Grant,A Christmas to Remember,holiday,None,None,one source only,fuzzy,100.57,27037
6,Zimmers Hole,Bound by Fire,None,None,HevyDevy Records,one source only,fuzzy,100.56,12153
7,Talulah Gosh,Rock Legends: Volume 69,None,None,None,one source only,fuzzy,100.50,1572382
8,Guru,"Jazzmatazz, Volume 1",None,None,Blue Note,one source only,fuzzy,100.26,216021
9,J. P. Cormier,Another Morning,None,None,None,one source only,fuzzy,100.17,1946229



── Releases enriched by WebTables only (not in CDS) ──


,main_artist_name,release_title,wt_genre,wt_label,wt_domain,wt_match_type,wt_score,release_id
0,Adele,21,None,COLUMBIA,kfai.org,exact,100.0,1046996
1,Beyond,Fatal Power of Death,None,Iron Bonehead Productions,kfai.org,exact,100.0,1335444
2,Jedi Scum,Jedi Scum,None,Goat Power Recreation,kfjc.org,exact,100.0,1316404
3,Jimmy Johnson,I'm a Jockey,None,Verve,kfai.org,exact,100.0,1297535
4,Dub-I,Creation,None,Dubophonic,kfai.org,exact,100.0,1307322
5,Coroner,R.I.P.,None,Noise Records,kfai.org,exact,100.0,412711
6,Marcia Ball,Hot Tamale Baby,None,Rounder,kfai.org,exact,100.0,1023813
7,Millie Jackson,Rock 'N' Soul,None,iCHIBAN,wnur.org,exact,100.0,1329607
8,Juveniles,Juveniles,New Music Legacy,Paradis,wfdu.radioactivity.fm,exact,100.0,1301742
9,Eat Skull,III,None,Woodsist,wxdu.org,exact,100.0,1313738



── Match score distribution ──
       score_bucket wt_match_type     n
95-99   (very high)         fuzzy  2612
     90-94   (high)         fuzzy   629
    100     (exact)         exact 12232
    100     (exact)         fuzzy     1

── Top domains contributing WDC enrichment ──
           wt_domain  releases  with_genre  with_label
            kfai.org      3514           0        3445
            wnur.org      3348           0        1895
            kboo.org      3344           0        2194
            wkdu.org      1333           0          59
             fln.org       545           0           0
            kvrx.org       424           0           0
            wfmu.org       339           0         151
            kfjc.org       285           0         188
      dailyvault.com       280           0           0
            wunh.org       217           0         177
            kexp.org       207           0           5
            wxdu.org       154           0         126
cc-sea

In [4]:
import duckdb
import pandas as pd

pd.set_option('display.max_colwidth', None)
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ════════════════════════════════════════════════════════════════
# Build the final enriched MusicBrainz table
#
# METHOD: LEFT JOIN the WDC final matches onto the MB base table.
# LEFT JOIN preserves ALL 5.3M MB releases — rows with no WDC
# match simply get NULL in the WDC columns.
# We pick ONE best WDC match per release_id using ROW_NUMBER()
# ranked by: match_type (exact > fuzzy) → final_score → release_id
# ════════════════════════════════════════════════════════════════
con.execute("""
    CREATE OR REPLACE TABLE musicbrainz_release_enriched_v2 AS

    WITH

    -- Pick best WDC match per release_id (deduplicate)
    -- One MB release can appear in multiple WDC playlists
    wdc_best_per_release AS (
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY release_id
                ORDER BY
                    CASE match_type WHEN 'exact' THEN 0 ELSE 1 END ASC,
                    final_score DESC,
                    source_url ASC
            ) AS rn
        FROM wdc_mb_final_matches
        WHERE release_id IS NOT NULL
    )

    SELECT
        -- All original MB + CDS enrichment columns
        b.base_record_id,
        b.release_id,
        b.release_title,
        b.main_artist_name,
        b.artist_credit_full,
        b.barcode,
        b.mb_year,
        b.total_tracks,
        b.base_record_status,
        b.is_linked_to_musicbrainz,

        -- CDS enrichment (from Phase 1)
        b.external_category     AS cds_category,
        b.external_genre        AS cds_genre,
        b.enrichment_source     AS cds_enrichment_source,
        b.enrichment_status     AS cds_enrichment_status,

        -- WebTables enrichment (NEW — from Phase 2)
        w.wt_genre              AS wt_genre,
        w.wt_label              AS wt_label,
        w.wt_year               AS wt_year,
        w.source_url            AS wt_source_url,
        w.domain                AS wt_domain,
        w.match_type            AS wt_match_type,
        w.final_score           AS wt_match_score,

        -- Enrichment flags
        (b.is_linked_to_musicbrainz = TRUE)  AS is_enriched_from_cds,
        (w.release_id IS NOT NULL)           AS is_enriched_from_wt,

        -- Cross-source corroboration flag
        -- TRUE when BOTH CDS and WebTables matched this release
        (b.is_linked_to_musicbrainz = TRUE
         AND w.release_id IS NOT NULL)       AS is_corroborated,

        -- Genre agreement between CDS and WDC
        CASE
            WHEN b.external_genre IS NULL
              OR w.wt_genre IS NULL
              OR CAST(b.external_genre AS VARCHAR) IN ('None','')
              OR w.wt_genre IN ('None','')
                THEN 'one source only'
            WHEN lower(CAST(b.external_genre AS VARCHAR))
               = lower(w.wt_genre)
                THEN 'agree'
            ELSE 'differ'
        END                                  AS genre_agreement

    FROM music_enrichment_base_for_webtables_v1 b
    LEFT JOIN wdc_best_per_release w
      ON b.release_id = w.release_id
     AND w.rn = 1
""")

n_total = con.execute(
    "SELECT COUNT(*) FROM musicbrainz_release_enriched_v2"
).fetchone()[0]
print(f"✅ musicbrainz_release_enriched_v2: {n_total:,} rows")

# ════════════════════════════════════════════════════════════════
# Final thesis report
# ════════════════════════════════════════════════════════════════
print("\n" + "═"*65)
print("  FINAL THESIS REPORT: MUSICBRAINZ ENRICHMENT COMPLETE")
print("═"*65)

stats = con.execute("""
    SELECT
        COUNT(*)                                            AS total_releases,

        -- CDS enrichment
        COUNT(CASE WHEN is_enriched_from_cds = TRUE
                   THEN 1 END)                             AS enriched_by_cds,
        ROUND(100.0 *
            COUNT(CASE WHEN is_enriched_from_cds = TRUE
                       THEN 1 END) / COUNT(*), 4)          AS cds_pct,

        -- WebTables enrichment
        COUNT(CASE WHEN is_enriched_from_wt = TRUE
                   THEN 1 END)                             AS enriched_by_wt,
        ROUND(100.0 *
            COUNT(CASE WHEN is_enriched_from_wt = TRUE
                       THEN 1 END) / COUNT(*), 4)          AS wt_pct,

        -- Corroborated by BOTH
        COUNT(CASE WHEN is_corroborated = TRUE
                   THEN 1 END)                             AS corroborated_both,
        ROUND(100.0 *
            COUNT(CASE WHEN is_corroborated = TRUE
                       THEN 1 END) / COUNT(*), 4)          AS corroborated_pct,

        -- Enriched by at least ONE source
        COUNT(CASE WHEN is_enriched_from_cds = TRUE
                     OR is_enriched_from_wt  = TRUE
                   THEN 1 END)                             AS enriched_any,

        -- Not enriched at all
        COUNT(CASE WHEN is_enriched_from_cds = FALSE
                    AND is_enriched_from_wt  = FALSE
                   THEN 1 END)                             AS not_enriched

    FROM musicbrainz_release_enriched_v2
    WHERE release_id IS NOT NULL
""").fetchone()

print(f"""
  MusicBrainz total releases          : {stats[0]:>10,}
  ─────────────────────────────────────────────────────
  Enriched by CDS                     : {stats[1]:>10,}  ({stats[2]:.4f}%)
  Enriched by WebTables               : {stats[3]:>10,}  ({stats[4]:.4f}%)
  ─────────────────────────────────────────────────────
  Corroborated by BOTH sources        : {stats[5]:>10,}  ({stats[6]:.4f}%)
  ─────────────────────────────────────────────────────
  Enriched by at least one source     : {stats[7]:>10,}
  Still not enriched                  : {stats[8]:>10,}
""")
print("═"*65)

# Genre agreement between CDS and WebTables
print("\n── Genre agreement (CDS vs WebTables) ──")
print("(Only rows where BOTH sources provided a genre)")
print(con.execute("""
    SELECT
        genre_agreement,
        COUNT(*) AS n
    FROM musicbrainz_release_enriched_v2
    WHERE release_id IS NOT NULL
    GROUP BY genre_agreement
    ORDER BY n DESC
""").fetchdf().to_string(index=False))

# What WebTables adds that CDS does not have
print("\n── What WebTables adds beyond CDS ──")
print(con.execute("""
    SELECT
        COUNT(CASE WHEN is_enriched_from_wt = TRUE
                    AND wt_label IS NOT NULL
                   THEN 1 END)  AS wt_label_count,
        COUNT(CASE WHEN is_enriched_from_wt = TRUE
                    AND wt_genre IS NOT NULL
                   THEN 1 END)  AS wt_genre_count,
        COUNT(CASE WHEN is_enriched_from_wt = TRUE
                    AND wt_year IS NOT NULL
                   THEN 1 END)  AS wt_year_count,
        COUNT(CASE WHEN is_enriched_from_wt = TRUE
                    AND wt_source_url IS NOT NULL
                   THEN 1 END)  AS wt_provenance_url_count
    FROM musicbrainz_release_enriched_v2
    WHERE release_id IS NOT NULL
""").fetchdf().to_string(index=False))

# Sample — corroborated releases (both sources agree)
print("\n── Sample: releases corroborated by BOTH CDS + WebTables ──")
display(con.execute("""
    SELECT
        main_artist_name,
        release_title,
        cds_genre,
        wt_genre,
        wt_label,
        genre_agreement,
        wt_match_type,
        ROUND(wt_match_score, 1) AS wt_score,
        release_id
    FROM musicbrainz_release_enriched_v2
    WHERE is_corroborated = TRUE
    ORDER BY genre_agreement, wt_score DESC
    LIMIT 20
""").fetchdf())

# Sample — enriched by WDC only (not in CDS)
print("\n── Sample: releases enriched by WebTables only ──")
display(con.execute("""
    SELECT
        main_artist_name,
        release_title,
        wt_genre,
        wt_label,
        wt_domain,
        wt_match_type,
        ROUND(wt_match_score, 1) AS wt_score,
        release_id
    FROM musicbrainz_release_enriched_v2
    WHERE is_enriched_from_wt  = TRUE
      AND is_enriched_from_cds = FALSE
      AND (wt_genre IS NOT NULL OR wt_label IS NOT NULL)
    ORDER BY wt_score DESC
    LIMIT 20
""").fetchdf())

Connected ✅


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ musicbrainz_release_enriched_v2: 5,314,525 rows

═════════════════════════════════════════════════════════════════
  FINAL THESIS REPORT: MUSICBRAINZ ENRICHMENT COMPLETE
═════════════════════════════════════════════════════════════════

  MusicBrainz total releases          :  5,309,878
  ─────────────────────────────────────────────────────
  Enriched by CDS                     :      4,200  (0.0791%)
  Enriched by WebTables               :     15,474  (0.2914%)
  ─────────────────────────────────────────────────────
  Corroborated by BOTH sources        :        182  (0.0034%)
  ─────────────────────────────────────────────────────
  Enriched by at least one source     :     19,492
  Still not enriched                  :  5,290,386

═════════════════════════════════════════════════════════════════

── Genre agreement (CDS vs WebTables) ──
(Only rows where BOTH sources provided a genre)
genre_agreement       n
one source only 5309877
         differ       1

── What WebTables adds b

,main_artist_name,release_title,cds_genre,wt_genre,wt_label,genre_agreement,wt_match_type,wt_score,release_id
0,QNTAL,QNTAL II,electronic,Rock,Metropolis Records,differ,fuzzy,98.3,50220
1,System of a Down,Steal This Album!,metal,None,american,one source only,fuzzy,101.3,3241
2,Mannheim Steamroller,A Fresh Aire Christmas,new age,None,None,one source only,fuzzy,101.0,11382
3,Prince,Rave In2 the Joy Fantastic,None,None,Arista,one source only,fuzzy,100.7,286098
4,Orbital,The Altogether,None,None,FFRR,one source only,fuzzy,100.6,42582
5,Zimmers Hole,Bound by Fire,None,None,HevyDevy Records,one source only,fuzzy,100.6,12153
6,Amy Grant,A Christmas to Remember,holiday,None,None,one source only,fuzzy,100.6,27037
7,Talulah Gosh,Rock Legends: Volume 69,None,None,None,one source only,fuzzy,100.5,1572382
8,Guru,"Jazzmatazz, Volume 1",None,None,Blue Note,one source only,fuzzy,100.3,216021
9,J. P. Cormier,Another Morning,None,None,None,one source only,fuzzy,100.2,1946229



── Sample: releases enriched by WebTables only ──


,main_artist_name,release_title,wt_genre,wt_label,wt_domain,wt_match_type,wt_score,release_id
0,Grouper,A I A,None,Yellow Electric GR006,cc-seas.columbia.edu,exact,100.0,1306436
1,Donna the Buffalo,"Tonight, Tomorrow and Yesterday",None,Sugar Hill,kboo.org,exact,100.0,1301933
2,Kenny Garrett,Simply Said,None,warner brothers,kboo.org,exact,100.0,205500
3,Theatre of Tragedy,A Rose for the Dead,None,Swanlake,wfmu.org,exact,100.0,34799
4,0.720 Aleación,0.720 Aleacion,None,Aleación,kboo.org,exact,100.0,1336041
5,Fusioon,Absolute Fusioon,None,B-Music,wnur.org,exact,100.0,1234194
6,The Octopus Project,One Ten Hundred Thousand Million,None,Peek-a-boo,wnur.org,exact,100.0,1304875
7,Sonic Youth,The Eternal,None,Matador,kboo.org,exact,100.0,531959
8,Maria Kalaniemi,Maria Kalaniemi,None,Green Linnet,kfai.org,exact,100.0,1323963
9,Billy Martin,Shimmy,None,The Royal Potato Family,wunh.org,exact,100.0,1234475


In [5]:
import duckdb
import os

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

out_dir = "../data_processed/exports"
os.makedirs(out_dir, exist_ok=True)

# ── Table 1: Full enriched MusicBrainz (5.3M rows) ─────────────
# WARNING: this will be a large CSV (~3-4 GB)
# Only export if you have enough disk space
print("Exporting musicbrainz_release_enriched_v2 …")
con.execute(f"""
    COPY musicbrainz_release_enriched_v2
    TO '{out_dir}/musicbrainz_release_enriched_v2.csv'
    WITH (HEADER, DELIMITER ',')
""")
print("✅ Exported: musicbrainz_release_enriched_v2.csv")

# ── Table 2: WDC enrichment delta (small — 15,474 rows) ────────
print("Exporting wdc_mb_enrichment_delta …")
con.execute(f"""
    COPY wdc_mb_enrichment_delta
    TO '{out_dir}/wdc_mb_enrichment_delta.csv'
    WITH (HEADER, DELIMITER ',')
""")
print("✅ Exported: wdc_mb_enrichment_delta.csv")

# ── Table 3: Corroboration table (tiny — 182 rows) ─────────────
print("Exporting wdc_cds_corroboration …")
con.execute(f"""
    COPY wdc_cds_corroboration
    TO '{out_dir}/wdc_cds_corroboration.csv'
    WITH (HEADER, DELIMITER ',')
""")
print("✅ Exported: wdc_cds_corroboration.csv")

# ── Table 4: Enriched-only subset (19,492 rows) ─────────────────
# A lighter version — only the releases that actually got enriched
# Much more manageable than the full 5.3M row file
print("Exporting enriched-only subset …")
con.execute(f"""
    COPY (
        SELECT *
        FROM musicbrainz_release_enriched_v2
        WHERE is_enriched_from_cds = TRUE
           OR is_enriched_from_wt  = TRUE
    )
    TO '{out_dir}/musicbrainz_enriched_subset.csv'
    WITH (HEADER, DELIMITER ',')
""")
print("✅ Exported: musicbrainz_enriched_subset.csv")

# ── File size report ────────────────────────────────────────────
print("\n── Export file sizes ──")
files = [
    "musicbrainz_release_enriched_v2.csv",
    "wdc_mb_enrichment_delta.csv",
    "wdc_cds_corroboration.csv",
    "musicbrainz_enriched_subset.csv",
]
for f in files:
    path = os.path.join(out_dir, f)
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"  {f:<50} : {size_mb:>8.1f} MB")

print("\n✅ All exports complete")
print(f"📁 Location: {os.path.abspath(out_dir)}")

Connected ✅
Exporting musicbrainz_release_enriched_v2 …


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Exported: musicbrainz_release_enriched_v2.csv
Exporting wdc_mb_enrichment_delta …
✅ Exported: wdc_mb_enrichment_delta.csv
Exporting wdc_cds_corroboration …
✅ Exported: wdc_cds_corroboration.csv
Exporting enriched-only subset …
✅ Exported: musicbrainz_enriched_subset.csv

── Export file sizes ──
  musicbrainz_release_enriched_v2.csv                :    789.2 MB
  wdc_mb_enrichment_delta.csv                        :      2.0 MB
  wdc_cds_corroboration.csv                          :      0.0 MB
  musicbrainz_enriched_subset.csv                    :      4.1 MB

✅ All exports complete
📁 Location: C:\Users\User\OneDrive\Desktop\Eman_Thesis\data_processed\exports


In [6]:
import duckdb
import pandas as pd

pd.set_option('display.max_colwidth', None)
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ── Overall genre agreement distribution ───────────────────────
print("\n── Genre agreement: full breakdown ──")
print(con.execute("""
    SELECT
        genre_agreement,
        COUNT(*)                                    AS total_rows,
        COUNT(CASE WHEN release_id IS NOT NULL
                   THEN 1 END)                      AS mb_releases,
        COUNT(CASE WHEN release_id IS NULL
                   THEN 1 END)                      AS cds_residual_rows
    FROM musicbrainz_release_enriched_v2
    GROUP BY genre_agreement
    ORDER BY total_rows DESC
""").fetchdf().to_string(index=False))

# ── Break down 'one source only' by WHICH source has the genre ──
print("\n── For 'one source only': which source has the genre? ──")
print(con.execute("""
    SELECT
        CASE
            WHEN cds_genre IS NOT NULL
                 AND cds_genre NOT IN ('None','')
                 AND (wt_genre IS NULL OR wt_genre IN ('None',''))
                THEN 'CDS has genre, WDC does not'
            WHEN wt_genre IS NOT NULL
                 AND wt_genre NOT IN ('None','')
                 AND (cds_genre IS NULL OR cds_genre IN ('None',''))
                THEN 'WDC has genre, CDS does not'
            WHEN cds_genre IS NULL AND wt_genre IS NULL
                THEN 'neither has genre (not enriched)'
            ELSE 'other'
        END AS genre_source,
        COUNT(*) AS n
    FROM musicbrainz_release_enriched_v2
    WHERE genre_agreement = 'one source only'
      AND release_id IS NOT NULL
    GROUP BY genre_source
    ORDER BY n DESC
""").fetchdf().to_string(index=False))

# ── Is enriched but no genre from either source ─────────────────
print("\n── Enriched releases WITH and WITHOUT genre ──")
print(con.execute("""
    SELECT
        CASE
            WHEN is_enriched_from_cds = TRUE
                 AND (cds_genre IS NOT NULL
                      AND cds_genre NOT IN ('None',''))
                THEN 'enriched + has CDS genre'
            WHEN is_enriched_from_wt = TRUE
                 AND wt_genre IS NOT NULL
                 AND wt_genre NOT IN ('None','')
                THEN 'enriched + has WDC genre'
            WHEN is_enriched_from_cds = TRUE
                 OR is_enriched_from_wt = TRUE
                THEN 'enriched but NO genre from either source'
            ELSE 'not enriched at all'
        END AS enrichment_genre_status,
        COUNT(*) AS n
    FROM musicbrainz_release_enriched_v2
    WHERE release_id IS NOT NULL
    GROUP BY enrichment_genre_status
    ORDER BY n DESC
""").fetchdf().to_string(index=False))

# ── For 'one source only' corroborated rows specifically ─────────
print("\n── 'One source only' in the 182 corroborated releases ──")
print(con.execute("""
    SELECT
        CASE
            WHEN cds_genre IS NOT NULL
                 AND cds_genre NOT IN ('None','')
                 AND (wt_genre IS NULL OR wt_genre IN ('None',''))
                THEN 'CDS has genre, WDC does not'
            WHEN wt_genre IS NOT NULL
                 AND wt_genre NOT IN ('None','')
                 AND (cds_genre IS NULL OR cds_genre IN ('None',''))
                THEN 'WDC has genre, CDS does not'
            ELSE 'neither has genre'
        END AS genre_source,
        COUNT(*) AS n
    FROM musicbrainz_release_enriched_v2
    WHERE is_corroborated = TRUE
      AND genre_agreement = 'one source only'
    GROUP BY genre_source
    ORDER BY n DESC
""").fetchdf().to_string(index=False))

# ── Sample: one source only where CDS has genre ──────────────────
print("\n── Sample: one source only — CDS has genre, WDC does not ──")
display(con.execute("""
    SELECT
        main_artist_name,
        release_title,
        cds_genre,
        wt_genre,
        wt_label,
        wt_domain,
        release_id
    FROM musicbrainz_release_enriched_v2
    WHERE genre_agreement = 'one source only'
      AND cds_genre IS NOT NULL
      AND cds_genre NOT IN ('None','')
      AND (wt_genre IS NULL OR wt_genre IN ('None',''))
      AND release_id IS NOT NULL
    LIMIT 10
""").fetchdf())

# ── Sample: one source only where WDC has genre ──────────────────
print("\n── Sample: one source only — WDC has genre, CDS does not ──")
display(con.execute("""
    SELECT
        main_artist_name,
        release_title,
        cds_genre,
        wt_genre,
        wt_label,
        wt_domain,
        release_id
    FROM musicbrainz_release_enriched_v2
    WHERE genre_agreement = 'one source only'
      AND wt_genre IS NOT NULL
      AND wt_genre NOT IN ('None','')
      AND (cds_genre IS NULL OR cds_genre IN ('None',''))
      AND release_id IS NOT NULL
    LIMIT 10
""").fetchdf())

Connected ✅

── Genre agreement: full breakdown ──
genre_agreement  total_rows  mb_releases  cds_residual_rows
one source only     5314524      5309877               4647
         differ           1            1                  0

── For 'one source only': which source has the genre? ──
                    genre_source       n
neither has genre (not enriched) 5307068
     CDS has genre, WDC does not    2713
     WDC has genre, CDS does not      96

── Enriched releases WITH and WITHOUT genre ──
                 enrichment_genre_status       n
                     not enriched at all 5290386
enriched but NO genre from either source   16682
                enriched + has CDS genre    2714
                enriched + has WDC genre      96

── 'One source only' in the 182 corroborated releases ──
               genre_source   n
CDS has genre, WDC does not 124
          neither has genre  57

── Sample: one source only — CDS has genre, WDC does not ──


,main_artist_name,release_title,cds_genre,wt_genre,wt_label,wt_domain,release_id
0,Supernova,"Sorry, the Number You've Dialed Has Been Disconnected",alternative,None,None,None,178363
1,Barb Jungr,Bare,jazz,None,None,None,178743
2,Mind over MIDI,Project 3,techno,None,None,None,180031
3,Rosana,Rosana,latin,None,None,None,182899
4,Various Artists,Kickin Mental Detergent,techno,None,None,None,183775
5,Joe Louis Walker,Silvertone Blues,blues,None,None,None,185838
6,Frank Sinatra,Frank Sinatra Conducts Tone Poems of Color,jazz,None,None,None,187789
7,James Reyne,Live in Rio,rock,None,None,None,188195
8,Pizzicato Five,Playboy Playgirl,pop,None,None,None,188270
9,Oldelaf et Monsieur D.,Chansons cons,comedy,None,None,None,188653



── Sample: one source only — WDC has genre, CDS does not ──


,main_artist_name,release_title,cds_genre,wt_genre,wt_label,wt_domain,release_id
0,Junior Wells,On Tap,None,Across The Trax,None,wfdu.radioactivity.fm,451862
1,Drake,If You’re Reading This It's Too Late,None,Hip Hop,Republic,amoeba.com,1610249
2,Muddy Waters,"""Unk"" in Funk",None,Across The Trax,None,wfdu.radioactivity.fm,425855
3,Roy Ayers,Stoned Soul Picnic,None,Soul,Atlantic,cratediggersgold.com,432808
4,Van Morrison,Common One,None,Rock,Warner Bros.,amoeba.com,454086
5,Menahan Street Band,The Crossing,None,New Music Legacy,Daptone,wfdu.radioactivity.fm,1224288
6,Bongwater,The Power of Pussy,None,Rock,Instinct Records,amoeba.com,271455
7,Kraftwerk,45RPM,None,Kraut Rock,Capitol,cratediggersgold.com,2115763
8,Ty Segall,Twins,None,New Music Legacy,Drag City,wfdu.radioactivity.fm,1208171
9,Led Zeppelin,Boxed Set 2,None,Blues,Atlantic,jazz901.radioactivity.fm,53256


In [7]:
import duckdb
import pandas as pd
import numpy as np

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

print("\n── Tables available for evaluation ──")
print(con.execute("""
    SELECT table_name, 
           estimated_size AS approx_rows
    FROM duckdb_tables()
    WHERE table_name IN (
        'musicbrainz_release_enriched_v2',
        'wdc_mb_enrichment_delta',
        'wdc_mb_final_matches',
        'wdc_mb_fuzzy_best',
        'wdc_cds_corroboration',
        'cds_to_mb_best_exact_v4',
        'cds_to_mb_best_enrichment',
        'cds'
    )
    ORDER BY table_name
""").fetchdf().to_string(index=False))

Connected ✅

── Tables available for evaluation ──
                     table_name  approx_rows
                            cds         9763
      cds_to_mb_best_enrichment          564
        cds_to_mb_best_exact_v4         3482
musicbrainz_release_enriched_v2      5314525
          wdc_cds_corroboration          182
        wdc_mb_enrichment_delta        15474
           wdc_mb_final_matches        23921
              wdc_mb_fuzzy_best        11782


In [8]:
print("═"*65)
print("  SECTION 1: SIMILARITY MEASURE ANALYSIS")
print("═"*65)
print("""
WHAT: Analyses the distribution of Jaro-Winkler similarity scores
      and Levenshtein distances across all matched pairs.
WHY:  A good similarity measure should produce a clear separation
      between correct matches (high score) and wrong ones (low score).
      If scores cluster around one value, the measure is not
      discriminating well.
""")

# ── 1a. Overall score distribution ──────────────────────────────
print("── 1a. WDC match score distribution ──")
print(con.execute("""
    SELECT
        CASE
            WHEN wt_match_score = 100      THEN '100     (perfect exact)'
            WHEN wt_match_score >= 97      THEN '97-99   (near perfect)'
            WHEN wt_match_score >= 95      THEN '95-96   (very high)'
            WHEN wt_match_score >= 92      THEN '92-94   (high)'
            WHEN wt_match_score >= 90      THEN '90-91   (auto-accept floor)'
            WHEN wt_match_score >= 85      THEN '85-89   (review range)'
            ELSE                                'below 85 (rejected)'
        END                                AS score_band,
        COUNT(*)                           AS n,
        ROUND(100.0 * COUNT(*) /
              SUM(COUNT(*)) OVER(), 2)     AS pct_of_total
    FROM wdc_mb_enrichment_delta
    GROUP BY score_band
    ORDER BY score_band DESC
""").fetchdf().to_string(index=False))

═════════════════════════════════════════════════════════════════
  SECTION 1: SIMILARITY MEASURE ANALYSIS
═════════════════════════════════════════════════════════════════

WHAT: Analyses the distribution of Jaro-Winkler similarity scores
      and Levenshtein distances across all matched pairs.
WHY:  A good similarity measure should produce a clear separation
      between correct matches (high score) and wrong ones (low score).
      If scores cluster around one value, the measure is not
      discriminating well.

── 1a. WDC match score distribution ──
                 score_band     n  pct_of_total
     97-99   (near perfect)  1894         12.24
        95-96   (very high)   718          4.64
             92-94   (high)   446          2.88
90-91   (auto-accept floor)   183          1.18
    100     (perfect exact) 12233         79.06


In [9]:
# ── 1b. Statistics by match type ────────────────────────────────
print("── 1b. Similarity statistics by match type ──")
print(con.execute("""
    SELECT
        wt_match_type,
        COUNT(*)                            AS n,
        ROUND(AVG(wt_match_score),   2)     AS avg_score,
        ROUND(MIN(wt_match_score),   2)     AS min_score,
        ROUND(MAX(wt_match_score),   2)     AS max_score,
        ROUND(STDDEV(wt_match_score),2)     AS std_score
    FROM wdc_mb_enrichment_delta
    GROUP BY wt_match_type
""").fetchdf().to_string(index=False))

# ── 1c. Jaro-Winkler vs Levenshtein cross-analysis ──────────────
print("\n── 1c. Jaro-Winkler artist sim vs Levenshtein artist dist ──")
print("(Shows the relationship between the two metrics)")
print(con.execute("""
    SELECT
        CASE
            WHEN wt_artist_sim >= 98 THEN 'JW >= 98'
            WHEN wt_artist_sim >= 95 THEN 'JW 95-97'
            WHEN wt_artist_sim >= 90 THEN 'JW 90-94'
            ELSE                          'JW 80-89'
        END                             AS jw_band,
        COUNT(*)                        AS n,
        ROUND(AVG(wt_artist_sim), 2)    AS avg_jw,
        ROUND(AVG(wt_album_sim),  2)    AS avg_album_sim
    FROM wdc_mb_enrichment_delta
    WHERE wt_match_type = 'fuzzy'
    GROUP BY jw_band
    ORDER BY jw_band DESC
""").fetchdf().to_string(index=False))

# ── 1d. Score gap (confidence indicator) ────────────────────────
print("\n── 1d. Score gap distribution (best vs second-best) ──")
print("Large gap = clear winner = high confidence")
print(con.execute("""
    SELECT
        CASE
            WHEN score_gap IS NULL    THEN 'no competitor (unique)'
            WHEN score_gap >= 20      THEN 'gap >= 20  (very clear)'
            WHEN score_gap >= 10      THEN 'gap 10-19  (clear)'
            WHEN score_gap >= 5       THEN 'gap  5-9   (moderate)'
            ELSE                           'gap  0-4   (close race)'
        END                           AS gap_band,
        COUNT(*)                      AS n,
        ROUND(AVG(final_score), 2)    AS avg_score
    FROM wdc_mb_fuzzy_best
    WHERE confidence_class = 'auto_accept'
    GROUP BY gap_band
    ORDER BY n DESC
""").fetchdf().to_string(index=False))

print("""
INTERPRETATION:
  - High proportion of 'no competitor' or 'gap >= 20' rows
    indicates the matching is precise and unambiguous.
  - Many 'gap 0-4' rows would indicate the scoring formula
    struggles to discriminate between candidates.
""")

── 1b. Similarity statistics by match type ──
wt_match_type     n  avg_score  min_score  max_score  std_score
        fuzzy  3242      96.86       90.0     101.25       2.31
        exact 12232     100.00      100.0     100.00       0.00

── 1c. Jaro-Winkler artist sim vs Levenshtein artist dist ──
(Shows the relationship between the two metrics)
 jw_band    n  avg_jw  avg_album_sim
JW >= 98 1919   99.77          96.10
JW 95-97  415   96.43          98.79
JW 90-94  626   92.72          97.36
JW 80-89  282   86.10          98.33

── 1d. Score gap distribution (best vs second-best) ──
Large gap = clear winner = high confidence
               gap_band    n  avg_score
gap  0-4   (close race) 3198      96.84
 no competitor (unique) 1371      97.37
     gap 10-19  (clear)  246      96.74
  gap  5-9   (moderate)  209      95.12
gap >= 20  (very clear)  188      97.79

INTERPRETATION:
  - High proportion of 'no competitor' or 'gap >= 20' rows
    indicates the matching is precise and unambiguo

In [10]:
print("═"*65)
print("  SECTION 2: PRECISION / RECALL / F1 ESTIMATION")
print("═"*65)
print("""
APPROACH: No complete gold standard exists, so we use a proxy method.
  - Exact string matches (artist_key = mb_artist_key AND
    album_key = mb_album_key) are TRUE POSITIVE proxies.
  - Total unique MB releases touched by any WDC row = 
    recall denominator (upper bound on possible TPs).
  - At each score threshold T, we count:
      accepted = pairs with score >= T
      TP_proxy = accepted pairs that are also exact matches
      FP_proxy = accepted - TP_proxy
    Then: precision = TP / accepted
          recall    = TP / total_possible_TP
          F1        = 2*P*R / (P+R)
""")

# Total possible TPs (recall denominator)
n_possible = con.execute("""
    SELECT COUNT(DISTINCT release_id)
    FROM wdc_mb_final_matches
    WHERE release_id IS NOT NULL
""").fetchone()[0]

print(f"Total unique MB releases reachable by WDC: {n_possible:,}")
print()

thresholds = [70, 75, 80, 83, 85, 88, 90, 92, 95, 97, 100]
rows = []

for t in thresholds:
    # All pairs at or above threshold
    accepted = con.execute(f"""
        SELECT COUNT(*) FROM wdc_mb_fuzzy_best
        WHERE final_score >= {t}
    """).fetchone()[0]

    # Proxy TPs = pairs that also satisfy exact key match
    tp = con.execute(f"""
        SELECT COUNT(*) FROM wdc_mb_fuzzy_best
        WHERE final_score >= {t}
          AND wt_album_key  = mb_album_key
          AND wt_artist_key = mb_artist_key
    """).fetchone()[0]

    precision = round(tp / accepted, 4) if accepted > 0 else 0
    recall    = round(tp / n_possible,  4) if n_possible > 0 else 0
    f1        = round(2 * precision * recall / (precision + recall), 4) \
                if (precision + recall) > 0 else 0

    rows.append({
        "threshold" : t,
        "accepted"  : accepted,
        "TP_proxy"  : tp,
        "FP_proxy"  : accepted - tp,
        "precision" : precision,
        "recall"    : recall,
        "F1"        : f1
    })

df_metrics = pd.DataFrame(rows)
print("── Precision / Recall / F1 at varying thresholds ──")
print(df_metrics.to_string(index=False))

# Highlight best F1
best_row = df_metrics.loc[df_metrics['F1'].idxmax()]
print(f"\n✅ Best F1 score: {best_row['F1']} at threshold {best_row['threshold']}")
print(f"   Precision: {best_row['precision']}  |  Recall: {best_row['recall']}")

═════════════════════════════════════════════════════════════════
  SECTION 2: PRECISION / RECALL / F1 ESTIMATION
═════════════════════════════════════════════════════════════════

APPROACH: No complete gold standard exists, so we use a proxy method.
  - Exact string matches (artist_key = mb_artist_key AND
    album_key = mb_album_key) are TRUE POSITIVE proxies.
  - Total unique MB releases touched by any WDC row = 
    recall denominator (upper bound on possible TPs).
  - At each score threshold T, we count:
      accepted = pairs with score >= T
      TP_proxy = accepted pairs that are also exact matches
      FP_proxy = accepted - TP_proxy
    Then: precision = TP / accepted
          recall    = TP / total_possible_TP
          F1        = 2*P*R / (P+R)

Total unique MB releases reachable by WDC: 15,474

── Precision / Recall / F1 at varying thresholds ──
 threshold  accepted  TP_proxy  FP_proxy  precision  recall  F1
        70     10958         0     10958        0.0     0.0   0


In [11]:
print("── CDS pipeline precision by strategy ──")
print("""
NOTE: CDS exact matching precision is very high by design.
      Requiring BOTH normalised artist AND title keys to match
      simultaneously makes false positives extremely rare.
""")
print(con.execute("""
    SELECT
        'Exact key match'               AS strategy,
        COUNT(*)                        AS matches,
        COUNT(CASE WHEN lower(trim(artist)) = lower(trim(main_artist_name))
                   THEN 1 END)          AS raw_artist_match,
        COUNT(CASE WHEN lower(trim(title))
                         = lower(trim(release_title))
                   THEN 1 END)          AS raw_title_match,
        ROUND(100.0 *
            COUNT(CASE WHEN lower(trim(artist))
                              = lower(trim(main_artist_name))
                       THEN 1 END) / COUNT(*), 2)
                                        AS artist_precision_pct,
        ROUND(100.0 *
            COUNT(CASE WHEN lower(trim(title))
                              = lower(trim(release_title))
                       THEN 1 END) / COUNT(*), 2)
                                        AS title_precision_pct
    FROM cds_to_mb_best_exact_v4
""").fetchdf().to_string(index=False))

print("\n── CDS fuzzy match confidence breakdown ──")
print(con.execute("""
    SELECT
        enrichment_confidence,
        COUNT(*)                        AS n,
        ROUND(AVG(score), 2)            AS avg_score,
        ROUND(AVG(title_sim), 2)        AS avg_title_sim
    FROM cds_to_mb_best_enrichment
    GROUP BY enrichment_confidence
    ORDER BY enrichment_confidence
""").fetchdf().to_string(index=False))

── CDS pipeline precision by strategy ──

NOTE: CDS exact matching precision is very high by design.
      Requiring BOTH normalised artist AND title keys to match
      simultaneously makes false positives extremely rare.

       strategy  matches  raw_artist_match  raw_title_match  artist_precision_pct  title_precision_pct
Exact key match     3482              3118             2835                 89.55                81.42

── CDS fuzzy match confidence breakdown ──
enrichment_confidence   n  avg_score  avg_title_sim
                 high 535      97.83          98.41
                  low  24      98.37          98.92
               medium   5      98.58          99.56


In [12]:
print("═"*65)
print("  SECTION 3: SIMILARITY DEPENDENCY ANALYSIS")
print("═"*65)
print("""
DEFINITION: Similarity dependency measures whether knowing the
similarity on one field (artist) predicts the similarity on
another field (album).

If corr(artist_sim, album_sim) is LOW → the two fields carry
independent information → weighting both is justified.

If corr is HIGH → one field dominates the other → the weight
ratio matters more because they are redundant.
""")

print("── Pearson correlations between similarity signals ──")
print(con.execute("""
    SELECT
        ROUND(CORR(wt_artist_sim, wt_album_sim),  4)
                            AS corr_artist_vs_album,
        ROUND(CORR(wt_artist_sim, wt_match_score),4)
                            AS corr_artist_vs_final,
        ROUND(CORR(wt_album_sim,  wt_match_score),4)
                            AS corr_album_vs_final,
        ROUND(STDDEV(wt_artist_sim), 2)
                            AS std_artist_sim,
        ROUND(STDDEV(wt_album_sim),  2)
                            AS std_album_sim
    FROM wdc_mb_enrichment_delta
    WHERE wt_match_type = 'fuzzy'
""").fetchdf().to_string(index=False))

print("\n── Cross-tabulation: artist_sim band vs avg album_sim ──")
print(con.execute("""
    SELECT
        CASE
            WHEN wt_artist_sim >= 98 THEN '1. artist >= 98'
            WHEN wt_artist_sim >= 95 THEN '2. artist 95-97'
            WHEN wt_artist_sim >= 90 THEN '3. artist 90-94'
            WHEN wt_artist_sim >= 85 THEN '4. artist 85-89'
            ELSE                          '5. artist 80-84'
        END                             AS artist_band,
        COUNT(*)                        AS n,
        ROUND(AVG(wt_album_sim),  2)    AS avg_album_sim,
        ROUND(MIN(wt_album_sim),  2)    AS min_album_sim,
        ROUND(MAX(wt_album_sim),  2)    AS max_album_sim,
        ROUND(STDDEV(wt_album_sim),2)   AS std_album_sim
    FROM wdc_mb_enrichment_delta
    WHERE wt_match_type = 'fuzzy'
    GROUP BY artist_band
    ORDER BY artist_band
""").fetchdf().to_string(index=False))

print("""
INTERPRETATION:
  If avg_album_sim stays roughly constant across all artist_band rows
  → artist_sim and album_sim are INDEPENDENT → good.
  If avg_album_sim rises with artist_band
  → they are DEPENDENT → one field carries the other.
""")

═════════════════════════════════════════════════════════════════
  SECTION 3: SIMILARITY DEPENDENCY ANALYSIS
═════════════════════════════════════════════════════════════════

DEFINITION: Similarity dependency measures whether knowing the
similarity on one field (artist) predicts the similarity on
another field (album).

If corr(artist_sim, album_sim) is LOW → the two fields carry
independent information → weighting both is justified.

If corr is HIGH → one field dominates the other → the weight
ratio matters more because they are redundant.

── Pearson correlations between similarity signals ──
 corr_artist_vs_album  corr_artist_vs_final  corr_album_vs_final  std_artist_sim  std_album_sim
              -0.2515                0.6661               0.5497            4.44           3.23

── Cross-tabulation: artist_sim band vs avg album_sim ──
    artist_band    n  avg_album_sim  min_album_sim  max_album_sim  std_album_sim
1. artist >= 98 1919          96.10          88.00          100.0

In [13]:
print("═"*65)
print("  SECTION 4: MATCHING DEPENDENCY ANALYSIS")
print("═"*65)
print("""
DEFINITION: Matching dependency asks: if a record matches on
field A, does it also match on field B?

For your pipeline the key question is:
  P(album_sim >= 90 | artist_sim >= 90) = ?

If this probability is LOW → requiring both fields for acceptance
is essential, because an artist match alone does not guarantee
an album match.

If this probability is HIGH → the two fields are redundant and
matching on artist alone would suffice.
""")

print("── Conditional matching probabilities ──")
print(con.execute("""
    SELECT
        'P(album>=90 | artist>=90)' AS dependency_statement,
        ROUND(100.0 *
            COUNT(CASE WHEN wt_album_sim  >= 90 THEN 1 END) /
            NULLIF(COUNT(CASE WHEN wt_artist_sim >= 90
                              THEN 1 END), 0), 2)    AS pct
    FROM wdc_mb_enrichment_delta
    WHERE wt_match_type = 'fuzzy'

    UNION ALL

    SELECT 'P(artist>=90 | album>=90)',
        ROUND(100.0 *
            COUNT(CASE WHEN wt_artist_sim >= 90 THEN 1 END) /
            NULLIF(COUNT(CASE WHEN wt_album_sim >= 90
                              THEN 1 END), 0), 2)
    FROM wdc_mb_enrichment_delta
    WHERE wt_match_type = 'fuzzy'

    UNION ALL

    SELECT 'P(both>=90)',
        ROUND(100.0 *
            COUNT(CASE WHEN wt_artist_sim >= 90
                        AND wt_album_sim  >= 90 THEN 1 END) /
            NULLIF(COUNT(*), 0), 2)
    FROM wdc_mb_enrichment_delta
    WHERE wt_match_type = 'fuzzy'

    UNION ALL

    SELECT 'P(artist>=90 alone, album<90)',
        ROUND(100.0 *
            COUNT(CASE WHEN wt_artist_sim >= 90
                        AND wt_album_sim  <  90 THEN 1 END) /
            NULLIF(COUNT(*), 0), 2)
    FROM wdc_mb_enrichment_delta
    WHERE wt_match_type = 'fuzzy'
""").fetchdf().to_string(index=False))

print("\n── CDS: raw field match rates ──")
print(con.execute("""
    SELECT
        COUNT(*)                                        AS total,
        COUNT(CASE WHEN lower(trim(artist))
                        = lower(trim(main_artist_name))
                   THEN 1 END)                          AS artist_exact,
        COUNT(CASE WHEN lower(trim(title))
                        = lower(trim(release_title))
                   THEN 1 END)                          AS title_exact,
        COUNT(CASE WHEN lower(trim(artist))
                        = lower(trim(main_artist_name))
                    AND lower(trim(title))
                        = lower(trim(release_title))
                   THEN 1 END)                          AS both_exact,
        ROUND(100.0 *
            COUNT(CASE WHEN lower(trim(artist))
                            = lower(trim(main_artist_name))
                       THEN 1 END) / COUNT(*), 2)       AS artist_pct,
        ROUND(100.0 *
            COUNT(CASE WHEN lower(trim(title))
                            = lower(trim(release_title))
                       THEN 1 END) / COUNT(*), 2)       AS title_pct,
        ROUND(100.0 *
            COUNT(CASE WHEN lower(trim(artist))
                            = lower(trim(main_artist_name))
                        AND lower(trim(title))
                            = lower(trim(release_title))
                       THEN 1 END) / COUNT(*), 2)       AS both_pct
    FROM cds_to_mb_best_exact_v4
""").fetchdf().to_string(index=False))

print("""
INTERPRETATION:
  If P(album match | artist match) is substantially less than 100%,
  then requiring both fields for acceptance is necessary.
  A match on artist alone does not guarantee the same album.
""")

═════════════════════════════════════════════════════════════════
  SECTION 4: MATCHING DEPENDENCY ANALYSIS
═════════════════════════════════════════════════════════════════

DEFINITION: Matching dependency asks: if a record matches on
field A, does it also match on field B?

For your pipeline the key question is:
  P(album_sim >= 90 | artist_sim >= 90) = ?

If this probability is LOW → requiring both fields for acceptance
is essential, because an artist match alone does not guarantee
an album match.

If this probability is HIGH → the two fields are redundant and
matching on artist alone would suffice.

── Conditional matching probabilities ──
         dependency_statement    pct
    P(album>=90 | artist>=90) 106.28
    P(artist>=90 | album>=90)  94.09
                  P(both>=90)  88.40
P(artist>=90 alone, album<90)   2.90

── CDS: raw field match rates ──
 total  artist_exact  title_exact  both_exact  artist_pct  title_pct  both_pct
  3482          3118         2835        2572     

In [14]:
print("═"*65)
print("  SECTION 5: FUNCTIONAL DEPENDENCY ANALYSIS")
print("═"*65)
print("""
DEFINITION: A → B (A functionally determines B) means each
value of A maps to exactly one value of B.

Key FDs to verify:
  FD1: release_id → release_title       (should hold: 1 ID = 1 title)
  FD2: (artist, album) → release_id     (partially holds after dedup)
  FD3: artist → genre                   (should NOT hold: multi-genre)
  FD4: cds_genre ↔ wt_genre agreement   (soft FD via corroboration)
""")

print("── FD1: release_id → release_title ──")
print(con.execute("""
    SELECT
        COUNT(DISTINCT release_id)      AS unique_release_ids,
        COUNT(DISTINCT release_title)   AS unique_titles,
        -- Violations: one release_id maps to more than one title
        COUNT(*) FILTER (WHERE cnt > 1) AS fd_violations
    FROM (
        SELECT release_id, COUNT(DISTINCT release_title) AS cnt
        FROM musicbrainz_release_enriched_v2
        WHERE release_id IS NOT NULL
        GROUP BY release_id
    )
""").fetchdf().to_string(index=False))

print("\n── FD2: (wt_artist, wt_album) → release_id ──")
print(con.execute("""
    SELECT
        COUNT(DISTINCT wt_artist || '|' || wt_album)
                            AS unique_wdc_pairs,
        COUNT(DISTINCT release_id)
                            AS unique_mb_releases,
        -- Ratio close to 1.0 = good 1:1 FD
        ROUND(1.0 *
            COUNT(DISTINCT release_id) /
            NULLIF(COUNT(DISTINCT wt_artist || '|' || wt_album), 0),
        4)                  AS fd_ratio,
        -- Violations: one wdc pair maps to >1 MB release
        COUNT(*) FILTER (WHERE cnt > 1)
                            AS fd_violations
    FROM (
        SELECT
            wt_artist || '|' || wt_album AS pair,
            COUNT(DISTINCT release_id)   AS cnt
        FROM wdc_mb_final_matches
        WHERE release_id IS NOT NULL
          AND wt_artist IS NOT NULL
          AND wt_album  IS NOT NULL
        GROUP BY pair
    ) t
    JOIN (
        SELECT
            COUNT(DISTINCT wt_artist || '|' || wt_album)
                AS unique_wdc_pairs,
            COUNT(DISTINCT release_id) AS unique_mb_releases
        FROM wdc_mb_final_matches
        WHERE release_id IS NOT NULL
    ) base ON 1=1
""").fetchdf().to_string(index=False))

print("\n── FD3: artist → genre (should NOT be strict FD) ──")
print(con.execute("""
    SELECT
        COUNT(DISTINCT main_artist_name)            AS unique_artists,
        COUNT(DISTINCT main_artist_name || '|' ||
              coalesce(cds_genre,'NULL'))            AS artist_genre_pairs,
        -- avg genres per artist > 1 confirms NOT a strict FD
        ROUND(1.0 *
            COUNT(DISTINCT main_artist_name || '|' ||
                  coalesce(cds_genre,'NULL')) /
            NULLIF(COUNT(DISTINCT main_artist_name), 0),
        3)                                          AS avg_genres_per_artist,
        MAX(cnt)                                    AS max_genres_one_artist
    FROM (
        SELECT main_artist_name,
               COUNT(DISTINCT coalesce(cds_genre,'NULL')) AS cnt
        FROM musicbrainz_release_enriched_v2
        WHERE cds_genre IS NOT NULL
          AND cds_genre NOT IN ('None','')
        GROUP BY main_artist_name
    ) t
    CROSS JOIN (
        SELECT COUNT(DISTINCT main_artist_name) AS n_artists
        FROM musicbrainz_release_enriched_v2
    ) base
""").fetchdf().to_string(index=False))

print("\n── FD4: genre agreement across sources (soft FD) ──")
print(con.execute("""
    SELECT
        genre_agreement,
        COUNT(*)                                AS n,
        ROUND(100.0 * COUNT(*) /
              SUM(COUNT(*)) OVER(), 2)          AS pct
    FROM musicbrainz_release_enriched_v2
    WHERE is_corroborated = TRUE
    GROUP BY genre_agreement
    ORDER BY n DESC
""").fetchdf().to_string(index=False))

print("""
INTERPRETATION:
  FD1 should show 0 violations (each release_id has one title).
  FD2 ratio close to 1.0 confirms matching did not collapse
      multiple different releases onto the same ID.
  FD3 avg_genres_per_artist > 1 confirms genre is NOT
      functionally determined by artist alone — justifying
      the need for external enrichment sources.
  FD4 low 'agree' count confirms genre classification varies
      between sources — a known data quality challenge.
""")

═════════════════════════════════════════════════════════════════
  SECTION 5: FUNCTIONAL DEPENDENCY ANALYSIS
═════════════════════════════════════════════════════════════════

DEFINITION: A → B (A functionally determines B) means each
value of A maps to exactly one value of B.

Key FDs to verify:
  FD1: release_id → release_title       (should hold: 1 ID = 1 title)
  FD2: (artist, album) → release_id     (partially holds after dedup)
  FD3: artist → genre                   (should NOT hold: multi-genre)
  FD4: cds_genre ↔ wt_genre agreement   (soft FD via corroboration)

── FD1: release_id → release_title ──


BinderException: Binder Error: Referenced column "release_title" not found in FROM clause!
Candidate bindings: "release_id"

LINE 4:         COUNT(DISTINCT release_title)   AS unique_titles,
                               ^

In [3]:
print("═"*65)
print("  SECTION 6: DUPLICATE DETECTION")
print("═"*65)
print("""
Four types of duplicates to detect:
  6a. MB internal: same artist+album with multiple release_ids
      (different commercial editions — intentional in MusicBrainz)
  6b. WDC row duplicates: same URL+artist+album multiple times
      (same playlist page scraped twice)
  6c. Multi-source confirmation: same MB release in N web domains
      (desirable — stronger corroboration evidence)
  6d. CDS duplicates: same artist+title with different cds_ids
      (multi-disc sets, different catalogue entries)
""")

print("── 6a. MB editions (same artist+album, different release_id) ──")
print(con.execute("""
    SELECT
        n_editions,
        COUNT(*)    AS n_artist_album_pairs
    FROM (
        SELECT
            main_artist_name,
            release_title,
            COUNT(DISTINCT release_id) AS n_editions
        FROM musicbrainz_release_enriched_v2
        WHERE release_id      IS NOT NULL
          AND main_artist_name IS NOT NULL
          AND release_title    IS NOT NULL
        GROUP BY main_artist_name, release_title
    )
    GROUP BY n_editions
    ORDER BY n_editions DESC
    LIMIT 10
""").fetchdf().to_string(index=False))

print("\n── Top 10 MB releases with most editions ──")
display(con.execute("""
    SELECT
        main_artist_name,
        release_title,
        COUNT(DISTINCT release_id) AS n_editions
    FROM musicbrainz_release_enriched_v2
    WHERE release_id      IS NOT NULL
      AND main_artist_name IS NOT NULL
    GROUP BY main_artist_name, release_title
    ORDER BY n_editions DESC
    LIMIT 10
""").fetchdf())

═════════════════════════════════════════════════════════════════
  SECTION 6: DUPLICATE DETECTION
═════════════════════════════════════════════════════════════════

Four types of duplicates to detect:
  6a. MB internal: same artist+album with multiple release_ids
      (different commercial editions — intentional in MusicBrainz)
  6b. WDC row duplicates: same URL+artist+album multiple times
      (same playlist page scraped twice)
  6c. Multi-source confirmation: same MB release in N web domains
      (desirable — stronger corroboration evidence)
  6d. CDS duplicates: same artist+title with different cds_ids
      (multi-disc sets, different catalogue entries)

── 6a. MB editions (same artist+album, different release_id) ──


NameError: name 'con' is not defined

In [2]:
print("── 6b. WDC row duplicates ──")
print(con.execute("""
    SELECT
        COUNT(*)                                        AS total_rows,
        COUNT(DISTINCT source_url || wt_artist
                                   || wt_album)        AS unique_triples,
        COUNT(*) -
        COUNT(DISTINCT source_url || wt_artist
                                   || wt_album)        AS duplicate_rows,
        ROUND(100.0 * (
            COUNT(*) -
            COUNT(DISTINCT source_url || wt_artist || wt_album)
        ) / COUNT(*), 2)                               AS dup_rate_pct
    FROM wdc_cleaned_for_matching
    WHERE wt_artist IS NOT NULL
      AND wt_album  IS NOT NULL
""").fetchdf().to_string(index=False))

print("\n── 6c. MB releases confirmed by multiple web domains ──")
print(con.execute("""
    SELECT
        n_domains,
        COUNT(*) AS n_mb_releases
    FROM (
        SELECT
            release_id,
            COUNT(DISTINCT domain) AS n_domains
        FROM wdc_mb_final_matches
        WHERE release_id IS NOT NULL
        GROUP BY release_id
    )
    GROUP BY n_domains
    ORDER BY n_domains DESC
    LIMIT 10
""").fetchdf().to_string(index=False))

print("\n── Top 10 MB releases confirmed by most independent domains ──")
display(con.execute("""
    SELECT
        b.main_artist_name,
        b.release_title,
        COUNT(DISTINCT f.domain)    AS n_domains,
        COUNT(*)                    AS total_wdc_rows,
        MIN(ROUND(f.final_score,1)) AS min_score,
        MAX(ROUND(f.final_score,1)) AS max_score
    FROM wdc_mb_final_matches f
    JOIN music_enrichment_base_for_webtables_v1 b
      ON f.release_id = b.release_id
    WHERE f.release_id IS NOT NULL
    GROUP BY b.main_artist_name, b.release_title
    ORDER BY n_domains DESC
    LIMIT 10
""").fetchdf())

print("\n── 6d. CDS duplicates (same artist+title, multiple cds_ids) ──")
print(con.execute("""
    SELECT
        COUNT(*)                        AS total_cds,
        COUNT(DISTINCT artist || title) AS unique_artist_title,
        COUNT(*) -
        COUNT(DISTINCT artist || title) AS duplicate_entries,
        ROUND(100.0 * (
            COUNT(*) - COUNT(DISTINCT artist || title)
        ) / COUNT(*), 2)               AS dup_rate_pct
    FROM cds
""").fetchdf().to_string(index=False))

print("\n── Most duplicated CDS entries ──")
print(con.execute("""
    SELECT artist, title, COUNT(*) AS n_cds_entries
    FROM cds
    GROUP BY artist, title
    HAVING COUNT(*) > 2
    ORDER BY n_cds_entries DESC
    LIMIT 10
""").fetchdf().to_string(index=False))

── 6b. WDC row duplicates ──


NameError: name 'con' is not defined

In [1]:

# Gather key numbers
n_wt_total   = con.execute("SELECT COUNT(*) FROM wdc_mb_enrichment_delta").fetchdf()
n_exact_wt   = con.execute("SELECT COUNT(*) FROM wdc_mb_enrichment_delta WHERE wt_match_type='exact'").fetchone()[0]
n_fuzzy_wt   = con.execute("SELECT COUNT(*) FROM wdc_mb_enrichment_delta WHERE wt_match_type='fuzzy'").fetchone()[0]
avg_score    = con.execute("SELECT ROUND(AVG(wt_match_score),2) FROM wdc_mb_enrichment_delta WHERE wt_match_type='fuzzy'").fetchone()[0]
corr_val     = con.execute("SELECT ROUND(CORR(wt_artist_sim,wt_album_sim),4) FROM wdc_mb_enrichment_delta WHERE wt_match_type='fuzzy'").fetchone()[0]
n_corr       = con.execute("SELECT COUNT(*) FROM wdc_cds_corroboration").fetchone()[0]
n_cds_exact  = con.execute("SELECT COUNT(*) FROM cds_to_mb_best_exact_v4").fetchone()[0]



NameError: name 'con' is not defined